In [2]:
import pandas as pd
from pathlib import Path

In [21]:
PROCESSED_DIR = Path("../data/processed")

historical_file = PROCESSED_DIR / "historical_matches.csv"

historical_df = pd.read_csv(historical_file)

historical_df.shape

(1140, 24)

In [22]:
# Date conversion
historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc = True
)

historical_df.dtypes

match_id                                str
date                    datetime64[us, UTC]
matchday                              int64
home_team                               str
away_team                               str
home_goals                            int64
away_goals                            int64
home_xg                             float64
away_xg                             float64
home_shots                          float64
away_shots                          float64
home_shots_on_target                float64
away_shots_on_target                float64
home_possession                     float64
away_possession                     float64
home_passes                         float64
away_passes                         float64
home_corners                        float64
away_corners                        float64
winner                                  str
season                                  str
home_odds                           float64
draw_odds                       

In [23]:
# For feature engineering purposes, we want to make home-team rows (2,280 rows)
# Makes it easier to calculate rolling averages since for example Arsenal will always be in the same team column.
# All the stats will be written from Arsenal's perspective.
home_rows = historical_df[
    [
        "match_id",
        "date",
        "season",
        "matchday",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "home_xg",
        "away_xg",
        "home_shots",
        "away_shots",
        "home_shots_on_target",
        "away_shots_on_target",
        "home_possession",
        "away_possession",
        "home_passes",
        "away_passes",
        "home_corners",
        "away_corners"
    ]
].copy()

In [24]:
home_rows.shape

(1140, 20)

In [25]:
# For every row the home team will become "team" and the away team will become "opponent".
# Stats will become "for" and "against".

home_rows = home_rows.rename(columns = {
    "home_team": "team",
    "away_team": "opponent",

    "home_goals": "goals_for",
    "away_goals": "goals_against",

    "home_xg": "xg_for",
    "away_xg": "xg_against",

    "home_shots": "shots_for",
    "away_shots": "shots_against",

    "home_shots_on_target": "shots_on_target_for",
    "away_shots_on_target": "shots_on_target_against",

    "home_possession": "possession_for",
    "away_possession": "possession_against",

    "home_passes": "passes_for",
    "away_passes": "passes_against",

    "home_corners": "corners_for",
    "away_corners": "corners_against"
})

home_rows["venue"] = "home"

In [26]:
home_rows.head()

,match_id,date,season,matchday,team,opponent,goals_for,goals_against,xg_for,xg_against,...,shots_against,shots_on_target_for,shots_on_target_against,possession_for,possession_against,passes_for,passes_against,corners_for,corners_against,venue
0,mt_012211017,2023-10-07 14:00:00+00:00,2023_24,8,Manchester United,Brentford,2,1,1.43,0.95,...,11.0,8.0,3.0,64.0,36.0,557.0,326.0,7.0,5.0,home
1,mt_012211028,2023-10-21 14:00:00+00:00,2023_24,9,Newcastle United,Crystal Palace,4,0,2.39,1.21,...,17.0,7.0,3.0,60.0,40.0,573.0,389.0,6.0,8.0,home
2,mt_012211041,2023-10-29 14:00:00+00:00,2023_24,10,Liverpool,Nottingham Forest,3,0,3.19,0.45,...,9.0,8.0,1.0,73.0,27.0,728.0,266.0,8.0,3.0,home
3,mt_012211070,2023-09-30 16:30:00+00:00,2023_24,7,Tottenham Hotspur,Liverpool,2,1,2.23,1.31,...,12.0,8.0,4.0,65.0,35.0,662.0,364.0,11.0,5.0,home
4,mt_012211208,2023-11-04 15:00:00+00:00,2023_24,11,Brentford,West Ham United,3,2,1.63,1.82,...,12.0,4.0,2.0,59.0,41.0,560.0,373.0,4.0,3.0,home


In [27]:
# Do the same for away-team now
away_rows = historical_df[
    [
        "match_id",
        "date",
        "season",
        "matchday",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "home_xg",
        "away_xg",
        "home_shots",
        "away_shots",
        "home_shots_on_target",
        "away_shots_on_target",
        "home_possession",
        "away_possession",
        "home_passes",
        "away_passes",
        "home_corners",
        "away_corners"
    ]
].copy()

In [28]:
away_rows = away_rows.rename(columns={
    "away_team": "team",
    "home_team": "opponent",

    "away_goals": "goals_for",
    "home_goals": "goals_against",

    "away_xg": "xg_for",
    "home_xg": "xg_against",

    "away_shots": "shots_for",
    "home_shots": "shots_against",

    "away_shots_on_target": "shots_on_target_for",
    "home_shots_on_target": "shots_on_target_against",

    "away_possession": "possession_for",
    "home_possession": "possession_against",

    "away_passes": "passes_for",
    "home_passes": "passes_against",

    "away_corners": "corners_for",
    "home_corners": "corners_against"
})

away_rows["venue"] = "away"

In [29]:
away_rows.head()

,match_id,date,season,matchday,opponent,team,goals_against,goals_for,xg_against,xg_for,...,shots_for,shots_on_target_against,shots_on_target_for,possession_against,possession_for,passes_against,passes_for,corners_against,corners_for,venue
0,mt_012211017,2023-10-07 14:00:00+00:00,2023_24,8,Manchester United,Brentford,2,1,1.43,0.95,...,11.0,8.0,3.0,64.0,36.0,557.0,326.0,7.0,5.0,away
1,mt_012211028,2023-10-21 14:00:00+00:00,2023_24,9,Newcastle United,Crystal Palace,4,0,2.39,1.21,...,17.0,7.0,3.0,60.0,40.0,573.0,389.0,6.0,8.0,away
2,mt_012211041,2023-10-29 14:00:00+00:00,2023_24,10,Liverpool,Nottingham Forest,3,0,3.19,0.45,...,9.0,8.0,1.0,73.0,27.0,728.0,266.0,8.0,3.0,away
3,mt_012211070,2023-09-30 16:30:00+00:00,2023_24,7,Tottenham Hotspur,Liverpool,2,1,2.23,1.31,...,12.0,8.0,4.0,65.0,35.0,662.0,364.0,11.0,5.0,away
4,mt_012211208,2023-11-04 15:00:00+00:00,2023_24,11,Brentford,West Ham United,3,2,1.63,1.82,...,12.0,4.0,2.0,59.0,41.0,560.0,373.0,4.0,3.0,away


In [32]:
# Combine home rows and away rows

team_columns = [
    "match_id",
    "date",
    "season",
    "matchday",
    "team",
    "opponent",
    "venue",
    "goals_for",
    "goals_against",
    "xg_for",
    "xg_against",
    "shots_for",
    "shots_against",
    "shots_on_target_for",
    "shots_on_target_against",
    "possession_for",
    "possession_against",
    "passes_for",
    "passes_against",
    "corners_for",
    "corners_against"
]

home_rows = home_rows[team_columns]
away_rows = away_rows[team_columns]

In [33]:
# Combine home and away rows

team_matches = pd.concat(
    [home_rows, away_rows],
    ignore_index = True
)

team_matches.shape

(2280, 21)

In [34]:
# Sort each team's matches chronologically

team_matches = team_matches.sort_values(
    ["team", "date"]
).reset_index(drop = True)

In [35]:
# Inspecting Arsenal's matches
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "team",
        "opponent",
        "venue",
        "goals_for",
        "goals_against"
    ]
].head(10)

,date,team,opponent,venue,goals_for,goals_against
0,2023-08-12 12:00:00+00:00,Arsenal,Nottingham Forest,home,2,1
1,2023-08-21 19:00:00+00:00,Arsenal,Crystal Palace,away,1,0
2,2023-08-26 14:00:00+00:00,Arsenal,Fulham,home,2,2
3,2023-09-03 15:30:00+00:00,Arsenal,Manchester United,home,3,1
4,2023-09-17 15:30:00+00:00,Arsenal,Everton,away,1,0
5,2023-09-24 13:00:00+00:00,Arsenal,Tottenham Hotspur,home,2,2
6,2023-09-30 14:00:00+00:00,Arsenal,Bournemouth,away,4,0
7,2023-10-08 15:30:00+00:00,Arsenal,Manchester City,home,1,0
8,2023-10-21 16:30:00+00:00,Arsenal,Chelsea,away,2,2
9,2023-10-28 14:00:00+00:00,Arsenal,Sheffield United,home,5,0


In [36]:
# Looking for each team's average goals scored over previous 5 matches
# Creating a pre-match rolling feature.

team_matches["avg_goals_for_last5"] = (
    team_matches

    # Calculating separately for each team.
    .groupby("team")["goals_for"]
    .transform(
        # Moving the data down one match (shift(1)) so the current match is excluded.
        # shift(1) prevents data leakage.
        # Averaging the previous 5 matches (rolling(5))
        lambda x: x.shift(1).rolling(5).mean()
    )
)

In [37]:
# Inspecting Arsenal again

team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "goals_for",
        "avg_goals_for_last5"
    ]
].head(10)

,date,opponent,goals_for,avg_goals_for_last5
0,2023-08-12 12:00:00+00:00,Nottingham Forest,2,NaN
1,2023-08-21 19:00:00+00:00,Crystal Palace,1,NaN
2,2023-08-26 14:00:00+00:00,Fulham,2,NaN
3,2023-09-03 15:30:00+00:00,Manchester United,3,NaN
4,2023-09-17 15:30:00+00:00,Everton,1,NaN
5,2023-09-24 13:00:00+00:00,Tottenham Hotspur,2,1.8
6,2023-09-30 14:00:00+00:00,Bournemouth,4,1.8
7,2023-10-08 15:30:00+00:00,Manchester City,1,2.4
8,2023-10-21 16:30:00+00:00,Chelsea,2,2.2
9,2023-10-28 14:00:00+00:00,Sheffield United,5,2.0


In [38]:
# Applying same rolling logic to the rest of the main team stats

rolling_stats = [
    "goals_for",
    "goals_against",
    "xg_for",
    "xg_against",
    "shots_for",
    "shots_against",
    "shots_on_target_for",
    "shots_on_target_against",
    "possession_for",
    "possession_against",
    "passes_for",
    "passes_against",
    "corners_for",
    "corners_against"
]

for stat in rolling_stats:
    team_matches[f"avg_{stat}_last5"] = (
        team_matches
        .groupby("team")[stat]
        .transform(
            lambda x: x.shift(1).rolling(5).mean()
        )
    )

In [39]:
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "avg_goals_for_last5",
        "avg_goals_against_last5",
        "avg_xg_for_last5",
        "avg_xg_against_last5",
        "avg_shots_for_last5"
    ]
].head(10)

,date,opponent,avg_goals_for_last5,avg_goals_against_last5,avg_xg_for_last5,avg_xg_against_last5,avg_shots_for_last5
0,2023-08-12 12:00:00+00:00,Nottingham Forest,NaN,NaN,NaN,NaN,NaN
1,2023-08-21 19:00:00+00:00,Crystal Palace,NaN,NaN,NaN,NaN,NaN
2,2023-08-26 14:00:00+00:00,Fulham,NaN,NaN,NaN,NaN,NaN
3,2023-09-03 15:30:00+00:00,Manchester United,NaN,NaN,NaN,NaN,NaN
4,2023-09-17 15:30:00+00:00,Everton,NaN,NaN,NaN,NaN,NaN
5,2023-09-24 13:00:00+00:00,Tottenham Hotspur,1.8,0.8,1.868,0.806,15.6
6,2023-09-30 14:00:00+00:00,Bournemouth,1.8,1.0,2.108,0.862,15.2
7,2023-10-08 15:30:00+00:00,Manchester City,2.4,1.0,2.390,0.790,15.4
8,2023-10-21 16:30:00+00:00,Chelsea,2.2,0.6,1.828,0.774,14.0
9,2023-10-28 14:00:00+00:00,Sheffield United,2.0,0.8,1.572,0.856,13.2


In [40]:
# Converting results into league points.

team_matches["points"] = 0

team_matches.loc[
    team_matches["goals_for"] > team_matches["goals_against"],
    "points"
] = 3

team_matches.loc[
    team_matches["goals_for"] == team_matches["goals_against"],
    "points"
] = 1

In [41]:
# Creating a leakage-safe rolling total over previous 5 matches.

team_matches["points_last5"] = (
    team_matches
    .groupby("team")["points"]
    .transform(
        lambda x: x.shift(1).rolling(5).sum()
    )
)

In [42]:
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "goals_for",
        "goals_against",
        "points",
        "points_last5"
    ]
].head(10)

,date,opponent,goals_for,goals_against,points,points_last5
0,2023-08-12 12:00:00+00:00,Nottingham Forest,2,1,3,NaN
1,2023-08-21 19:00:00+00:00,Crystal Palace,1,0,3,NaN
2,2023-08-26 14:00:00+00:00,Fulham,2,2,1,NaN
3,2023-09-03 15:30:00+00:00,Manchester United,3,1,3,NaN
4,2023-09-17 15:30:00+00:00,Everton,1,0,3,NaN
5,2023-09-24 13:00:00+00:00,Tottenham Hotspur,2,2,1,13.0
6,2023-09-30 14:00:00+00:00,Bournemouth,4,0,3,11.0
7,2023-10-08 15:30:00+00:00,Manchester City,1,0,3,11.0
8,2023-10-21 16:30:00+00:00,Chelsea,2,2,1,13.0
9,2023-10-28 14:00:00+00:00,Sheffield United,5,0,3,11.0


In [43]:
# Create a W/D/L column

team_matches["result"] = "D"

team_matches.loc[
    team_matches["goals_for"] > team_matches["goals_against"],
    "result"
] = "W"

team_matches.loc[
    team_matches["goals_for"] < team_matches["goals_against"],
    "result"
] = "L"

In [44]:
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "venue",
        "goals_for",
        "goals_against",
        "result",
        "points_last5"
    ]
].head(10)

,date,opponent,venue,goals_for,goals_against,result,points_last5
0,2023-08-12 12:00:00+00:00,Nottingham Forest,home,2,1,W,NaN
1,2023-08-21 19:00:00+00:00,Crystal Palace,away,1,0,W,NaN
2,2023-08-26 14:00:00+00:00,Fulham,home,2,2,D,NaN
3,2023-09-03 15:30:00+00:00,Manchester United,home,3,1,W,NaN
4,2023-09-17 15:30:00+00:00,Everton,away,1,0,W,NaN
5,2023-09-24 13:00:00+00:00,Tottenham Hotspur,home,2,2,D,13.0
6,2023-09-30 14:00:00+00:00,Bournemouth,away,4,0,W,11.0
7,2023-10-08 15:30:00+00:00,Manchester City,home,1,0,W,11.0
8,2023-10-21 16:30:00+00:00,Chelsea,away,2,2,D,13.0
9,2023-10-28 14:00:00+00:00,Sheffield United,home,5,0,W,11.0


In [45]:
# Creating a rolling goal difference feature over previous 5 matches.

team_matches["goal_difference"] = (
    team_matches["goals_for"] - team_matches["goals_against"]
)

In [46]:
team_matches["goal_difference_last5"] = (
    team_matches
    .groupby("team")["goal_difference"]
    .transform(
        lambda x: x.shift(1).rolling(5).sum()
    )
)

In [47]:
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "result",
        "points_last5",
        "goal_difference_last5"
    ]
].head(10)

,date,opponent,result,points_last5,goal_difference_last5
0,2023-08-12 12:00:00+00:00,Nottingham Forest,W,NaN,NaN
1,2023-08-21 19:00:00+00:00,Crystal Palace,W,NaN,NaN
2,2023-08-26 14:00:00+00:00,Fulham,D,NaN,NaN
3,2023-09-03 15:30:00+00:00,Manchester United,W,NaN,NaN
4,2023-09-17 15:30:00+00:00,Everton,W,NaN,NaN
5,2023-09-24 13:00:00+00:00,Tottenham Hotspur,D,13.0,5.0
6,2023-09-30 14:00:00+00:00,Bournemouth,W,11.0,4.0
7,2023-10-08 15:30:00+00:00,Manchester City,W,11.0,7.0
8,2023-10-21 16:30:00+00:00,Chelsea,D,13.0,8.0
9,2023-10-28 14:00:00+00:00,Sheffield United,W,11.0,6.0


In [48]:
# Adding a days of rest feature
# Useful because teams with few days between matches may be more fatigued.

team_matches["days_rest"] = (
    team_matches
    .groupby("team")["date"]
    .diff()
    .dt.total_seconds()
    / 86400
)

In [49]:
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "venue",
        "days_rest"
    ]
].head(10)

,date,opponent,venue,days_rest
0,2023-08-12 12:00:00+00:00,Nottingham Forest,home,NaN
1,2023-08-21 19:00:00+00:00,Crystal Palace,away,9.291667
2,2023-08-26 14:00:00+00:00,Fulham,home,4.791667
3,2023-09-03 15:30:00+00:00,Manchester United,home,8.062500
4,2023-09-17 15:30:00+00:00,Everton,away,14.000000
5,2023-09-24 13:00:00+00:00,Tottenham Hotspur,home,6.895833
6,2023-09-30 14:00:00+00:00,Bournemouth,away,6.041667
7,2023-10-08 15:30:00+00:00,Manchester City,home,8.062500
8,2023-10-21 16:30:00+00:00,Chelsea,away,13.041667
9,2023-10-28 14:00:00+00:00,Sheffield United,home,6.895833


In [50]:
# Rounding days of rest to look cleaner.

team_matches["days_rest"] = team_matches["days_rest"].round(1)

In [53]:
# Inspecting how a team performs home and away

team_matches["points_last3_same_venue"] = (
    team_matches
    # Means previous 3 matches at the same venue type
    .groupby(["team", "venue"])["points"]
    .transform(
        lambda x: x.shift(1).rolling(3).sum()
    )
)

In [81]:
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "venue",
        "points",
        "points_last5",
        "points_last3_same_venue"
    ]
].head(15)

,date,opponent,venue,points,points_last5,points_last3_same_venue
0,2023-08-12 12:00:00+00:00,Nottingham Forest,home,3,NaN,NaN
1,2023-08-21 19:00:00+00:00,Crystal Palace,away,3,NaN,NaN
2,2023-08-26 14:00:00+00:00,Fulham,home,1,NaN,NaN
3,2023-09-03 15:30:00+00:00,Manchester United,home,3,NaN,NaN
4,2023-09-17 15:30:00+00:00,Everton,away,3,NaN,NaN
5,2023-09-24 13:00:00+00:00,Tottenham Hotspur,home,1,13.0,7.0
6,2023-09-30 14:00:00+00:00,Bournemouth,away,3,11.0,NaN
7,2023-10-08 15:30:00+00:00,Manchester City,home,3,11.0,5.0
8,2023-10-21 16:30:00+00:00,Chelsea,away,1,13.0,9.0
9,2023-10-28 14:00:00+00:00,Sheffield United,home,3,11.0,7.0


In [129]:
# Identifying our feature columns that we want to include.

feature_columns = [
    "match_id",
    "team",
    "avg_goals_for_last5",
    "avg_goals_against_last5",
    "avg_xg_for_last5",
    "avg_xg_against_last5",
    "avg_shots_for_last5",
    "avg_shots_against_last5",
    "avg_shots_on_target_for_last5",
    "avg_shots_on_target_against_last5",
    "avg_possession_for_last5",
    "avg_possession_against_last5",
    "avg_passes_for_last5",
    "avg_passes_against_last5",
    "avg_corners_for_last5",
    "avg_corners_against_last5",
    "points_last5",
    "points_last10",
    "win_rate_last5",
    "goal_difference_last5",
    "xg_difference_last5",
    "points_last3_same_venue",
    "days_rest"
]

In [146]:
team_features = team_matches[feature_columns].copy()

team_features.shape

(2280, 23)

In [147]:
# Split these team-feature tows into home-team features and away-team features.

home_features = team_features[
    team_matches["venue"] == "home"
].copy()

away_features = team_features[
    team_matches["venue"] == "away"
].copy()

In [148]:
# Rename the columns so they're labeled for the final match-level dataset

home_features = home_features.rename(columns={
    "team": "home_team",
    "avg_goals_for_last5": "home_avg_goals_for_last5",
    "avg_goals_against_last5": "home_avg_goals_against_last5",
    "avg_xg_for_last5": "home_avg_xg_for_last5",
    "avg_xg_against_last5": "home_avg_xg_against_last5",
    "avg_shots_for_last5": "home_avg_shots_for_last5",
    "avg_shots_against_last5": "home_avg_shots_against_last5",
    "avg_shots_on_target_for_last5": "home_avg_shots_on_target_for_last5",
    "avg_shots_on_target_against_last5": "home_avg_shots_on_target_against_last5",
    "avg_possession_for_last5": "home_avg_possession_for_last5",
    "avg_possession_against_last5": "home_avg_possession_against_last5",
    "avg_passes_for_last5": "home_avg_passes_for_last5",
    "avg_passes_against_last5": "home_avg_passes_against_last5",
    "avg_corners_for_last5": "home_avg_corners_for_last5",
    "avg_corners_against_last5": "home_avg_corners_against_last5",
    "points_last5": "home_points_last5",
    "points_last10": "home_points_last10",
    "win_rate_last5": "home_win_rate_last5",
    "goal_difference_last5": "home_goal_difference_last5",
    "xg_difference_last5": "home_xg_difference_last5",
    "points_last3_same_venue": "home_points_last3_same_venue",
    "days_rest": "home_days_rest"
})

In [149]:
away_features = away_features.rename(columns={
    "team": "away_team",
    "avg_goals_for_last5": "away_avg_goals_for_last5",
    "avg_goals_against_last5": "away_avg_goals_against_last5",
    "avg_xg_for_last5": "away_avg_xg_for_last5",
    "avg_xg_against_last5": "away_avg_xg_against_last5",
    "avg_shots_for_last5": "away_avg_shots_for_last5",
    "avg_shots_against_last5": "away_avg_shots_against_last5",
    "avg_shots_on_target_for_last5": "away_avg_shots_on_target_for_last5",
    "avg_shots_on_target_against_last5": "away_avg_shots_on_target_against_last5",
    "avg_possession_for_last5": "away_avg_possession_for_last5",
    "avg_possession_against_last5": "away_avg_possession_against_last5",
    "avg_passes_for_last5": "away_avg_passes_for_last5",
    "avg_passes_against_last5": "away_avg_passes_against_last5",
    "avg_corners_for_last5": "away_avg_corners_for_last5",
    "avg_corners_against_last5": "away_avg_corners_against_last5",
    "points_last5": "away_points_last5",
    "points_last10": "away_points_last10",
    "win_rate_last5": "away_win_rate_last5",
    "goal_difference_last5": "away_goal_difference_last5",
    "xg_difference_last5": "away_xg_difference_last5",
    "points_last3_same_venue": "away_points_last3_same_venue",
    "days_rest": "away_days_rest"
})

In [150]:
print(home_features.shape)
print(away_features.shape)

(1140, 23)
(1140, 23)


In [151]:
# Merge both sides back onto the original match table.

model_df = historical_df.merge(
    home_features,
    on = ["match_id", "home_team"],
    how = "left"
)

model_df = model_df.merge(
    away_features,
    on = ["match_id", "away_team"],
    how = "left"
)

model_df.shape

# We started with 24 columns, and added 18 home-side features and 18 away-side features.

(1140, 66)

In [89]:
# Verify using one match.

model_df[
    [
        "date",
        "home_team",
        "away_team",
        "home_points_last5",
        "away_points_last5",
        "home_avg_xg_for_last5",
        "away_avg_xg_for_last5",
        "home_days_rest",
        "away_days_rest"
    ]
].head(10)

,date,home_team,away_team,home_points_last5,away_points_last5,home_avg_xg_for_last5,away_avg_xg_for_last5,home_days_rest,away_days_rest
0,2023-10-07 14:00:00+00:00,Manchester United,Brentford,6.0,3.0,1.400,1.414,7.0,6.0
1,2023-10-21 14:00:00+00:00,Newcastle United,Crystal Palace,10.0,8.0,2.356,1.058,13.0,13.9
2,2023-10-29 14:00:00+00:00,Liverpool,Nottingham Forest,10.0,4.0,2.288,1.244,8.1,8.0
3,2023-09-30 16:30:00+00:00,Tottenham Hotspur,Liverpool,13.0,15.0,1.932,2.396,6.1,6.1
4,2023-11-04 15:00:00+00:00,Brentford,West Ham United,7.0,4.0,1.808,1.234,7.1,6.1
5,2023-08-12 14:00:00+00:00,Sheffield United,Crystal Palace,NaN,NaN,NaN,NaN,NaN,NaN
6,2023-12-03 14:00:00+00:00,West Ham United,Crystal Palace,6.0,3.0,1.258,1.392,8.0,8.0
7,2023-12-07 19:30:00+00:00,Everton,Newcastle United,10.0,10.0,1.240,1.716,5.1,5.0
8,2024-04-20 14:00:00+00:00,Luton Town,Brentford,4.0,6.0,0.682,1.520,7.0,7.0
9,2023-12-10 14:00:00+00:00,Luton Town,Manchester City,4.0,6.0,0.580,1.882,4.7,3.7


In [90]:
# Sort chronologically

model_df = model_df.sort_values(
    "date"
).reset_index(drop = True)

model_df[
    [
        "date",
        "home_team",
        "away_team",
        "home_points_last5",
        "away_points_last5"
    ]
].head(15)

,date,home_team,away_team,home_points_last5,away_points_last5
0,2023-08-11 19:00:00+00:00,Burnley,Manchester City,NaN,NaN
1,2023-08-12 12:00:00+00:00,Arsenal,Nottingham Forest,NaN,NaN
2,2023-08-12 14:00:00+00:00,Everton,Fulham,NaN,NaN
3,2023-08-12 14:00:00+00:00,Brighton & Hove Albion,Luton Town,NaN,NaN
4,2023-08-12 14:00:00+00:00,Sheffield United,Crystal Palace,NaN,NaN
5,2023-08-12 14:00:00+00:00,Bournemouth,West Ham United,NaN,NaN
6,2023-08-12 16:30:00+00:00,Newcastle United,Aston Villa,NaN,NaN
7,2023-08-13 13:00:00+00:00,Brentford,Tottenham Hotspur,NaN,NaN
8,2023-08-13 15:30:00+00:00,Chelsea,Liverpool,NaN,NaN
9,2023-08-14 19:00:00+00:00,Manchester United,Wolverhampton,NaN,NaN


In [91]:
# Check the missing rolling features.

model_df[
    [
        "home_points_last5",
        "away_points_last5",
        "home_avg_xg_for_last5",
        "away_avg_xg_for_last5",
        "home_points_last3_same_venue",
        "away_points_last3_same_venue",
        "home_days_rest",
        "away_days_rest"
    ]
].isnull().sum()

home_points_last5               64
away_points_last5               61
home_avg_xg_for_last5           72
away_avg_xg_for_last5           69
home_points_last3_same_venue    75
away_points_last3_same_venue    75
home_days_rest                  14
away_days_rest                  11
dtype: int64

In [92]:
# Create a column including recent win rate

team_matches["win"] = (
    team_matches["result"] == "W"
).astype("Int64")

In [93]:
team_matches["win_rate_last5"] = (
    team_matches.groupby("team")["win"]
    .transform(
        lambda x: x.shift(1).rolling(5).mean()
    )
)

In [94]:
# Inspecting Arsenals' win rate

team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "result",
        "points_last5",
        "win_rate_last5"
    ]
].head(12)

,date,opponent,result,points_last5,win_rate_last5
0,2023-08-12 12:00:00+00:00,Nottingham Forest,W,NaN,NaN
1,2023-08-21 19:00:00+00:00,Crystal Palace,W,NaN,NaN
2,2023-08-26 14:00:00+00:00,Fulham,D,NaN,NaN
3,2023-09-03 15:30:00+00:00,Manchester United,W,NaN,NaN
4,2023-09-17 15:30:00+00:00,Everton,W,NaN,NaN
5,2023-09-24 13:00:00+00:00,Tottenham Hotspur,D,13.0,0.8
6,2023-09-30 14:00:00+00:00,Bournemouth,W,11.0,0.6
7,2023-10-08 15:30:00+00:00,Manchester City,W,11.0,0.6
8,2023-10-21 16:30:00+00:00,Chelsea,D,13.0,0.8
9,2023-10-28 14:00:00+00:00,Sheffield United,W,11.0,0.6


In [95]:
model_df[
    [
        "home_team",
        "away_team",
        "home_win_rate_last5",
        "away_win_rate_last5"
    ]
].head(10)

,home_team,away_team,home_win_rate_last5,away_win_rate_last5
0,Burnley,Manchester City,NaN,NaN
1,Arsenal,Nottingham Forest,NaN,NaN
2,Everton,Fulham,NaN,NaN
3,Brighton & Hove Albion,Luton Town,NaN,NaN
4,Sheffield United,Crystal Palace,NaN,NaN
5,Bournemouth,West Ham United,NaN,NaN
6,Newcastle United,Aston Villa,NaN,NaN
7,Brentford,Tottenham Hotspur,NaN,NaN
8,Chelsea,Liverpool,NaN,NaN
9,Manchester United,Wolverhampton,NaN,NaN


In [ ]:

model_df[
    [
        "date",
        "home_team",
        "away_team",
        "home_win_rate_last5",
        "away_win_rate_last5"
    ]
].dropna().head(10)

,date,home_team,away_team,home_win_rate_last5,away_win_rate_last5
49,2023-09-23 14:00:00+00:00,Crystal Palace,Fulham,0.4,0.4
50,2023-09-23 14:00:00+00:00,Manchester City,Nottingham Forest,1.0,0.4
52,2023-09-23 16:30:00+00:00,Brentford,Everton,0.2,0.0
54,2023-09-24 13:00:00+00:00,Chelsea,Aston Villa,0.2,0.6
55,2023-09-24 13:00:00+00:00,Arsenal,Tottenham Hotspur,0.8,0.8
56,2023-09-24 13:00:00+00:00,Liverpool,West Ham United,0.8,0.6
57,2023-09-24 13:00:00+00:00,Brighton & Hove Albion,Bournemouth,0.8,0.0
58,2023-09-24 15:30:00+00:00,Sheffield United,Newcastle United,0.0,0.4
59,2023-09-30 11:30:00+00:00,Aston Villa,Brighton & Hove Albion,0.8,0.8
60,2023-09-30 14:00:00+00:00,Bournemouth,Arsenal,0.0,0.6


In [98]:
# Look at points over previous 10 matches.
team_matches["points_last10"] = (
    team_matches
    .groupby("team")["points"]
    .transform(
        lambda x: x.shift(1).rolling(10).sum()
    )
)

In [99]:
# Testing

team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "result",
        "points_last5",
        "points_last10"
    ]
].head(15)

,date,opponent,result,points_last5,points_last10
0,2023-08-12 12:00:00+00:00,Nottingham Forest,W,NaN,NaN
1,2023-08-21 19:00:00+00:00,Crystal Palace,W,NaN,NaN
2,2023-08-26 14:00:00+00:00,Fulham,D,NaN,NaN
3,2023-09-03 15:30:00+00:00,Manchester United,W,NaN,NaN
4,2023-09-17 15:30:00+00:00,Everton,W,NaN,NaN
5,2023-09-24 13:00:00+00:00,Tottenham Hotspur,D,13.0,NaN
6,2023-09-30 14:00:00+00:00,Bournemouth,W,11.0,NaN
7,2023-10-08 15:30:00+00:00,Manchester City,W,11.0,NaN
8,2023-10-21 16:30:00+00:00,Chelsea,D,13.0,NaN
9,2023-10-28 14:00:00+00:00,Sheffield United,W,11.0,NaN


In [119]:
model_df[
    [
        "home_team",
        "away_team",
        "home_points_last10",
        "away_points_last10"
    ]
].dropna().head(10)

,home_team,away_team,home_points_last10,away_points_last10
4,Brentford,West Ham United,13.0,14.0
6,West Ham United,Crystal Palace,13.0,11.0
7,Everton,Newcastle United,16.0,23.0
8,Luton Town,Brentford,5.0,7.0
9,Luton Town,Manchester City,8.0,15.0
10,Brentford,Sheffield United,7.0,6.0
11,Liverpool,Fulham,21.0,11.0
12,Aston Villa,Chelsea,20.0,16.0
13,Wolverhampton,West Ham United,14.0,10.0
14,Sheffield United,West Ham United,8.0,20.0


In [121]:
# Create a feature for xG difference

team_matches["xg_difference"] = (
    team_matches["xg_for"] - team_matches["xg_against"]
)

In [122]:
# Look at previous 5 matches

team_matches["xg_difference_last5"] = (
    team_matches
    .groupby("team")["xg_difference"]
    .transform(
        lambda x: x.shift(1).rolling(5).sum()
    )
)

In [123]:
# Verify using Arsenal

team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "xg_for",
        "xg_against",
        "xg_difference",
        "xg_difference_last5"
    ]
].head(12)

,date,opponent,xg_for,xg_against,xg_difference,xg_difference_last5
0,2023-08-12 12:00:00+00:00,Nottingham Forest,0.83,1.18,-0.35,NaN
1,2023-08-21 19:00:00+00:00,Crystal Palace,2.02,1.01,1.01,NaN
2,2023-08-26 14:00:00+00:00,Fulham,3.20,0.56,2.64,NaN
3,2023-09-03 15:30:00+00:00,Manchester United,2.27,0.94,1.33,NaN
4,2023-09-17 15:30:00+00:00,Everton,1.02,0.34,0.68,NaN
5,2023-09-24 13:00:00+00:00,Tottenham Hotspur,2.03,1.46,0.57,5.31
6,2023-09-30 14:00:00+00:00,Bournemouth,3.43,0.65,2.78,6.23
7,2023-10-08 15:30:00+00:00,Manchester City,0.39,0.48,-0.09,8.00
8,2023-10-21 16:30:00+00:00,Chelsea,0.99,1.35,-0.36,5.27
9,2023-10-28 14:00:00+00:00,Sheffield United,2.77,0.03,2.74,3.58


In [136]:
model_df[
    [
        "home_team",
        "away_team",
        "home_xg_difference_last5",
        "away_xg_difference_last5"
    ]
].dropna().head(10)

,home_team,away_team,home_xg_difference_last5,away_xg_difference_last5
0,Manchester United,Brentford,-0.17,0.38
1,Newcastle United,Crystal Palace,7.23,-2.04
2,Liverpool,Nottingham Forest,5.03,0.16
3,Tottenham Hotspur,Liverpool,2.87,6.31
4,Brentford,West Ham United,2.08,-2.78
6,West Ham United,Crystal Palace,-1.13,-0.14
7,Everton,Newcastle United,-0.08,3.65
8,Luton Town,Brentford,-7.02,0.74
9,Luton Town,Manchester City,-9.53,2.15
10,Brentford,Sheffield United,0.25,-3.79


In [154]:
# Look at how many matches currently have a complete set of the engineered features.

engineered_features = [
    col for col in model_df.columns
    if (
        col.startswith("home_avg_")
        or col.startswith("away_avg_")
        or col.startswith("home_points_")
        or col.startswith("away_points_")
        or col.startswith("home_win_rate_")
        or col.startswith("away_win_rate_")
        or col.startswith("home_goal_difference_")
        or col.startswith("away_goal_difference_")
        or col.startswith("home_xg_difference_")
        or col.startswith("away_xg_difference_")
        or col in ["home_days_rest", "away_days_rest"]
    )
]

print("Number of engineered features:", len(engineered_features))

print(
    "Matches with complete features:",
    model_df[engineered_features].dropna().shape[0]
)

print(
    "Matches with at least one missing feature:",
    model_df[engineered_features].isnull().any(axis = 1).sum()
)

Number of engineered features: 42
Matches with complete features: 969
Matches with at least one missing feature: 171


In [138]:
# Check to see where these features are missing

model_df["has_missing_features"] = (
    model_df[engineered_features]
    .isnull()
    .any(axis = 1)
)

model_df.groupby("season")["has_missing_features"].agg(
    total_matches = "count",
    matches_with_missing = "sum"
)

,total_matches,matches_with_missing
season,,
2023_24,380,115
2024_25,380,33
2025_26,380,23


In [139]:
# Check to see why the most recent seasons have missing data

model_df[
    (model_df["has_missing_features"]) &
    (model_df["season"] == "2025_26")
][
    [
        "date",
        "matchday",
        "home_team",
        "away_team"
    ]
].sort_values("date")

,date,matchday,home_team,away_team
877,2025-08-16 14:00:00+00:00,1,Tottenham Hotspur,Burnley
1065,2025-08-16 14:00:00+00:00,1,Sunderland,West Ham United
1039,2025-08-18 19:00:00+00:00,1,Leeds United,Everton
851,2025-08-23 14:00:00+00:00,2,Burnley,Sunderland
932,2025-08-23 16:30:00+00:00,2,Arsenal,Leeds United
1008,2025-08-30 14:00:00+00:00,3,Manchester United,Burnley
1114,2025-08-30 14:00:00+00:00,3,Sunderland,Brentford
826,2025-08-30 16:30:00+00:00,3,Leeds United,Newcastle United
1047,2025-09-13 14:00:00+00:00,4,Fulham,Leeds United
990,2025-09-13 14:00:00+00:00,4,Crystal Palace,Sunderland


In [140]:
team_matches["days_rest"].describe()

count    2255.000000
mean        8.976231
std        13.988195
min         2.700000
25%         5.200000
50%         7.000000
75%         8.100000
max       454.000000
Name: days_rest, dtype: float64

In [ ]:
# Getting rid of summer breaks
# They increase the rest days variable drastically

team_matches[
    team_matches["days_rest"] > 30
][
    [
        "date",
        "season",
        "team",
        "opponent",
        "days_rest"
    ]
].head(20)

,date,season,team,opponent,days_rest
38,2024-08-17 14:00:00+00:00,2024_25,Arsenal,Wolverhampton,90.0
76,2025-08-17 15:30:00+00:00,2025_26,Arsenal,Manchester United,84.0
152,2024-08-17 16:30:00+00:00,2024_25,Aston Villa,West Ham United,90.1
190,2025-08-16 11:30:00+00:00,2025_26,Aston Villa,Newcastle United,82.9
266,2024-08-17 14:00:00+00:00,2024_25,Bournemouth,Nottingham Forest,90.0
304,2025-08-15 19:00:00+00:00,2025_26,Bournemouth,Liverpool,82.2
380,2024-08-18 13:00:00+00:00,2024_25,Brentford,Crystal Palace,90.9
418,2025-08-17 13:00:00+00:00,2025_26,Brentford,Nottingham Forest,83.9
494,2024-08-17 14:00:00+00:00,2024_25,Brighton & Hove Albion,Everton,90.0
532,2025-08-16 14:00:00+00:00,2025_26,Brighton & Hove Albion,Fulham,83.0


In [142]:
team_matches["days_rest"] = (
    team_matches["days_rest"]
    .clip(upper = 14)
)

In [143]:
team_matches["days_rest"].describe()

count    2255.000000
mean        7.456408
std         3.238459
min         2.700000
25%         5.200000
50%         7.000000
75%         8.100000
max        14.000000
Name: days_rest, dtype: float64

In [144]:
team_matches[
    team_matches["days_rest"] == 14
][
    [
        "date",
        "season",
        "team",
        "opponent",
        "days_rest"
    ]
].head(15)

,date,season,team,opponent,days_rest
4,2023-09-17 15:30:00+00:00,2023_24,Arsenal,Everton,14.0
12,2023-11-25 17:30:00+00:00,2023_24,Arsenal,Brentford,14.0
20,2024-01-20 12:30:00+00:00,2023_24,Arsenal,Crystal Palace,14.0
28,2024-03-31 15:30:00+00:00,2023_24,Arsenal,Manchester City,14.0
38,2024-08-17 14:00:00+00:00,2024_25,Arsenal,Wolverhampton,14.0
41,2024-09-15 13:00:00+00:00,2024_25,Arsenal,Tottenham Hotspur,14.0
45,2024-10-19 16:30:00+00:00,2024_25,Arsenal,Bournemouth,14.0
67,2025-04-01 18:45:00+00:00,2024_25,Arsenal,Fulham,14.0
76,2025-08-17 15:30:00+00:00,2025_26,Arsenal,Manchester United,14.0
83,2025-10-18 16:30:00+00:00,2025_26,Arsenal,Fulham,14.0


In [152]:
# Define the target and build a modeling table using our pre-match features

model_df["target"] = model_df["winner"].map({
    "home": 0,
    "draw": 1,
    "away": 2
})

In [153]:
model_df[["winner", "target"]].value_counts()

winner  target
home    0         492
away    2         369
draw    1         279
Name: count, dtype: int64

In [155]:
engineered_features = [
    col for col in model_df.columns
    if (
        col.startswith("home_avg_")
        or col.startswith("away_avg_")
        or col.startswith("home_points_")
        or col.startswith("away_points_")
        or col.startswith("home_win_rate_")
        or col.startswith("away_win_rate_")
        or col.startswith("home_goal_difference_")
        or col.startswith("away_goal_difference_")
        or col.startswith("home_xg_difference_")
        or col.startswith("away_xg_difference_")
        or col in ["home_days_rest", "away_days_rest"]
    )
]

print("Number of model features:", len(engineered_features))
print(engineered_features)

Number of model features: 42
['home_avg_goals_for_last5', 'home_avg_goals_against_last5', 'home_avg_xg_for_last5', 'home_avg_xg_against_last5', 'home_avg_shots_for_last5', 'home_avg_shots_against_last5', 'home_avg_shots_on_target_for_last5', 'home_avg_shots_on_target_against_last5', 'home_avg_possession_for_last5', 'home_avg_possession_against_last5', 'home_avg_passes_for_last5', 'home_avg_passes_against_last5', 'home_avg_corners_for_last5', 'home_avg_corners_against_last5', 'home_points_last5', 'home_points_last10', 'home_win_rate_last5', 'home_goal_difference_last5', 'home_xg_difference_last5', 'home_points_last3_same_venue', 'home_days_rest', 'away_avg_goals_for_last5', 'away_avg_goals_against_last5', 'away_avg_xg_for_last5', 'away_avg_xg_against_last5', 'away_avg_shots_for_last5', 'away_avg_shots_against_last5', 'away_avg_shots_on_target_for_last5', 'away_avg_shots_on_target_against_last5', 'away_avg_possession_for_last5', 'away_avg_possession_against_last5', 'away_avg_passes_for_l

In [156]:
# Creating a clean dataset
# Keeping matches where all 42 pre-match features and the target result are available.
model_ready_df = model_df.dropna(
    subset = engineered_features + ["target"]
).copy()

print("Original matches:", len(model_df))
print("Model-ready matches:", len(model_ready_df))
print(
    "Removed due to missing features:",
    len(model_df) - len(model_ready_df)
)

Original matches: 1140
Model-ready matches: 969
Removed due to missing features: 171


In [157]:
# Grouping matches by season, we don't want to train on 25/26 and test it on 24/25 that would be going backwards.
model_ready_df.groupby("season").size()

season
2023_24    265
2024_25    347
2025_26    357
dtype: int64

In [158]:
# Conduct a 63/37 train/test split.
# Use seasons 23/24 and 24/25 to predict season 25/26

train_df = model_ready_df[
    model_ready_df["season"].isin(["2023_24", "2024_25"])
].copy()

test_df = model_ready_df[
    model_ready_df["season"] == "2025_26"
].copy()

print("Training matches:", len(train_df))
print("Testing matches:", len(test_df))

Training matches: 612
Testing matches: 357


In [2]:
# Save the finished feature-engineered dataset

FEATURE_FILE = PROCESSED_DIR / "model_features.csv"

model_df.to_csv(
    FEATURE_FILE,
    index = False
)

print("Saved:", FEATURE_FILE)
print("Shape:", model_df.shape)

NameError: name 'model_df' is not defined

# FEATURE ENGINEERING PART 2

In [3]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

model_df = pd.read_csv(
    PROCESSED_DIR / "model_features.csv"
)

print(model_df.shape)

EmptyDataError: No columns to parse from file

In [3]:
# Adding matchup-difference features
# The models performed terribly in terms of predicting draw
# Having a differential between 2 teams might be a better relationship to look at

model_df["xg_form_difference"] = (
    model_df["home_avg_xg_for_last5"]
    - model_df["away_avg_xg_for_last5"]
)

In [4]:
# Inspect the DataFrame

model_df[
    [
        "home_team",
        "away_team",
        "home_avg_xg_for_last5",
        "away_avg_xg_for_last5",
        "xg_form_difference"
    ]
].dropna().head(10)

,home_team,away_team,home_avg_xg_for_last5,away_avg_xg_for_last5,xg_form_difference
0,Manchester United,Brentford,1.400,1.414,-0.014
1,Newcastle United,Crystal Palace,2.356,1.058,1.298
2,Liverpool,Nottingham Forest,2.288,1.244,1.044
3,Tottenham Hotspur,Liverpool,1.932,2.396,-0.464
4,Brentford,West Ham United,1.808,1.234,0.574
6,West Ham United,Crystal Palace,1.258,1.392,-0.134
7,Everton,Newcastle United,1.240,1.716,-0.476
8,Luton Town,Brentford,0.682,1.520,-0.838
9,Luton Town,Manchester City,0.580,1.882,-1.302
10,Brentford,Sheffield United,1.510,1.070,0.440


In [5]:
# Adding the focused matchup-difference features together.

model_df["points_form_difference"] = (
    model_df["home_points_last5"]
    - model_df["away_points_last5"]
)

model_df["goal_difference_form_difference"] = (
    model_df["home_goal_difference_last5"]
    - model_df["away_goal_difference_last5"]
)

model_df["xg_difference_form_difference"] = (
    model_df["home_xg_difference_last5"]
    - model_df["away_xg_difference_last5"]
)

model_df["shots_on_target_form_difference"] = (
    model_df["home_avg_shots_on_target_for_last5"]
    - model_df["away_avg_shots_on_target_for_last5"]
)

model_df["win_rate_difference"] = (
    model_df["home_win_rate_last5"]
    - model_df["away_win_rate_last5"]
)

In [6]:
# Inspect the DataFrame again
# None of this is new info from the match, they're just presenting our existing pre-match info in a way that directly describes the matchup.

model_df[
    [
        "home_team",
        "away_team",
        "xg_form_difference",
        "points_form_difference",
        "goal_difference_form_difference",
        "xg_difference_form_difference",
        "shots_on_target_form_difference",
        "win_rate_difference"
    ]
].dropna().head(10)

,home_team,away_team,xg_form_difference,points_form_difference,goal_difference_form_difference,xg_difference_form_difference,shots_on_target_form_difference,win_rate_difference
0,Manchester United,Brentford,-0.014,3.0,0.0,-0.55,1.2,0.4
1,Newcastle United,Crystal Palace,1.298,2.0,9.0,9.27,1.8,0.2
2,Liverpool,Nottingham Forest,1.044,6.0,7.0,4.87,1.2,0.6
3,Tottenham Hotspur,Liverpool,-0.464,-2.0,-2.0,-3.44,1.4,-0.2
4,Brentford,West Ham United,0.574,3.0,6.0,4.86,0.6,0.2
6,West Ham United,Crystal Palace,-0.134,3.0,2.0,-0.99,-0.8,0.2
7,Everton,Newcastle United,-0.476,0.0,-3.0,-3.73,0.0,0.0
8,Luton Town,Brentford,-0.838,-2.0,-7.0,-7.76,-0.2,0.0
9,Luton Town,Manchester City,-1.302,-2.0,-7.0,-11.68,-2.4,0.0
10,Brentford,Sheffield United,0.440,0.0,6.0,4.04,-0.6,0.0


In [17]:
# Saving the new features into the .csv file

FEATURE_FILE = PROCESSED_DIR / "model_features.csv"

model_df.to_csv(
    FEATURE_FILE,
    index = False
)

print("Saved:", FEATURE_FILE)
print("Shape:", model_df.shape)

Saved: ..\data\processed\model_features.csv
Shape: (1140, 73)


# FEATURE ENGINEERING PART 3

In [18]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

historical_df = pd.read_csv(
    PROCESSED_DIR / "historical_matches.csv"
)

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True
)

model_df = pd.read_csv(
    PROCESSED_DIR / "model_features.csv"
)

print("Historical:", historical_df.shape)
print("Model V2:", model_df.shape)

Historical: (1140, 24)
Model V2: (1140, 73)


In [19]:
# Giving a team an ELO rating might help the model properly predict matches
# A team having a higher ELO (Arsenal) will have a higher chance of beating a team with a lower ELO rating (Burnley)

# Sort all matches in chronological order
elo_matches = historical_df.sort_values(
    ["date", "match_id"]
).reset_index(drop = True)

print(elo_matches.shape)

elo_matches[
    ["date", "season", "home_team", "away_team", "winner"]
].head()

(1140, 24)


,date,season,home_team,away_team,winner
0,2023-08-11 19:00:00+00:00,2023_24,Burnley,Manchester City,away
1,2023-08-12 12:00:00+00:00,2023_24,Arsenal,Nottingham Forest,home
2,2023-08-12 14:00:00+00:00,2023_24,Sheffield United,Crystal Palace,away
3,2023-08-12 14:00:00+00:00,2023_24,Bournemouth,West Ham United,draw
4,2023-08-12 14:00:00+00:00,2023_24,Everton,Fulham,away


In [21]:
# Create the ELO calculation

# Starting point
INITIAL_ELO = 1500

# Controls how quickly ratings change
K_FACTOR = 20

# Gives the home team an ELO boost when calculating the expected result
# AKA home field advantage
HOME_ADVANTAGE = 100

elo_ratings = {}
elo_records = []

for _, match in elo_matches.iterrows():
    home_team = match["home_team"]
    away_team = match["away_team"]

    # Give a new team an initial rating of 1500
    home_elo = elo_ratings.get(home_team, INITIAL_ELO)
    away_elo = elo_ratings.get(away_team, INITIAL_ELO)

    # Store Pre-match ratings
    # Happens before the result updates the ratings, preventing daka leakage
    elo_records.append({
        "match_id": match["match_id"],
        "home_elo": home_elo,
        "away_elo": away_elo,
        "elo_difference": home_elo - away_elo
    })

    # Expected result including home advantage
    expected_home = 1 / (
        1 + 10 ** (
            (away_elo - (home_elo + HOME_ADVANTAGE)) / 400
        )
    )

    expected_away = 1 - expected_home

    # Actual result
    if match["winner"] == "home":
        actual_home = 1.0
        actual_away = 0.0

    elif match["winner"] == "away":
        actual_home = 0.0
        actual_away = 1.0

    else:
        actual_home = 0.5
        actual_away = 0.5
    
    # Update ratings after storing pre-match ELO
    elo_ratings[home_team] = (
        home_elo
        + K_FACTOR * (actual_home - expected_home)
    )

    elo_ratings[away_team] = (
        away_elo
        + K_FACTOR * (actual_away - expected_away)
    )

In [22]:
# Turn this into a DataFrame

elo_df = pd.DataFrame(elo_records)

print(elo_df.shape)
elo_df.head(10)

(1140, 4)


,match_id,home_elo,away_elo,elo_difference
0,mt_581188156,1500.0,1500.0,0.0
1,mt_250055055,1500.0,1500.0,0.0
2,mt_012211213,1500.0,1500.0,0.0
3,mt_195599593,1500.0,1500.0,0.0
4,mt_581188182,1500.0,1500.0,0.0
5,mt_978877878,1500.0,1500.0,0.0
6,mt_406600601,1500.0,1500.0,0.0
7,mt_743344370,1500.0,1500.0,0.0
8,mt_624422383,1500.0,1500.0,0.0
9,mt_367766802,1500.0,1500.0,0.0


In [23]:
# Make sure the ratings start changing after teams play

# This shows Arsenal's first 10 matches chronologically
# We check that Arsenal's first match ELO rating starts at 1500
# ELO changes in later matches based on previous results.
# The opponent also has a changing ELO
# Values shown for a match are still the ratings before that match result
elo_check = elo_matches[
    ["match_id", "date", "home_team", "away_team", "winner"]
].merge(
    elo_df,
    on = "match_id",
    how = "left"
)

elo_check[
    (elo_check["home_team"] == "Arsenal")
    | (elo_check["away_team"] == "Arsenal")
][
    [
        "date",
        "home_team",
        "away_team",
        "winner",
        "home_elo",
        "away_elo",
        "elo_difference"
    ]
].head(10)

,date,home_team,away_team,winner,home_elo,away_elo,elo_difference
1,2023-08-12 12:00:00+00:00,Arsenal,Nottingham Forest,home,1500.000000,1500.000000,0.000000
18,2023-08-21 19:00:00+00:00,Crystal Palace,Arsenal,away,1512.801300,1507.198700,5.602600
22,2023-08-26 14:00:00+00:00,Arsenal,Fulham,draw,1520.147921,1499.591574,20.556346
38,2023-09-03 15:30:00+00:00,Arsenal,Manchester United,home,1516.810895,1507.080837,9.730058
47,2023-09-17 15:30:00+00:00,Everton,Arsenal,away,1470.182774,1523.753593,-53.570819
55,2023-09-24 13:00:00+00:00,Arsenal,Tottenham Hotspur,draw,1535.082034,1539.379125,-4.297091
61,2023-09-30 14:00:00+00:00,Bournemouth,Arsenal,away,1472.721343,1532.395099,-59.673756
79,2023-10-08 15:30:00+00:00,Arsenal,Manchester City,home,1543.550597,1547.286046,-3.735448
86,2023-10-21 16:30:00+00:00,Chelsea,Arsenal,draw,1500.783498,1550.848671,-50.065173
92,2023-10-28 14:00:00+00:00,Arsenal,Sheffield United,home,1552.276095,1425.392403,126.883693


In [24]:
# Merge ELO into V2

model_df = model_df.merge(
    elo_df,
    on = "match_id",
    how = "left"
)

print(model_df.shape)

print(
    model_df[
        ["home_elo", "away_elo", "elo_difference"]
    ].isna().sum()
)

(1140, 76)
home_elo          0
away_elo          0
elo_difference    0
dtype: int64


In [26]:
# Make sure ELO is carrying across seasons 
season_transition_check = model_df[
    model_df["season"].isin(["2023_24", "2024_25"])
][
    [
        "date",
        "season",
        "home_team",
        "away_team",
        "home_elo",
        "away_elo"
    ]
].sort_values("date")

print("Last 5 matchs of 2023/24:")
display(
    season_transition_check[
        season_transition_check["season"] == "2023_24"
    ].tail()
)

print("\nFirst 5 matches of 2024/25:")
display(
    season_transition_check[
        season_transition_check["season"] == "2024_25"
    ].head()
)

Last 5 matchs of 2023/24:


,date,season,home_team,away_team,home_elo,away_elo
330,2024-05-19 15:00:00+00:00,2023_24,Luton Town,Fulham,1391.177398,1466.296387
272,2024-05-19 15:00:00+00:00,2023_24,Brentford,Newcastle United,1455.045602,1515.882920
22,2024-05-19 15:00:00+00:00,2023_24,Sheffield United,Tottenham Hotspur,1344.511045,1529.547391
328,2024-05-19 15:00:00+00:00,2023_24,Chelsea,Bournemouth,1558.703654,1487.339967
374,2024-05-19 15:00:00+00:00,2023_24,Brighton & Hove Albion,Manchester United,1484.476067,1516.895953



First 5 matches of 2024/25:


,date,season,home_team,away_team,home_elo,away_elo
440,2024-08-16 19:00:00+00:00,2024_25,Manchester United,Fulham,1528.816900,1477.011298
496,2024-08-17 11:30:00+00:00,2024_25,Ipswich Town,Liverpool,1500.000000,1617.795533
741,2024-08-17 14:00:00+00:00,2024_25,Arsenal,Wolverhampton,1655.767556,1457.244866
560,2024-08-17 14:00:00+00:00,2024_25,Nottingham Forest,Bournemouth,1431.868792,1481.907685
597,2024-08-17 14:00:00+00:00,2024_25,Everton,Brighton & Hove Albion,1488.652871,1472.555120


In [27]:
# Save V3
FEATURE_FILE = PROCESSED_DIR / "model_features.csv"

model_df.to_csv(
    FEATURE_FILE,
    index = False
)

print("Saved:", FEATURE_FILE)
print("Shape:", model_df.shape)
print("File size:", FEATURE_FILE.stat().st_size)

Saved: ..\data\processed\model_features.csv
Shape: (1140, 76)
File size: 556561


# FEATURE ENGINEERING PART 4

In [28]:
# ELO worked but it could be "more in depth"
# Offseason regression towards 1500
# Lower starting ELO for promoted teams

def build_elo_features(
    matches,
    initial_elo=1500,
    promoted_elo=1450,
    k_factor=20,
    home_advantage=100,
    offseason_regression=0.20
):
    matches = matches.sort_values(
        ["date", "match_id"]
    ).reset_index(drop=True)

    elo_ratings = {}
    team_last_season = {}
    elo_records = []

    for _, match in matches.iterrows():

        season = match["season"]
        home_team = match["home_team"]
        away_team = match["away_team"]

        # HOME TEAM
        if home_team not in elo_ratings:
            home_elo = promoted_elo
            elo_ratings[home_team] = home_elo

        else:
            home_elo = elo_ratings[home_team]

            # First appearance in a new season:
            # regress rating toward 1500
            if team_last_season.get(home_team) != season:
                home_elo = (
                    home_elo
                    + offseason_regression
                    * (initial_elo - home_elo)
                )

                elo_ratings[home_team] = home_elo

        # AWAY TEAM
        if away_team not in elo_ratings:
            away_elo = promoted_elo
            elo_ratings[away_team] = away_elo

        else:
            away_elo = elo_ratings[away_team]

            if team_last_season.get(away_team) != season:
                away_elo = (
                    away_elo
                    + offseason_regression
                    * (initial_elo - away_elo)
                )

                elo_ratings[away_team] = away_elo

        # Store PRE-MATCH Elo
        elo_records.append({
            "match_id": match["match_id"],
            "home_elo_v4": home_elo,
            "away_elo_v4": away_elo,
            "elo_difference_v4": home_elo - away_elo
        })

        # Expected outcome
        expected_home = 1 / (
            1 + 10 ** (
                (away_elo - (home_elo + home_advantage)) / 400
            )
        )

        expected_away = 1 - expected_home

        # Actual outcome
        if match["winner"] == "home":
            actual_home = 1.0
            actual_away = 0.0

        elif match["winner"] == "away":
            actual_home = 0.0
            actual_away = 1.0

        else:
            actual_home = 0.5
            actual_away = 0.5

        # Update AFTER match
        elo_ratings[home_team] = (
            home_elo
            + k_factor * (actual_home - expected_home)
        )

        elo_ratings[away_team] = (
            away_elo
            + k_factor * (actual_away - expected_away)
        )

        team_last_season[home_team] = season
        team_last_season[away_team] = season

    return pd.DataFrame(elo_records)

# In this version, any team appearing for the first time in the dataset gets 1450.

In [29]:
elo_v4_df = build_elo_features(
    historical_df,
    promoted_elo = 1450,
    offseason_regression = 0.20
)

print(elo_v4_df.shape)
elo_v4_df.head()

(1140, 4)


,match_id,home_elo_v4,away_elo_v4,elo_difference_v4
0,mt_581188156,1450.0,1450.0,0.0
1,mt_250055055,1450.0,1450.0,0.0
2,mt_012211213,1450.0,1450.0,0.0
3,mt_195599593,1450.0,1450.0,0.0
4,mt_581188182,1450.0,1450.0,0.0


In [30]:
# Verify the offseason regression and new team 1450 rating are working

elo_v4_check = historical_df[
    [
        "match_id",
        "date",
        "season",
        "home_team",
        "away_team"
    ]
].merge(
    elo_v4_df,
    on = "match_id",
    how = "left"
)

elo_v4_check[
    elo_v4_check["season"] == "2024_25"
][
    [
        "date",
        "home_team",
        "away_team",
        "home_elo_v4",
        "away_elo_v4",
        "elo_difference_v4"
    ]
].sort_values("date").head(10)

,date,home_team,away_team,home_elo_v4,away_elo_v4,elo_difference_v4
440,2024-08-16 19:00:00+00:00,Manchester United,Fulham,1483.053520,1441.609039,41.444481
496,2024-08-17 11:30:00+00:00,Ipswich Town,Liverpool,1450.000000,1554.236426,-104.236426
560,2024-08-17 14:00:00+00:00,Nottingham Forest,Bournemouth,1405.495034,1445.526148,-40.031114
741,2024-08-17 14:00:00+00:00,Arsenal,Wolverhampton,1584.614045,1425.795892,158.818152
597,2024-08-17 14:00:00+00:00,Everton,Brighton & Hove Albion,1450.922297,1438.044096,12.878200
397,2024-08-17 14:00:00+00:00,Newcastle United,Southampton,1481.604291,1450.000000,31.604291
498,2024-08-17 16:30:00+00:00,West Ham United,Aston Villa,1451.517359,1501.176921,-49.659561
644,2024-08-18 13:00:00+00:00,Brentford,Crystal Palace,1415.138527,1463.363241,-48.224714
441,2024-08-18 15:30:00+00:00,Chelsea,Manchester City,1511.308749,1595.618932,-84.310183
497,2024-08-19 19:00:00+00:00,Leicester City,Tottenham Hotspur,1450.000000,1489.718062,-39.718062


In [40]:
# Don't just assume 20% regression towards 1500 is the best option
# Take a look at several other possible values and train it with historical seasons
# Same with promoted teams ELO

promoted_elo_values = [
    1200,
    1225,
    1250,
    1275,
    1300
]

offseason_regression_values = [
    0.35,
    0.40,
    0.45,
    0.50
]

elo_configurations = {}

for promoted_elo in promoted_elo_values:
    for regression in offseason_regression_values:

        config_name = f"elo_{promoted_elo}_reg_{regression}"

        elo_configurations[config_name] = build_elo_features(
            historical_df,
            promoted_elo = promoted_elo,
            offseason_regression = regression
        )

print("Number of configurations:", len(elo_configurations))
print(list(elo_configurations.keys()))

Number of configurations: 20
['elo_1200_reg_0.35', 'elo_1200_reg_0.4', 'elo_1200_reg_0.45', 'elo_1200_reg_0.5', 'elo_1225_reg_0.35', 'elo_1225_reg_0.4', 'elo_1225_reg_0.45', 'elo_1225_reg_0.5', 'elo_1250_reg_0.35', 'elo_1250_reg_0.4', 'elo_1250_reg_0.45', 'elo_1250_reg_0.5', 'elo_1275_reg_0.35', 'elo_1275_reg_0.4', 'elo_1275_reg_0.45', 'elo_1275_reg_0.5', 'elo_1300_reg_0.35', 'elo_1300_reg_0.4', 'elo_1300_reg_0.45', 'elo_1300_reg_0.5']


In [41]:
engineered_features = [
    col for col in model_df.columns
    if (
        col.startswith("home_avg_")
        or col.startswith("away_avg_")
        or col.startswith("home_points_")
        or col.startswith("away_points_")
        or col.startswith("home_win_rate_")
        or col.startswith("away_win_rate_")
        or col.startswith("home_goal_difference_")
        or col.startswith("away_goal_difference_")
        or col.startswith("home_xg_difference_")
        or col.startswith("away_xg_difference_")
        or col in ["home_days_rest", "away_days_rest"]
    )
]

differential_features = [
    "xg_form_difference",
    "points_form_difference",
    "goal_difference_form_difference",
    "xg_difference_form_difference",
    "shots_on_target_form_difference",
    "win_rate_difference"
]

engineered_features_v2 = engineered_features + differential_features

print("V1 features:", len(engineered_features))
print("V2 features:", len(engineered_features_v2))

V1 features: 42
V2 features: 48


In [42]:
# Score the configurations by using 2023/24 for training and 2024/25 for validation
# Use RF structure and same V2 features since the ELO settings are being tested
# Essentially trainining 20 Random Forest models to see which configuration fits best.

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, log_loss

validation_results = []

for config_name, elo_config_df in elo_configurations.items():

    # Merge this Elo configuration with our V2 dataset
    temp_df = model_df.drop(
        columns=[
            "home_elo",
            "away_elo",
            "elo_difference"
        ],
        errors="ignore"
    ).merge(
        elo_config_df,
        on="match_id",
        how="left"
    )

    # V4 Elo column names
    elo_features_temp = [
        "home_elo_v4",
        "away_elo_v4",
        "elo_difference_v4"
    ]

    features_temp = (
        engineered_features_v2
        + elo_features_temp
    )

    # Only complete rows
    temp_ready = temp_df.dropna(
        subset=features_temp + ["target"]
    ).copy()

    # 2023/24 = training
    train_temp = temp_ready[
        temp_ready["season"] == "2023_24"
    ]

    # 2024/25 = validation
    validation_temp = temp_ready[
        temp_ready["season"] == "2024_25"
    ]

    X_train_temp = train_temp[features_temp]
    y_train_temp = train_temp["target"]

    X_val_temp = validation_temp[features_temp]
    y_val_temp = validation_temp["target"]

    # Same Random Forest settings
    temp_model = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    temp_model.fit(
        X_train_temp,
        y_train_temp
    )

    val_pred = temp_model.predict(X_val_temp)
    val_proba = temp_model.predict_proba(X_val_temp)

    validation_results.append({
        "configuration": config_name,
        "accuracy": accuracy_score(
            y_val_temp,
            val_pred
        ),
        "log_loss": log_loss(
            y_val_temp,
            val_proba
        )
    })

In [43]:
# Create a list showing which configuration had the lower log loss

validation_results_df = pd.DataFrame(
    validation_results
).sort_values(
    "log_loss"
).reset_index(drop = True)

validation_results_df

,configuration,accuracy,log_loss
0,elo_1200_reg_0.5,0.507205,1.022877
1,elo_1250_reg_0.5,0.501441,1.023563
2,elo_1225_reg_0.5,0.501441,1.023598
3,elo_1200_reg_0.4,0.504323,1.023715
4,elo_1225_reg_0.45,0.501441,1.023892
5,elo_1200_reg_0.45,0.504323,1.023951
6,elo_1225_reg_0.4,0.501441,1.024725
7,elo_1275_reg_0.5,0.504323,1.025090
8,elo_1250_reg_0.45,0.504323,1.025252
9,elo_1250_reg_0.4,0.498559,1.025487


In [44]:
# Build the V4 ELO

elo_v4_final = build_elo_features(
    historical_df,
    promoted_elo = 1200,
    offseason_regression = 0.50
)

print(elo_v4_final.shape)
elo_v4_final.head()

(1140, 4)


,match_id,home_elo_v4,away_elo_v4,elo_difference_v4
0,mt_581188156,1200.0,1200.0,0.0
1,mt_250055055,1200.0,1200.0,0.0
2,mt_012211213,1200.0,1200.0,0.0
3,mt_195599593,1200.0,1200.0,0.0
4,mt_581188182,1200.0,1200.0,0.0


In [45]:
# Merge into model_df

model_df = model_df.merge(
    elo_v4_final,
    on = "match_id",
    how = "left"
)

print(model_df.shape)

print(
    model_df[
        [
            "home_elo",
            "away_elo",
            "elo_difference",
            "home_elo_v4",
            "away_elo_v4",
            "elo_difference_v4"
        ]
    ].isna().sum()
)

(1140, 79)
home_elo             0
away_elo             0
elo_difference       0
home_elo_v4          0
away_elo_v4          0
elo_difference_v4    0
dtype: int64


In [46]:
# Save the new feature file

FEATURE_FILE = PROCESSED_DIR / "model_features.csv"

model_df.to_csv(
    FEATURE_FILE,
    index = False
)

print("Saved:", FEATURE_FILE)
print("Shape:", model_df.shape)

Saved: ..\data\processed\model_features.csv
Shape: (1140, 79)


# FEATURE ENGINEERING PART 5

In [47]:
elo_v4b = build_elo_features(
    historical_df,
    promoted_elo=1400,
    offseason_regression=0.25
)

elo_v4b = elo_v4b.rename(columns={
    "home_elo_v4": "home_elo_v4b",
    "away_elo_v4": "away_elo_v4b",
    "elo_difference_v4": "elo_difference_v4b"
})

print(elo_v4b.shape)
elo_v4b.head()

(1140, 4)


,match_id,home_elo_v4b,away_elo_v4b,elo_difference_v4b
0,mt_581188156,1400.0,1400.0,0.0
1,mt_250055055,1400.0,1400.0,0.0
2,mt_012211213,1400.0,1400.0,0.0
3,mt_195599593,1400.0,1400.0,0.0
4,mt_581188182,1400.0,1400.0,0.0


In [48]:
# Merge V4b into model_df

model_df = model_df.merge(
    elo_v4b,
    on = "match_id",
    how = "left"
)

print(model_df.shape)

print(
    model_df[
        [
            "home_elo_v4b",
            "away_elo_v4b",
            "elo_difference_v4b"
        ]
    ].isna().sum()
)

(1140, 82)
home_elo_v4b          0
away_elo_v4b          0
elo_difference_v4b    0
dtype: int64


In [49]:
# Save

FEATURE_FILE = PROCESSED_DIR / "model_features.csv"

model_df.to_csv(
    FEATURE_FILE,
    index = False
)

print("Saved:", FEATURE_FILE)
print("Shape:", model_df.shape)

Saved: ..\data\processed\model_features.csv
Shape: (1140, 82)


# FEATURE ENGINEERING PART 6

In [50]:
# Do same process, with different configuration

elo_v4c = build_elo_features(
    historical_df,
    promoted_elo=1300,
    offseason_regression=0.20
)

elo_v4c = elo_v4c.rename(columns={
    "home_elo_v4": "home_elo_v4c",
    "away_elo_v4": "away_elo_v4c",
    "elo_difference_v4": "elo_difference_v4c"
})

print(elo_v4c.shape)
elo_v4c.head()

(1140, 4)


,match_id,home_elo_v4c,away_elo_v4c,elo_difference_v4c
0,mt_581188156,1300.0,1300.0,0.0
1,mt_250055055,1300.0,1300.0,0.0
2,mt_012211213,1300.0,1300.0,0.0
3,mt_195599593,1300.0,1300.0,0.0
4,mt_581188182,1300.0,1300.0,0.0


In [51]:
model_df = model_df.merge(
    elo_v4c,
    on="match_id",
    how="left"
)

print(model_df.shape)

print(
    model_df[
        [
            "home_elo_v4c",
            "away_elo_v4c",
            "elo_difference_v4c"
        ]
    ].isna().sum()
)

(1140, 85)
home_elo_v4c          0
away_elo_v4c          0
elo_difference_v4c    0
dtype: int64


In [52]:
FEATURE_FILE = PROCESSED_DIR / "model_features.csv"

model_df.to_csv(
    FEATURE_FILE,
    index=False
)

print("Saved:", FEATURE_FILE)
print("Shape:", model_df.shape)

Saved: ..\data\processed\model_features.csv
Shape: (1140, 85)


# INCORPORATING ALL LEAGUES 

In [53]:
# Load new historical dataset

import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

historical_df = pd.read_csv(
    PROCESSED_DIR / "historical_matches.csv"
)

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True
)

print("Shape:", historical_df.shape)
print()
print(historical_df.dtypes)

Shape: (10653, 22)

match_id                                str
date                    datetime64[us, UTC]
season                                  str
competition                             str
matchday                            float64
home_team                               str
away_team                               str
home_goals                            int64
away_goals                            int64
home_xg                             float64
away_xg                             float64
home_shots                          float64
away_shots                          float64
home_shots_on_target                float64
away_shots_on_target                float64
home_possession                     float64
away_possession                     float64
home_passes                         float64
away_passes                         float64
home_corners                        float64
away_corners                        float64
winner                                  str
dtype: objec

In [54]:
# Make sure matches are in chronological order across all competitions

historical_df = historical_df.sort_values(
    ["date", "match_id"]
).reset_index(drop = True)

print(historical_df[[
    "date",
    "competition",
    "home_team",
    "away_team"
]].head(10))

                       date       competition         home_team  \
0 2020-08-08 13:00:00+00:00  champions_league      SP Tre Fiori   
1 2020-08-08 17:30:00+00:00  champions_league          FC Drita   
2 2020-08-11 16:00:00+00:00  champions_league          FC Drita   
3 2020-08-18 16:00:00+00:00  champions_league           Qarabağ   
4 2020-08-18 16:00:00+00:00     europa_league  Lincoln Red Imps   
5 2020-08-18 17:00:00+00:00  champions_league    Legia Warszawa   
6 2020-08-18 18:00:00+00:00     europa_league      UE Engordany   
7 2020-08-18 18:00:00+00:00  champions_league      Dynamo Brest   
8 2020-08-18 18:45:00+00:00  champions_league            Celtic   
9 2020-08-18 19:00:00+00:00  champions_league  FK Crvena zvezda   

               away_team  
0            Linfield FC  
1  Inter Club d'Escaldes  
2            Linfield FC  
3      FK Sileks Kratovo  
4           FC Prishtina  
5            Linfield FC  
6      FK Zeta Golubovci  
7                 Astana  
8           KR Reyk

In [55]:
# Convert match-level data into team-level rows.
# Each match will eventually become two rows: home team's perspective and away team's perspective

home_rows = historical_df[[
    "match_id",
    "date",
    "season",
    "competition",
    "matchday",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "home_xg",
    "away_xg",
    "home_shots",
    "away_shots",
    "home_shots_on_target",
    "away_shots_on_target",
    "home_possession",
    "away_possession",
    "home_passes",
    "away_passes",
    "home_corners",
    "away_corners"
]].copy()

home_rows = home_rows.rename(columns={
    "home_team": "team",
    "away_team": "opponent",
    "home_goals": "goals_for",
    "away_goals": "goals_against",
    "home_xg": "xg_for",
    "away_xg": "xg_against",
    "home_shots": "shots_for",
    "away_shots": "shots_against",
    "home_shots_on_target": "shots_on_target_for",
    "away_shots_on_target": "shots_on_target_against",
    "home_possession": "possession_for",
    "away_possession": "possession_against",
    "home_passes": "passes_for",
    "away_passes": "passes_against",
    "home_corners": "corners_for",
    "away_corners": "corners_against"
})

home_rows["venue"] = "home"

print(home_rows.shape)
print()
print(home_rows.head())

(10653, 22)

       match_id                      date   season       competition  \
0  mt_304172028 2020-08-08 13:00:00+00:00  2020_21  champions_league   
1  mt_847096460 2020-08-08 17:30:00+00:00  2020_21  champions_league   
2  mt_138668638 2020-08-11 16:00:00+00:00  2020_21  champions_league   
3  mt_304127364 2020-08-18 16:00:00+00:00  2020_21  champions_league   
4  mt_579499767 2020-08-18 16:00:00+00:00  2020_21     europa_league   

   matchday              team               opponent  goals_for  \
0       2.0      SP Tre Fiori            Linfield FC          0   
1       2.0          FC Drita  Inter Club d'Escaldes          2   
2       1.0          FC Drita            Linfield FC          0   
3      14.0           Qarabağ      FK Sileks Kratovo          4   
4      14.0  Lincoln Red Imps           FC Prishtina          3   

   goals_against  xg_for  ...  shots_against  shots_on_target_for  \
0              2     NaN  ...            NaN                  NaN   
1            

In [56]:
away_rows = historical_df[[
    "match_id",
    "date",
    "season",
    "competition",
    "matchday",
    "away_team",
    "home_team",
    "away_goals",
    "home_goals",
    "away_xg",
    "home_xg",
    "away_shots",
    "home_shots",
    "away_shots_on_target",
    "home_shots_on_target",
    "away_possession",
    "home_possession",
    "away_passes",
    "home_passes",
    "away_corners",
    "home_corners"
]].copy()

away_rows = away_rows.rename(columns={
    "away_team": "team",
    "home_team": "opponent",
    "away_goals": "goals_for",
    "home_goals": "goals_against",
    "away_xg": "xg_for",
    "home_xg": "xg_against",
    "away_shots": "shots_for",
    "home_shots": "shots_against",
    "away_shots_on_target": "shots_on_target_for",
    "home_shots_on_target": "shots_on_target_against",
    "away_possession": "possession_for",
    "home_possession": "possession_against",
    "away_passes": "passes_for",
    "home_passes": "passes_against",
    "away_corners": "corners_for",
    "home_corners": "corners_against"
})

away_rows["venue"] = "away"

print(away_rows.shape)
print()
print(away_rows.head())

(10653, 22)

       match_id                      date   season       competition  \
0  mt_304172028 2020-08-08 13:00:00+00:00  2020_21  champions_league   
1  mt_847096460 2020-08-08 17:30:00+00:00  2020_21  champions_league   
2  mt_138668638 2020-08-11 16:00:00+00:00  2020_21  champions_league   
3  mt_304127364 2020-08-18 16:00:00+00:00  2020_21  champions_league   
4  mt_579499767 2020-08-18 16:00:00+00:00  2020_21     europa_league   

   matchday                   team          opponent  goals_for  \
0       2.0            Linfield FC      SP Tre Fiori          2   
1       2.0  Inter Club d'Escaldes          FC Drita          1   
2       1.0            Linfield FC          FC Drita          3   
3      14.0      FK Sileks Kratovo           Qarabağ          0   
4      14.0           FC Prishtina  Lincoln Red Imps          0   

   goals_against  xg_for  ...  shots_against  shots_on_target_for  \
0              0     NaN  ...            NaN                  NaN   
1            

In [ ]:
# Combine home and away teams into full team-level dataset
# Gives one row per team per match, best structure to use when dealing with rolling form, rest days, congestion, and ELO

team_matches = pd.concat(
    [home_rows, away_rows],
    ignore_index = True
)

team_matches = team_matches.sort_values(
    ["team", "date", "match_id"]
).reset_index(drop = True)

print("Shape:", team_matches.shape)
print()
print(team_matches.head(10))

Shape: (21306, 22)

       match_id                      date   season        competition  \
0  mt_745711261 2024-08-22 17:00:00+00:00  2024_25  conference_league   
1  mt_408499521 2024-08-29 18:30:00+00:00  2024_25  conference_league   
2  mt_583674867 2024-10-03 16:45:00+00:00  2024_25  conference_league   
3  mt_625819314 2024-10-24 19:00:00+00:00  2024_25  conference_league   
4  mt_019783278 2024-11-07 20:00:00+00:00  2024_25  conference_league   
5  mt_197436387 2024-11-28 17:45:00+00:00  2024_25  conference_league   
6  mt_362501063 2024-12-12 17:45:00+00:00  2024_25  conference_league   
7  mt_836240326 2024-12-19 20:00:00+00:00  2024_25  conference_league   
8  mt_019020233 2025-02-13 20:00:00+00:00  2024_25  conference_league   
9  mt_401464799 2025-02-20 17:45:00+00:00  2024_25  conference_league   

   matchday              team               opponent  goals_for  \
0       NaN  1. FC Heidenheim              BK Häcken          2   
1       NaN  1. FC Heidenheim             

In [58]:
# Add simplest pre-match features first
# Result, points, GD, rest days

team_matches["result"] = np.select(
    [
        team_matches["goals_for"] > team_matches["goals_against"],
        team_matches["goals_for"] == team_matches["goals_against"]
    ],
    [
        "W",
        "D"
    ],
    default="L"
)

team_matches["points"] = np.select(
    [
        team_matches["result"] == "W",
        team_matches["result"] == "D"
    ],
    [
        3,
        1
    ],
    default=0
)

team_matches["goal_difference"] = (
    team_matches["goals_for"] - team_matches["goals_against"]
)

team_matches["days_rest"] = (
    team_matches.groupby("team")["date"]
    .diff()
    .dt.total_seconds()
    .div(86400)
)

print(team_matches[[
    "team",
    "date",
    "competition",
    "result",
    "points",
    "goal_difference",
    "days_rest"
]].head(12))

                team                      date        competition result  \
0   1. FC Heidenheim 2024-08-22 17:00:00+00:00  conference_league      W   
1   1. FC Heidenheim 2024-08-29 18:30:00+00:00  conference_league      W   
2   1. FC Heidenheim 2024-10-03 16:45:00+00:00  conference_league      W   
3   1. FC Heidenheim 2024-10-24 19:00:00+00:00  conference_league      W   
4   1. FC Heidenheim 2024-11-07 20:00:00+00:00  conference_league      W   
5   1. FC Heidenheim 2024-11-28 17:45:00+00:00  conference_league      L   
6   1. FC Heidenheim 2024-12-12 17:45:00+00:00  conference_league      L   
7   1. FC Heidenheim 2024-12-19 20:00:00+00:00  conference_league      D   
8   1. FC Heidenheim 2025-02-13 20:00:00+00:00  conference_league      W   
9   1. FC Heidenheim 2025-02-20 17:45:00+00:00  conference_league      L   
10        1. FC Köln 2022-09-08 16:45:00+00:00  conference_league      D   
11        1. FC Köln 2022-09-15 19:00:00+00:00  conference_league      W   

    points 

In [ ]:
# Create first all-competition rolling features
# Preventing data leakage

rolling_cols = [
    "points",
    "goals_for",
    "goals_against",
    "goal_difference"
]

for col in rolling_cols:
    team_matches[f"{col}_last5_all"] = (
        team_matches.groupby("team")[col]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    )

print(team_matches[[
    "team",
    "date",
    "competition",
    "points",
    "points_last5_all",
    "goal_difference_last5_all"
]].head(10))

               team                      date        competition  points  \
0  1. FC Heidenheim 2024-08-22 17:00:00+00:00  conference_league       3   
1  1. FC Heidenheim 2024-08-29 18:30:00+00:00  conference_league       3   
2  1. FC Heidenheim 2024-10-03 16:45:00+00:00  conference_league       3   
3  1. FC Heidenheim 2024-10-24 19:00:00+00:00  conference_league       3   
4  1. FC Heidenheim 2024-11-07 20:00:00+00:00  conference_league       3   
5  1. FC Heidenheim 2024-11-28 17:45:00+00:00  conference_league       0   
6  1. FC Heidenheim 2024-12-12 17:45:00+00:00  conference_league       0   
7  1. FC Heidenheim 2024-12-19 20:00:00+00:00  conference_league       1   
8  1. FC Heidenheim 2025-02-13 20:00:00+00:00  conference_league       3   
9  1. FC Heidenheim 2025-02-20 17:45:00+00:00  conference_league       0   

   points_last5_all  goal_difference_last5_all  
0               NaN                        NaN  
1               3.0                        1.0  
2               

In [60]:
# Add all competition win rate over previous 5 matches

team_matches["win"] = (
    team_matches["result"] == "W"
).astype(int)

team_matches["win_rate_last5_all"] = (
    team_matches.groupby("team")["win"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

print(team_matches[[
    "team",
    "date",
    "result",
    "win",
    "points_last5_all",
    "win_rate_last5_all"
]].head(10))

               team                      date result  win  points_last5_all  \
0  1. FC Heidenheim 2024-08-22 17:00:00+00:00      W    1               NaN   
1  1. FC Heidenheim 2024-08-29 18:30:00+00:00      W    1               3.0   
2  1. FC Heidenheim 2024-10-03 16:45:00+00:00      W    1               3.0   
3  1. FC Heidenheim 2024-10-24 19:00:00+00:00      W    1               3.0   
4  1. FC Heidenheim 2024-11-07 20:00:00+00:00      W    1               3.0   
5  1. FC Heidenheim 2024-11-28 17:45:00+00:00      L    0               3.0   
6  1. FC Heidenheim 2024-12-12 17:45:00+00:00      L    0               2.4   
7  1. FC Heidenheim 2024-12-19 20:00:00+00:00      D    0               1.8   
8  1. FC Heidenheim 2025-02-13 20:00:00+00:00      W    1               1.4   
9  1. FC Heidenheim 2025-02-20 17:45:00+00:00      L    0               1.4   

   win_rate_last5_all  
0                 NaN  
1                 1.0  
2                 1.0  
3                 1.0  
4         

In [61]:
# Add performance-stat rolling features for previous 5 matches.

performance_cols = [
    "xg_for",
    "xg_against",
    "shots_for",
    "shots_against",
    "shots_on_target_for",
    "shots_on_target_against",
    "possession_for",
    "passes_for",
    "corners_for"
]

for col in performance_cols:
    team_matches[f"{col}_last5_all"] = (
        team_matches.groupby("team")[col]
        .transform(
            lambda x: x.shift(1).rolling(
                5,
                min_periods=1
            ).mean()
        )
    )

print(team_matches[[
    "team",
    "date",
    "xg_for",
    "xg_for_last5_all",
    "shots_for_last5_all",
    "shots_on_target_for_last5_all"
]].head(10))

               team                      date  xg_for  xg_for_last5_all  \
0  1. FC Heidenheim 2024-08-22 17:00:00+00:00     NaN               NaN   
1  1. FC Heidenheim 2024-08-29 18:30:00+00:00     NaN               NaN   
2  1. FC Heidenheim 2024-10-03 16:45:00+00:00    1.89               NaN   
3  1. FC Heidenheim 2024-10-24 19:00:00+00:00    1.12          1.890000   
4  1. FC Heidenheim 2024-11-07 20:00:00+00:00    0.67          1.505000   
5  1. FC Heidenheim 2024-11-28 17:45:00+00:00    1.72          1.226667   
6  1. FC Heidenheim 2024-12-12 17:45:00+00:00    2.01          1.350000   
7  1. FC Heidenheim 2024-12-19 20:00:00+00:00    1.76          1.482000   
8  1. FC Heidenheim 2025-02-13 20:00:00+00:00    1.96          1.456000   
9  1. FC Heidenheim 2025-02-20 17:45:00+00:00    1.85          1.624000   

   shots_for_last5_all  shots_on_target_for_last5_all  
0                  NaN                            NaN  
1             5.000000                       3.000000  
2     

In [62]:
# Create two more rolling features: xG difference and shots-on-target difference

team_matches["xg_difference_last5_all"] = (
    team_matches["xg_for_last5_all"]
    - team_matches["xg_against_last5_all"]
)

team_matches["shots_on_target_difference_last5_all"] = (
    team_matches["shots_on_target_for_last5_all"]
    - team_matches["shots_on_target_against_last5_all"]
)

print(team_matches[[
    "team",
    "date",
    "xg_for_last5_all",
    "xg_against_last5_all",
    "xg_difference_last5_all",
    "shots_on_target_difference_last5_all"
]].head(10))

               team                      date  xg_for_last5_all  \
0  1. FC Heidenheim 2024-08-22 17:00:00+00:00               NaN   
1  1. FC Heidenheim 2024-08-29 18:30:00+00:00               NaN   
2  1. FC Heidenheim 2024-10-03 16:45:00+00:00               NaN   
3  1. FC Heidenheim 2024-10-24 19:00:00+00:00          1.890000   
4  1. FC Heidenheim 2024-11-07 20:00:00+00:00          1.505000   
5  1. FC Heidenheim 2024-11-28 17:45:00+00:00          1.226667   
6  1. FC Heidenheim 2024-12-12 17:45:00+00:00          1.350000   
7  1. FC Heidenheim 2024-12-19 20:00:00+00:00          1.482000   
8  1. FC Heidenheim 2025-02-13 20:00:00+00:00          1.456000   
9  1. FC Heidenheim 2025-02-20 17:45:00+00:00          1.624000   

   xg_against_last5_all  xg_difference_last5_all  \
0                   NaN                      NaN   
1                   NaN                      NaN   
2                   NaN                      NaN   
3              1.440000                    0.450   
4 

In [63]:
# Separate Premier League-only form an all-competition form

pl_team_matches = team_matches[
    team_matches["competition"] == "premier_league"
].copy()

pl_team_matches = pl_team_matches.sort_values(
    ["team", "date", "match_id"]
).reset_index(drop=True)

print("PL team rows:", pl_team_matches.shape)

print(pl_team_matches[[
    "team",
    "date",
    "season",
    "competition",
    "points"
]].head(10))

PL team rows: (4560, 43)
      team                      date   season     competition  points
0  Arsenal 2020-09-12 11:30:00+00:00  2020_21  premier_league       3
1  Arsenal 2020-09-19 19:00:00+00:00  2020_21  premier_league       3
2  Arsenal 2020-09-28 19:00:00+00:00  2020_21  premier_league       0
3  Arsenal 2020-10-04 13:00:00+00:00  2020_21  premier_league       3
4  Arsenal 2020-10-17 16:30:00+00:00  2020_21  premier_league       0
5  Arsenal 2020-10-25 19:15:00+00:00  2020_21  premier_league       0
6  Arsenal 2020-11-01 16:30:00+00:00  2020_21  premier_league       3
7  Arsenal 2020-11-08 19:15:00+00:00  2020_21  premier_league       0
8  Arsenal 2020-11-22 16:30:00+00:00  2020_21  premier_league       1
9  Arsenal 2020-11-29 19:15:00+00:00  2020_21  premier_league       0


In [64]:
pl_rolling_cols = [
    "points",
    "goals_for",
    "goals_against",
    "goal_difference"
]

for col in pl_rolling_cols:
    pl_team_matches[f"{col}_last5_pl"] = (
        pl_team_matches.groupby("team")[col]
        .transform(
            lambda x: x.shift(1).rolling(
                5,
                min_periods=1
            ).mean()
        )
    )

print(pl_team_matches[[
    "team",
    "date",
    "points",
    "points_last5_pl",
    "goal_difference_last5_pl"
]].head(10))

      team                      date  points  points_last5_pl  \
0  Arsenal 2020-09-12 11:30:00+00:00       3              NaN   
1  Arsenal 2020-09-19 19:00:00+00:00       3             3.00   
2  Arsenal 2020-09-28 19:00:00+00:00       0             3.00   
3  Arsenal 2020-10-04 13:00:00+00:00       3             2.00   
4  Arsenal 2020-10-17 16:30:00+00:00       0             2.25   
5  Arsenal 2020-10-25 19:15:00+00:00       0             1.80   
6  Arsenal 2020-11-01 16:30:00+00:00       3             1.20   
7  Arsenal 2020-11-08 19:15:00+00:00       0             1.20   
8  Arsenal 2020-11-22 16:30:00+00:00       1             1.20   
9  Arsenal 2020-11-29 19:15:00+00:00       0             0.80   

   goal_difference_last5_pl  
0                       NaN  
1                  3.000000  
2                  2.000000  
3                  0.666667  
4                  0.750000  
5                  0.400000  
6                 -0.400000  
7                 -0.400000  
8             

In [65]:
# Add PL-only performance stats

pl_performance_cols = [
    "xg_for",
    "xg_against",
    "shots_for",
    "shots_against",
    "shots_on_target_for",
    "shots_on_target_against",
    "possession_for",
    "passes_for",
    "corners_for"
]

for col in pl_performance_cols:
    pl_team_matches[f"{col}_last5_pl"] = (
        pl_team_matches.groupby("team")[col]
        .transform(
            lambda x: x.shift(1).rolling(
                5,
                min_periods=1
            ).mean()
        )
    )

print(pl_team_matches[[
    "team",
    "date",
    "xg_for",
    "xg_for_last5_pl",
    "shots_for_last5_pl",
    "shots_on_target_for_last5_pl"
]].head(10))

      team                      date  xg_for  xg_for_last5_pl  \
0  Arsenal 2020-09-12 11:30:00+00:00     NaN              NaN   
1  Arsenal 2020-09-19 19:00:00+00:00     NaN              NaN   
2  Arsenal 2020-09-28 19:00:00+00:00     NaN              NaN   
3  Arsenal 2020-10-04 13:00:00+00:00     NaN              NaN   
4  Arsenal 2020-10-17 16:30:00+00:00     NaN              NaN   
5  Arsenal 2020-10-25 19:15:00+00:00     NaN              NaN   
6  Arsenal 2020-11-01 16:30:00+00:00     NaN              NaN   
7  Arsenal 2020-11-08 19:15:00+00:00     NaN              NaN   
8  Arsenal 2020-11-22 16:30:00+00:00     NaN              NaN   
9  Arsenal 2020-11-29 19:15:00+00:00     NaN              NaN   

   shots_for_last5_pl  shots_on_target_for_last5_pl  
0                 NaN                           NaN  
1                13.0                          6.00  
2                10.0                          4.50  
3                 8.0                          4.00  
4             

In [66]:
# Add PL-only win rate and difference features

pl_team_matches["win_rate_last5_pl"] = (
    pl_team_matches.groupby("team")["win"]
    .transform(
        lambda x: x.shift(1).rolling(
            5,
            min_periods=1
        ).mean()
    )
)

pl_team_matches["xg_difference_last5_pl"] = (
    pl_team_matches["xg_for_last5_pl"]
    - pl_team_matches["xg_against_last5_pl"]
)

pl_team_matches["shots_on_target_difference_last5_pl"] = (
    pl_team_matches["shots_on_target_for_last5_pl"]
    - pl_team_matches["shots_on_target_against_last5_pl"]
)

print(pl_team_matches[[
    "team",
    "date",
    "points_last5_pl",
    "win_rate_last5_pl",
    "xg_difference_last5_pl",
    "shots_on_target_difference_last5_pl"
]].head(10))

      team                      date  points_last5_pl  win_rate_last5_pl  \
0  Arsenal 2020-09-12 11:30:00+00:00              NaN                NaN   
1  Arsenal 2020-09-19 19:00:00+00:00             3.00           1.000000   
2  Arsenal 2020-09-28 19:00:00+00:00             3.00           1.000000   
3  Arsenal 2020-10-04 13:00:00+00:00             2.00           0.666667   
4  Arsenal 2020-10-17 16:30:00+00:00             2.25           0.750000   
5  Arsenal 2020-10-25 19:15:00+00:00             1.80           0.600000   
6  Arsenal 2020-11-01 16:30:00+00:00             1.20           0.400000   
7  Arsenal 2020-11-08 19:15:00+00:00             1.20           0.400000   
8  Arsenal 2020-11-22 16:30:00+00:00             1.20           0.400000   
9  Arsenal 2020-11-29 19:15:00+00:00             0.80           0.200000   

   xg_difference_last5_pl  shots_on_target_difference_last5_pl  
0                     NaN                                  NaN  
1                     NaN        

In [67]:
check_cols = [
    "team",
    "date",
    "competition",
    "points_last5_all",
    "points_last5_pl",
    "win_rate_last5_all",
    "win_rate_last5_pl",
    "goal_difference_last5_all",
    "goal_difference_last5_pl"
]

print(pl_team_matches[check_cols].head(10))

      team                      date     competition  points_last5_all  \
0  Arsenal 2020-09-12 11:30:00+00:00  premier_league               NaN   
1  Arsenal 2020-09-19 19:00:00+00:00  premier_league               3.0   
2  Arsenal 2020-09-28 19:00:00+00:00  premier_league               3.0   
3  Arsenal 2020-10-04 13:00:00+00:00  premier_league               2.0   
4  Arsenal 2020-10-17 16:30:00+00:00  premier_league               2.0   
5  Arsenal 2020-10-25 19:15:00+00:00  premier_league               1.4   
6  Arsenal 2020-11-01 16:30:00+00:00  premier_league               1.8   
7  Arsenal 2020-11-08 19:15:00+00:00  premier_league               2.4   
8  Arsenal 2020-11-22 16:30:00+00:00  premier_league               1.8   
9  Arsenal 2020-11-29 19:15:00+00:00  premier_league               2.0   

   points_last5_pl  win_rate_last5_all  win_rate_last5_pl  \
0              NaN                 NaN                NaN   
1             3.00                 1.0           1.000000   
2 

In [68]:
# Add fixture congesion features from all-competition schedule

def count_recent_matches(group, days):
    dates = group["date"]

    return dates.apply(
        lambda current_date: (
            (dates < current_date)
            & (dates >= current_date - pd.Timedelta(days=days))
        ).sum()
    )

team_matches["matches_last_7_days"] = (
    team_matches.groupby("team", group_keys=False)
    .apply(lambda group: count_recent_matches(group, 7))
)

team_matches["matches_last_14_days"] = (
    team_matches.groupby("team", group_keys=False)
    .apply(lambda group: count_recent_matches(group, 14))
)

print(team_matches[[
    "team",
    "date",
    "competition",
    "days_rest",
    "matches_last_7_days",
    "matches_last_14_days"
]].head(12))

                team                      date        competition  days_rest  \
0   1. FC Heidenheim 2024-08-22 17:00:00+00:00  conference_league        NaN   
1   1. FC Heidenheim 2024-08-29 18:30:00+00:00  conference_league   7.062500   
2   1. FC Heidenheim 2024-10-03 16:45:00+00:00  conference_league  34.927083   
3   1. FC Heidenheim 2024-10-24 19:00:00+00:00  conference_league  21.093750   
4   1. FC Heidenheim 2024-11-07 20:00:00+00:00  conference_league  14.041667   
5   1. FC Heidenheim 2024-11-28 17:45:00+00:00  conference_league  20.906250   
6   1. FC Heidenheim 2024-12-12 17:45:00+00:00  conference_league  14.000000   
7   1. FC Heidenheim 2024-12-19 20:00:00+00:00  conference_league   7.093750   
8   1. FC Heidenheim 2025-02-13 20:00:00+00:00  conference_league  56.000000   
9   1. FC Heidenheim 2025-02-20 17:45:00+00:00  conference_league   6.906250   
10        1. FC Köln 2022-09-08 16:45:00+00:00  conference_league        NaN   
11        1. FC Köln 2022-09-15 19:00:00

In [69]:
# Add congestion features on the PL rows

congestion_features = team_matches[[
    "match_id",
    "team",
    "matches_last_7_days",
    "matches_last_14_days"
]].copy()

pl_team_matches = pl_team_matches.merge(
    congestion_features,
    on=["match_id", "team"],
    how="left"
)

print(pl_team_matches[[
    "team",
    "date",
    "competition",
    "days_rest",
    "matches_last_7_days",
    "matches_last_14_days"
]].head(10))

      team                      date     competition  days_rest  \
0  Arsenal 2020-09-12 11:30:00+00:00  premier_league        NaN   
1  Arsenal 2020-09-19 19:00:00+00:00  premier_league   7.312500   
2  Arsenal 2020-09-28 19:00:00+00:00  premier_league   5.010417   
3  Arsenal 2020-10-04 13:00:00+00:00  premier_league   2.760417   
4  Arsenal 2020-10-17 16:30:00+00:00  premier_league  13.145833   
5  Arsenal 2020-10-25 19:15:00+00:00  premier_league   3.097222   
6  Arsenal 2020-11-01 16:30:00+00:00  premier_league   2.854167   
7  Arsenal 2020-11-08 19:15:00+00:00  premier_league   2.968750   
8  Arsenal 2020-11-22 16:30:00+00:00  premier_league  13.885417   
9  Arsenal 2020-11-29 19:15:00+00:00  premier_league   3.055556   

   matches_last_7_days  matches_last_14_days  
0                    0                     0  
1                    0                     1  
2                    1                     2  
3                    2                     3  
4                    0     

In [70]:
# Check Arsenal's schedule

arsenal_schedule = team_matches[
    (team_matches["team"] == "Arsenal") &
    (team_matches["date"] >= "2020-09-20") &
    (team_matches["date"] <= "2020-10-05")
][[
    "date",
    "competition",
    "opponent",
    "venue",
    "result"
]]

print(arsenal_schedule)

                         date     competition          opponent venue result
500 2020-09-23 18:45:00+00:00         efl_cup    Leicester City  away      W
501 2020-09-28 19:00:00+00:00  premier_league         Liverpool  away      L
502 2020-10-01 18:45:00+00:00         efl_cup         Liverpool  away      D
503 2020-10-04 13:00:00+00:00  premier_league  Sheffield United  home      W


In [71]:
# Add a 10-match PL form window

pl_last10_cols = [
    "points",
    "goal_difference"
]

for col in pl_last10_cols:
    pl_team_matches[f"{col}_last10_pl"] = (
        pl_team_matches.groupby("team")[col]
        .transform(
            lambda x: x.shift(1).rolling(
                10,
                min_periods=1
            ).mean()
        )
    )

print(pl_team_matches[[
    "team",
    "date",
    "points_last5_pl",
    "points_last10_pl",
    "goal_difference_last5_pl",
    "goal_difference_last10_pl"
]].head(12))

       team                      date  points_last5_pl  points_last10_pl  \
0   Arsenal 2020-09-12 11:30:00+00:00              NaN               NaN   
1   Arsenal 2020-09-19 19:00:00+00:00             3.00          3.000000   
2   Arsenal 2020-09-28 19:00:00+00:00             3.00          3.000000   
3   Arsenal 2020-10-04 13:00:00+00:00             2.00          2.000000   
4   Arsenal 2020-10-17 16:30:00+00:00             2.25          2.250000   
5   Arsenal 2020-10-25 19:15:00+00:00             1.80          1.800000   
6   Arsenal 2020-11-01 16:30:00+00:00             1.20          1.500000   
7   Arsenal 2020-11-08 19:15:00+00:00             1.20          1.714286   
8   Arsenal 2020-11-22 16:30:00+00:00             1.20          1.500000   
9   Arsenal 2020-11-29 19:15:00+00:00             0.80          1.444444   
10  Arsenal 2020-12-06 16:30:00+00:00             0.80          1.300000   
11  Arsenal 2020-12-13 19:15:00+00:00             0.80          1.000000   

    goal_di

In [72]:
# Add 10 match window for win rate

pl_team_matches["win_rate_last10_pl"] = (
    pl_team_matches.groupby("team")["win"]
    .transform(
        lambda x: x.shift(1).rolling(
            10,
            min_periods=1
        ).mean()
    )
)

print(pl_team_matches[[
    "team",
    "date",
    "win_rate_last5_pl",
    "win_rate_last10_pl"
]].head(12))

       team                      date  win_rate_last5_pl  win_rate_last10_pl
0   Arsenal 2020-09-12 11:30:00+00:00                NaN                 NaN
1   Arsenal 2020-09-19 19:00:00+00:00           1.000000            1.000000
2   Arsenal 2020-09-28 19:00:00+00:00           1.000000            1.000000
3   Arsenal 2020-10-04 13:00:00+00:00           0.666667            0.666667
4   Arsenal 2020-10-17 16:30:00+00:00           0.750000            0.750000
5   Arsenal 2020-10-25 19:15:00+00:00           0.600000            0.600000
6   Arsenal 2020-11-01 16:30:00+00:00           0.400000            0.500000
7   Arsenal 2020-11-08 19:15:00+00:00           0.400000            0.571429
8   Arsenal 2020-11-22 16:30:00+00:00           0.400000            0.500000
9   Arsenal 2020-11-29 19:15:00+00:00           0.200000            0.444444
10  Arsenal 2020-12-06 16:30:00+00:00           0.200000            0.400000
11  Arsenal 2020-12-13 19:15:00+00:00           0.200000            0.300000

In [73]:
# Work towards ELO/opponent-strength side

elo_matches = historical_df[[
    "match_id",
    "date",
    "season",
    "competition",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals"
]].copy()

elo_matches = elo_matches.sort_values(
    ["date", "match_id"]
).reset_index(drop=True)

print("Elo matches:", elo_matches.shape)
print()
print(elo_matches.head(10))

Elo matches: (10653, 8)

       match_id                      date   season       competition  \
0  mt_304172028 2020-08-08 13:00:00+00:00  2020_21  champions_league   
1  mt_847096460 2020-08-08 17:30:00+00:00  2020_21  champions_league   
2  mt_138668638 2020-08-11 16:00:00+00:00  2020_21  champions_league   
3  mt_304127364 2020-08-18 16:00:00+00:00  2020_21  champions_league   
4  mt_579499767 2020-08-18 16:00:00+00:00  2020_21     europa_league   
5  mt_304142684 2020-08-18 17:00:00+00:00  2020_21  champions_league   
6  mt_452822596 2020-08-18 18:00:00+00:00  2020_21     europa_league   
7  mt_847069838 2020-08-18 18:00:00+00:00  2020_21  champions_league   
8  mt_963248296 2020-08-18 18:45:00+00:00  2020_21  champions_league   
9  mt_304127132 2020-08-18 19:00:00+00:00  2020_21  champions_league   

          home_team              away_team  home_goals  away_goals  
0      SP Tre Fiori            Linfield FC           0           2  
1          FC Drita  Inter Club d'Escaldes  

In [74]:
# Initialize ELO system
# Start every team at 1500 rating 

INITIAL_ELO = 1500
K_FACTOR = 20

elo_ratings = {}

all_teams = pd.concat([
    elo_matches["home_team"],
    elo_matches["away_team"]
]).unique()

for team in all_teams:
    elo_ratings[team] = INITIAL_ELO

print("Number of teams:", len(elo_ratings))
print()

for team in list(elo_ratings)[:10]:
    print(team, elo_ratings[team])

Number of teams: 522

SP Tre Fiori 1500
FC Drita 1500
Qarabağ 1500
Lincoln Red Imps 1500
Legia Warszawa 1500
UE Engordany 1500
Dynamo Brest 1500
Celtic 1500
FK Crvena zvezda 1500
FC Ararat-Armenia 1500


In [75]:
# Create expected-result function, accounts for opponent strength

def expected_score(team_elo, opponent_elo):
    return 1 / (
        1 + 10 ** ((opponent_elo - team_elo) / 400)
    )

print("Equal teams:", expected_score(1500, 1500))
print("Stronger team:", expected_score(1700, 1500))
print("Weaker team:", expected_score(1500, 1700))

Equal teams: 0.5
Stronger team: 0.7597469266479578
Weaker team: 0.2402530733520421


In [76]:
# Add actual result score for ELO updates

def actual_score(home_goals, away_goals):
    if home_goals > away_goals:
        return 1.0, 0.0
    elif home_goals < away_goals:
        return 0.0, 1.0
    else:
        return 0.5, 0.5

print("Home win:", actual_score(2, 1))
print("Draw:", actual_score(1, 1))
print("Away win:", actual_score(0, 2))

Home win: (1.0, 0.0)
Draw: (0.5, 0.5)
Away win: (0.0, 1.0)


In [77]:
# Test one actual ELO update

test_match = elo_matches.iloc[0]

home_team = test_match["home_team"]
away_team = test_match["away_team"]

home_elo = elo_ratings[home_team]
away_elo = elo_ratings[away_team]

home_expected = expected_score(home_elo, away_elo)
away_expected = expected_score(away_elo, home_elo)

home_actual, away_actual = actual_score(
    test_match["home_goals"],
    test_match["away_goals"]
)

new_home_elo = home_elo + K_FACTOR * (
    home_actual - home_expected
)

new_away_elo = away_elo + K_FACTOR * (
    away_actual - away_expected
)

print(home_team, home_elo, "->", new_home_elo)
print(away_team, away_elo, "->", new_away_elo)

SP Tre Fiori 1500 -> 1490.0
Linfield FC 1500 -> 1510.0


In [78]:
# Run ELO chronologically across all matches
# Need to save each team's ELO before the match happens (preventing leakage)

elo_ratings = {}

elo_history = []

for _, match in elo_matches.iterrows():
    home_team = match["home_team"]
    away_team = match["away_team"]

    home_elo = elo_ratings.get(home_team, INITIAL_ELO)
    away_elo = elo_ratings.get(away_team, INITIAL_ELO)

    home_expected = expected_score(home_elo, away_elo)
    away_expected = expected_score(away_elo, home_elo)

    home_actual, away_actual = actual_score(
        match["home_goals"],
        match["away_goals"]
    )

    # Save PRE-MATCH Elo ratings
    elo_history.append({
        "match_id": match["match_id"],
        "home_elo": home_elo,
        "away_elo": away_elo,
        "elo_difference": home_elo - away_elo
    })

    # Update ratings AFTER the match
    elo_ratings[home_team] = (
        home_elo
        + K_FACTOR * (home_actual - home_expected)
    )

    elo_ratings[away_team] = (
        away_elo
        + K_FACTOR * (away_actual - away_expected)
    )

elo_history_df = pd.DataFrame(elo_history)

print("Elo history shape:", elo_history_df.shape)
print()
print(elo_history_df.head(10))

Elo history shape: (10653, 4)

       match_id  home_elo  away_elo  elo_difference
0  mt_304172028    1500.0    1500.0             0.0
1  mt_847096460    1500.0    1500.0             0.0
2  mt_138668638    1510.0    1510.0             0.0
3  mt_304127364    1500.0    1500.0             0.0
4  mt_579499767    1500.0    1500.0             0.0
5  mt_304142684    1500.0    1520.0           -20.0
6  mt_452822596    1500.0    1500.0             0.0
7  mt_847069838    1500.0    1500.0             0.0
8  mt_963248296    1500.0    1500.0             0.0
9  mt_304127132    1500.0    1500.0             0.0


In [79]:
# Attach the pre-match ELO ratings to the Premier League dataset

pl_team_matches = pl_team_matches.merge(
    elo_history_df,
    on = "match_id",
    how = "left"
)

print(pl_team_matches[[
    "team",
    "date",
    "venue",
    "home_elo",
    "away_elo",
    "elo_difference"
]].head(10))

      team                      date venue     home_elo     away_elo  \
0  Arsenal 2020-09-12 11:30:00+00:00  away  1500.000000  1500.000000   
1  Arsenal 2020-09-19 19:00:00+00:00  home  1510.000000  1500.575011   
2  Arsenal 2020-09-28 19:00:00+00:00  away  1530.000000  1529.728079   
3  Arsenal 2020-10-04 13:00:00+00:00  home  1520.318268  1471.421744   
4  Arsenal 2020-10-17 16:30:00+00:00  away  1518.866699  1528.920132   
5  Arsenal 2020-10-25 19:15:00+00:00  home  1528.086868  1510.315657   
6  Arsenal 2020-11-01 16:30:00+00:00  away  1541.174416  1527.079303   
7  Arsenal 2020-11-08 19:15:00+00:00  home  1547.530664  1525.805622   
8  Arsenal 2020-11-22 16:30:00+00:00  away  1491.565720  1536.906180   
9  Arsenal 2020-11-29 19:15:00+00:00  home  1545.419309  1503.190005   

   elo_difference  
0        0.000000  
1        9.424989  
2        0.271921  
3       48.896525  
4      -10.053433  
5       17.771211  
6       14.095113  
7       21.725041  
8      -45.340461  
9      

In [80]:
# Convert ELO into team's perspective

pl_team_matches["team_elo"] = np.where(
    pl_team_matches["venue"] == "home",
    pl_team_matches["home_elo"],
    pl_team_matches["away_elo"]
)

pl_team_matches["opponent_elo"] = np.where(
    pl_team_matches["venue"] == "home",
    pl_team_matches["away_elo"],
    pl_team_matches["home_elo"]
)

pl_team_matches["team_elo_difference"] = (
    pl_team_matches["team_elo"]
    - pl_team_matches["opponent_elo"]
)

print(pl_team_matches[[
    "team",
    "date",
    "venue",
    "team_elo",
    "opponent_elo",
    "team_elo_difference"
]].head(10))

      team                      date venue     team_elo  opponent_elo  \
0  Arsenal 2020-09-12 11:30:00+00:00  away  1500.000000   1500.000000   
1  Arsenal 2020-09-19 19:00:00+00:00  home  1510.000000   1500.575011   
2  Arsenal 2020-09-28 19:00:00+00:00  away  1529.728079   1530.000000   
3  Arsenal 2020-10-04 13:00:00+00:00  home  1520.318268   1471.421744   
4  Arsenal 2020-10-17 16:30:00+00:00  away  1528.920132   1518.866699   
5  Arsenal 2020-10-25 19:15:00+00:00  home  1528.086868   1510.315657   
6  Arsenal 2020-11-01 16:30:00+00:00  away  1527.079303   1541.174416   
7  Arsenal 2020-11-08 19:15:00+00:00  home  1547.530664   1525.805622   
8  Arsenal 2020-11-22 16:30:00+00:00  away  1536.906180   1491.565720   
9  Arsenal 2020-11-29 19:15:00+00:00  home  1545.419309   1503.190005   

   team_elo_difference  
0             0.000000  
1             9.424989  
2            -0.271921  
3            48.896525  
4            10.053433  
5            17.771211  
6           -14.09511

In [81]:
# Inspect the highest-rated teams at the end of our historical period
# Helps make sure the ELO system is producing correct rankings

final_elo_df = pd.DataFrame(
    elo_ratings.items(),
    columns=["team", "elo"]
).sort_values(
    "elo",
    ascending=False
).reset_index(drop=True)

print(final_elo_df.head(20))

                      team          elo
0                  Arsenal  1832.254505
1          Manchester City  1830.464186
2        FC Bayern München  1752.968257
3              Aston Villa  1743.044129
4        Manchester United  1737.184241
5      Paris Saint-Germain  1730.687020
6              Real Madrid  1727.229877
7              Bournemouth  1703.200098
8                Liverpool  1702.213704
9   Brighton & Hove Albion  1674.408587
10               Brentford  1665.179112
11                 Chelsea  1662.082161
12               Barcelona  1661.434661
13        Newcastle United  1660.598862
14            Leeds United  1649.437661
15       Nottingham Forest  1641.998237
16          Crystal Palace  1640.548510
17                  Fulham  1638.815184
18                   Inter  1637.736670
19     Bayer 04 Leverkusen  1628.421828


In [82]:
# Check coverage, some PL clubs are above clubs like Madrid or Barcelona

team_match_counts = (
    team_matches.groupby("team")
    .size()
    .reset_index(name="matches_in_dataset")
)

final_elo_check = final_elo_df.merge(
    team_match_counts,
    on="team",
    how="left"
)

print(final_elo_check.head(20))

                      team          elo  matches_in_dataset
0                  Arsenal  1832.254505                 323
1          Manchester City  1830.464186                 349
2        FC Bayern München  1752.968257                  70
3              Aston Villa  1743.044129                 293
4        Manchester United  1737.184241                 323
5      Paris Saint-Germain  1730.687020                  74
6              Real Madrid  1727.229877                  78
7              Bournemouth  1703.200098                 273
8                Liverpool  1702.213704                 336
9   Brighton & Hove Albion  1674.408587                 271
10               Brentford  1665.179112                 272
11                 Chelsea  1662.082161                 336
12               Barcelona  1661.434661                  64
13        Newcastle United  1660.598862                 286
14            Leeds United  1649.437661                 274
15       Nottingham Forest  1641.998237 

In [83]:
# Check how much coverage the PL clubs have

pl_teams = pd.unique(
    pd.concat([
        historical_df.loc[
            historical_df["competition"] == "premier_league",
            "home_team"
        ],
        historical_df.loc[
            historical_df["competition"] == "premier_league",
            "away_team"
        ]
    ])
)

pl_elo_check = final_elo_check[
    final_elo_check["team"].isin(pl_teams)
].sort_values(
    "matches_in_dataset",
    ascending=False
)

print("Unique PL teams:", len(pl_elo_check))
print()
print(pl_elo_check.to_string(index=False))

Unique PL teams: 28

                  team         elo  matches_in_dataset
       Manchester City 1830.464186                 349
             Liverpool 1702.213704                 336
               Chelsea 1662.082161                 336
               Arsenal 1832.254505                 323
     Manchester United 1737.184241                 323
     Tottenham Hotspur 1605.237287                 312
        Leicester City 1437.515534                 300
          Norwich City 1505.742897                 295
     Nottingham Forest 1641.998237                 293
           Aston Villa 1743.044129                 293
       West Ham United 1590.641214                 292
           Southampton 1588.217095                 290
      Sheffield United 1471.097353                 289
  West Bromwich Albion 1439.813279                 288
      Newcastle United 1660.598862                 286
               Watford 1408.933556                 286
        Crystal Palace 1640.548510          

In [84]:
# Check if a promoted team's ELO rating carry from the Championship into the first PL match

brentford_transition = team_matches[
    (team_matches["team"] == "Brentford") &
    (team_matches["date"] >= "2021-05-01") &
    (team_matches["date"] <= "2021-08-20")
][[
    "date",
    "season",
    "competition",
    "opponent",
    "result"
]]

print(brentford_transition.to_string(index=False))

                     date  season    competition     opponent result
2021-05-01 14:00:00+00:00 2020_21   championship      Watford      W
2021-05-08 11:30:00+00:00 2020_21   championship Bristol City      W
2021-05-17 17:00:00+00:00 2020_21   championship  Bournemouth      L
2021-05-22 11:30:00+00:00 2020_21   championship  Bournemouth      W
2021-05-29 14:00:00+00:00 2020_21   championship Swansea City      W
2021-08-13 19:00:00+00:00 2021_22 premier_league      Arsenal      W


In [85]:
brentford_first_pl = pl_team_matches[
    (pl_team_matches["team"] == "Brentford") &
    (pl_team_matches["season"] == "2021_22")
][[
    "date",
    "opponent",
    "venue",
    "team_elo",
    "opponent_elo",
    "team_elo_difference"
]].head(3)

print(brentford_first_pl)

                         date        opponent venue     team_elo  \
608 2021-08-13 19:00:00+00:00         Arsenal  home  1612.555597   
609 2021-08-21 14:00:00+00:00  Crystal Palace  away  1621.723353   
610 2021-08-28 14:00:00+00:00     Aston Villa  away  1623.614537   

     opponent_elo  team_elo_difference  
608   1583.573431            28.982167  
609   1461.232900           160.490453  
610   1531.014098            92.600439  


In [86]:
# Check another promoted team

forest_first_pl = pl_team_matches[
    (pl_team_matches["team"] == "Nottingham Forest") &
    (pl_team_matches["season"] == "2022_23")
][[
    "date",
    "opponent",
    "venue",
    "team_elo",
    "opponent_elo",
    "team_elo_difference"
]].head(3)

print(forest_first_pl)

                          date          opponent venue     team_elo  \
3382 2022-08-06 14:00:00+00:00  Newcastle United  away  1592.449338   
3383 2022-08-14 13:00:00+00:00   West Ham United  home  1581.441789   
3384 2022-08-20 14:00:00+00:00           Everton  away  1590.482654   

      opponent_elo  team_elo_difference  
3382   1557.324327            35.125011  
3383   1548.015277            33.426512  
3384   1491.658858            98.823796  


In [87]:
# Build the match-level modeling dataset
# Model needs one row per match, with the home team's pre-match features and away team's pre match features side by side


# Separate those rows
pl_home_features = pl_team_matches[
    pl_team_matches["venue"] == "home"
].copy()

pl_away_features = pl_team_matches[
    pl_team_matches["venue"] == "away"
].copy()

print("Home rows:", pl_home_features.shape)
print("Away rows:", pl_away_features.shape)

print()
print(pl_home_features[[
    "match_id",
    "date",
    "team",
    "opponent"
]].head())

print()
print(pl_away_features[[
    "match_id",
    "date",
    "team",
    "opponent"
]].head())

Home rows: (2280, 70)
Away rows: (2280, 70)

       match_id                      date     team          opponent
1  mt_304225642 2020-09-19 19:00:00+00:00  Arsenal   West Ham United
3  mt_291883557 2020-10-04 13:00:00+00:00  Arsenal  Sheffield United
5  mt_452119541 2020-10-25 19:15:00+00:00  Arsenal    Leicester City
7  mt_579336752 2020-11-08 19:15:00+00:00  Arsenal       Aston Villa
9  mt_304225022 2020-11-29 19:15:00+00:00  Arsenal     Wolverhampton

       match_id                      date     team           opponent
0  mt_304225753 2020-09-12 11:30:00+00:00  Arsenal             Fulham
2  mt_138774968 2020-09-28 19:00:00+00:00  Arsenal          Liverpool
4  mt_610558232 2020-10-17 16:30:00+00:00  Arsenal    Manchester City
6  mt_726001207 2020-11-01 16:30:00+00:00  Arsenal  Manchester United
8  mt_452119517 2020-11-22 16:30:00+00:00  Arsenal       Leeds United


In [89]:
# Verify that each match_id occurs only once in each DataFrame before merging

home_match_ids = set(pl_home_features["match_id"])
away_match_ids = set(pl_away_features["match_id"])

print("Unique home match IDs:", len(home_match_ids))
print("Unique away match IDs:", len(away_match_ids))
print("Match IDs in both:", len(home_match_ids & away_match_ids))

print("Only in home:", len(home_match_ids - away_match_ids))
print("Only in away:", len(away_match_ids - home_match_ids))

Unique home match IDs: 2280
Unique away match IDs: 2280
Match IDs in both: 2280
Only in home: 0
Only in away: 0


In [90]:
# Build the one row per match dataset
# Merge two sides by match_id

pl_match_features = pl_home_features.merge(
    pl_away_features,
    on = "match_id",
    how = "inner",
    suffixes = ("_home", "_away")
)

print("Match-level shape:", pl_match_features.shape)

print(
    pl_match_features[[
        "match_id",
        "date_home",
        "team_home",
        "team_away"
    ]].head()
)

Match-level shape: (2280, 139)
       match_id                 date_home team_home         team_away
0  mt_304225642 2020-09-19 19:00:00+00:00   Arsenal   West Ham United
1  mt_291883557 2020-10-04 13:00:00+00:00   Arsenal  Sheffield United
2  mt_452119541 2020-10-25 19:15:00+00:00   Arsenal    Leicester City
3  mt_579336752 2020-11-08 19:15:00+00:00   Arsenal       Aston Villa
4  mt_304225022 2020-11-29 19:15:00+00:00   Arsenal     Wolverhampton


In [91]:
# Before selecting model features, create the target variable: what happened in each match
# Each row has both sides' current-match info, so use the home-side goals only to create the target and not as an input feature

pl_match_features["target"] = np.select(
    [
        pl_match_features["goals_for_home"] > pl_match_features["goals_against_home"],
        pl_match_features["goals_for_home"] == pl_match_features["goals_against_home"]
    ],
    ["H", "D"],
    default="A"
)

print(
    pl_match_features[[
        "date_home",
        "team_home",
        "team_away",
        "goals_for_home",
        "goals_against_home",
        "target"
    ]].head(10)
)

print()
print(pl_match_features["target"].value_counts())
print()
print(pl_match_features["target"].value_counts(normalize=True))

                  date_home team_home         team_away  goals_for_home  \
0 2020-09-19 19:00:00+00:00   Arsenal   West Ham United               2   
1 2020-10-04 13:00:00+00:00   Arsenal  Sheffield United               2   
2 2020-10-25 19:15:00+00:00   Arsenal    Leicester City               0   
3 2020-11-08 19:15:00+00:00   Arsenal       Aston Villa               0   
4 2020-11-29 19:15:00+00:00   Arsenal     Wolverhampton               1   
5 2020-12-13 19:15:00+00:00   Arsenal           Burnley               0   
6 2020-12-16 18:00:00+00:00   Arsenal       Southampton               1   
7 2020-12-26 17:30:00+00:00   Arsenal           Chelsea               3   
8 2021-01-14 20:00:00+00:00   Arsenal    Crystal Palace               0   
9 2021-01-18 20:00:00+00:00   Arsenal  Newcastle United               3   

   goals_against_home target  
0                   1      H  
1                   1      H  
2                   1      A  
3                   3      A  
4                  

In [92]:
# Verify seasons before feature selection
# Make sure all seasons have 380 matches

season_counts = (
    pl_match_features["season_home"]
    .value_counts()
    .sort_index()
)

print(season_counts)
print()
print("Total:", season_counts.sum())

season_home
2020_21    380
2021_22    380
2022_23    380
2023_24    380
2024_25    380
2025_26    380
Name: count, dtype: int64

Total: 2280


In [93]:
# Decide which of the 139 columns are reliable pre-match predictors

for i, col in enumerate(pl_match_features.columns):
    print(i, col)

0 match_id
1 date_home
2 season_home
3 competition_home
4 matchday_home
5 team_home
6 opponent_home
7 goals_for_home
8 goals_against_home
9 xg_for_home
10 xg_against_home
11 shots_for_home
12 shots_against_home
13 shots_on_target_for_home
14 shots_on_target_against_home
15 possession_for_home
16 possession_against_home
17 passes_for_home
18 passes_against_home
19 corners_for_home
20 corners_against_home
21 venue_home
22 result_home
23 points_home
24 goal_difference_home
25 days_rest_home
26 points_last5_all_home
27 goals_for_last5_all_home
28 goals_against_last5_all_home
29 goal_difference_last5_all_home
30 win_home
31 win_rate_last5_all_home
32 xg_for_last5_all_home
33 xg_against_last5_all_home
34 shots_for_last5_all_home
35 shots_against_last5_all_home
36 shots_on_target_for_last5_all_home
37 shots_on_target_against_last5_all_home
38 possession_for_last5_all_home
39 passes_for_last5_all_home
40 corners_for_last5_all_home
41 xg_difference_last5_all_home
42 shots_on_target_difference_l

In [94]:
# Avoid current match stats such as result_home, goals_for_home in order to prevent data leakage

# Create a clean feature list
model_features = [
    # Home team - all competitions form
    "points_last5_all_home",
    "goal_difference_last5_all_home",
    "win_rate_last5_all_home",
    "xg_difference_last5_all_home",
    "shots_on_target_difference_last5_all_home",

    # Away team - all competitions form
    "points_last5_all_away",
    "goal_difference_last5_all_away",
    "win_rate_last5_all_away",
    "xg_difference_last5_all_away",
    "shots_on_target_difference_last5_all_away",

    # Home team - Premier League form
    "points_last5_pl_home",
    "goal_difference_last5_pl_home",
    "win_rate_last5_pl_home",
    "xg_difference_last5_pl_home",
    "shots_on_target_difference_last5_pl_home",
    "points_last10_pl_home",
    "goal_difference_last10_pl_home",
    "win_rate_last10_pl_home",

    # Away team - Premier League form
    "points_last5_pl_away",
    "goal_difference_last5_pl_away",
    "win_rate_last5_pl_away",
    "xg_difference_last5_pl_away",
    "shots_on_target_difference_last5_pl_away",
    "points_last10_pl_away",
    "goal_difference_last10_pl_away",
    "win_rate_last10_pl_away",

    # Rest and fixture congestion
    "days_rest_home",
    "days_rest_away",
    "matches_last_7_days_home",
    "matches_last_7_days_away",
    "matches_last_14_days_home",
    "matches_last_14_days_away",

    # Pre-match Elo
    "team_elo_home",
    "team_elo_away",
    "team_elo_difference_home"
]

print("Number of model features:", len(model_features))

print()
print(pl_match_features[model_features].dtypes)

Number of model features: 35

points_last5_all_home                        float64
goal_difference_last5_all_home               float64
win_rate_last5_all_home                      float64
xg_difference_last5_all_home                 float64
shots_on_target_difference_last5_all_home    float64
points_last5_all_away                        float64
goal_difference_last5_all_away               float64
win_rate_last5_all_away                      float64
xg_difference_last5_all_away                 float64
shots_on_target_difference_last5_all_away    float64
points_last5_pl_home                         float64
goal_difference_last5_pl_home                float64
win_rate_last5_pl_home                       float64
xg_difference_last5_pl_home                  float64
shots_on_target_difference_last5_pl_home     float64
points_last10_pl_home                        float64
goal_difference_last10_pl_home               float64
win_rate_last10_pl_home                      float64
points_last5_pl_

In [95]:
# Check how much data is missing in each feature

missing_features = (
    pl_match_features[model_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_percent = (
    pl_match_features[model_features]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_summary = pd.DataFrame({
    "missing_count": missing_features,
    "missing_percent": missing_percent
})

print(missing_summary)

                                           missing_count  missing_percent
xg_difference_last5_pl_home                          774        33.947368
xg_difference_last5_all_home                         772        33.859649
xg_difference_last5_all_away                         771        33.815789
xg_difference_last5_pl_away                          771        33.815789
win_rate_last10_pl_home                               16         0.701754
points_last10_pl_home                                 16         0.701754
goal_difference_last10_pl_home                        16         0.701754
shots_on_target_difference_last5_pl_home              16         0.701754
win_rate_last5_pl_home                                16         0.701754
goal_difference_last5_pl_home                         16         0.701754
points_last5_pl_home                                  16         0.701754
win_rate_last10_pl_away                               12         0.526316
goal_difference_last10_pl_away        

In [96]:
# For the first baseline model, leave out xG since it's missing 33%

baseline_features = [
    feature
    for feature in model_features
    if "xg_" not in feature
]

print("Baseline features:", len(baseline_features))

print()
for feature in baseline_features:
    print(feature)

Baseline features: 31

points_last5_all_home
goal_difference_last5_all_home
win_rate_last5_all_home
shots_on_target_difference_last5_all_home
points_last5_all_away
goal_difference_last5_all_away
win_rate_last5_all_away
shots_on_target_difference_last5_all_away
points_last5_pl_home
goal_difference_last5_pl_home
win_rate_last5_pl_home
shots_on_target_difference_last5_pl_home
points_last10_pl_home
goal_difference_last10_pl_home
win_rate_last10_pl_home
points_last5_pl_away
goal_difference_last5_pl_away
win_rate_last5_pl_away
shots_on_target_difference_last5_pl_away
points_last10_pl_away
goal_difference_last10_pl_away
win_rate_last10_pl_away
days_rest_home
days_rest_away
matches_last_7_days_home
matches_last_7_days_away
matches_last_14_days_home
matches_last_14_days_away
team_elo_home
team_elo_away
team_elo_difference_home


In [97]:
# Check which matches are missing some of the baseline features

missing_baseline_rows = pl_match_features[
    pl_match_features[baseline_features]
    .isna()
    .any(axis=1)
]

print("Matches with at least one missing baseline feature:",
      len(missing_baseline_rows))

print()

print(
    missing_baseline_rows[[
        "date_home",
        "season_home",
        "team_home",
        "team_away"
    ]]
    .sort_values("date_home")
    .to_string(index=False)
)

Matches with at least one missing baseline feature: 20

                date_home season_home              team_home         team_away
2020-09-12 11:30:00+00:00     2020_21                 Fulham           Arsenal
2020-09-12 14:00:00+00:00     2020_21         Crystal Palace       Southampton
2020-09-12 16:30:00+00:00     2020_21              Liverpool      Leeds United
2020-09-12 19:00:00+00:00     2020_21        West Ham United  Newcastle United
2020-09-13 13:00:00+00:00     2020_21   West Bromwich Albion    Leicester City
2020-09-13 15:30:00+00:00     2020_21      Tottenham Hotspur           Everton
2020-09-14 17:00:00+00:00     2020_21       Sheffield United     Wolverhampton
2020-09-14 19:15:00+00:00     2020_21 Brighton & Hove Albion           Chelsea
2020-09-19 16:30:00+00:00     2020_21      Manchester United    Crystal Palace
2020-09-20 18:00:00+00:00     2020_21         Leicester City           Burnley
2020-09-21 17:00:00+00:00     2020_21            Aston Villa  Sheffield Uni

In [98]:
# Find out what's missing in those 20 matches

for _, row in missing_baseline_rows.sort_values("date_home").iterrows():
    missing = [
        feature
        for feature in baseline_features
        if pd.isna(row[feature])
    ]

    print(
        row["date_home"].date(),
        "|",
        row["team_home"],
        "vs",
        row["team_away"]
    )

    print("Missing:", missing)
    print()

2020-09-12 | Fulham vs Arsenal
Missing: ['points_last5_all_home', 'goal_difference_last5_all_home', 'win_rate_last5_all_home', 'shots_on_target_difference_last5_all_home', 'points_last5_all_away', 'goal_difference_last5_all_away', 'win_rate_last5_all_away', 'shots_on_target_difference_last5_all_away', 'points_last5_pl_home', 'goal_difference_last5_pl_home', 'win_rate_last5_pl_home', 'shots_on_target_difference_last5_pl_home', 'points_last10_pl_home', 'goal_difference_last10_pl_home', 'win_rate_last10_pl_home', 'points_last5_pl_away', 'goal_difference_last5_pl_away', 'win_rate_last5_pl_away', 'shots_on_target_difference_last5_pl_away', 'points_last10_pl_away', 'goal_difference_last10_pl_away', 'win_rate_last10_pl_away', 'days_rest_home', 'days_rest_away']

2020-09-12 | Crystal Palace vs Southampton
Missing: ['points_last5_all_home', 'goal_difference_last5_all_home', 'win_rate_last5_all_home', 'shots_on_target_difference_last5_all_home', 'points_last5_all_away', 'goal_difference_last5_al

In [99]:
# For the baseline, cleanest approach is to use median imputation fitted on the training data later

# Create CSV file with processed match dataset
FEATURES_PATH = PROCESSED_DIR / "pl_match_features.csv"

pl_match_features.to_csv(
    FEATURES_PATH,
    index=False
)

print("Saved:", FEATURES_PATH)
print("Shape:", pl_match_features.shape)

Saved: ..\data\processed\pl_match_features.csv
Shape: (2280, 140)


In [100]:
historical_df = pd.read_csv(
    "../data/processed/historical_matches.csv"
)

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True
)

pl_matches = historical_df[
    historical_df["competition"]
    == "premier_league"
].copy()


# Merge original pre-match Elo ratings
pl_matches = pl_matches.merge(
    elo_history_df,
    on="match_id",
    how="left"
)


home_team_rows = pd.DataFrame({
    "match_id": pl_matches["match_id"],
    "date": pl_matches["date"],
    "season": pl_matches["season"],

    "team": pl_matches["home_team"],
    "opponent": pl_matches["away_team"],

    "venue": "home",

    "goals_for": pl_matches["home_goals"],
    "goals_against": pl_matches["away_goals"],

    "team_elo": pl_matches["home_elo"],
    "opponent_elo": pl_matches["away_elo"]
})


away_team_rows = pd.DataFrame({
    "match_id": pl_matches["match_id"],
    "date": pl_matches["date"],
    "season": pl_matches["season"],

    "team": pl_matches["away_team"],
    "opponent": pl_matches["home_team"],

    "venue": "away",

    "goals_for": pl_matches["away_goals"],
    "goals_against": pl_matches["home_goals"],

    "team_elo": pl_matches["away_elo"],
    "opponent_elo": pl_matches["home_elo"]
})


pl_team_matches = pd.concat(
    [
        home_team_rows,
        away_team_rows
    ],
    ignore_index=True
)


pl_team_matches = (
    pl_team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


print(
    pl_team_matches.shape
)

print()

print(
    pl_team_matches.head()
)

print()

print(
    "Missing opponent Elo:",
    pl_team_matches[
        "opponent_elo"
    ].isna().sum()
)

(4560, 10)

       match_id                      date   season     team          opponent  \
0  mt_304225753 2020-09-12 11:30:00+00:00  2020_21  Arsenal            Fulham   
1  mt_304225642 2020-09-19 19:00:00+00:00  2020_21  Arsenal   West Ham United   
2  mt_138774968 2020-09-28 19:00:00+00:00  2020_21  Arsenal         Liverpool   
3  mt_291883557 2020-10-04 13:00:00+00:00  2020_21  Arsenal  Sheffield United   
4  mt_610558232 2020-10-17 16:30:00+00:00  2020_21  Arsenal   Manchester City   

  venue  goals_for  goals_against     team_elo  opponent_elo  
0  away          3              0  1500.000000   1500.000000  
1  home          2              1  1510.000000   1500.575011  
2  away          1              3  1529.728079   1530.000000  
3  home          2              1  1520.318268   1471.421744  
4  away          0              1  1528.920132   1518.866699  

Missing opponent Elo: 0


In [101]:
pl_team_matches[
    "opponent_elo_last5_pl"
] = (
    pl_team_matches
    .groupby("team")[
        "opponent_elo"
    ]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(
                window=5,
                min_periods=1
            )
            .mean()
        )
    )
)


print(
    pl_team_matches[
        [
            "team",
            "date",
            "opponent",
            "opponent_elo",
            "opponent_elo_last5_pl"
        ]
    ].head(12)
)

print()

print(
    "Missing values:",
    pl_team_matches[
        "opponent_elo_last5_pl"
    ].isna().sum()
)

       team                      date           opponent  opponent_elo  \
0   Arsenal 2020-09-12 11:30:00+00:00             Fulham   1500.000000   
1   Arsenal 2020-09-19 19:00:00+00:00    West Ham United   1500.575011   
2   Arsenal 2020-09-28 19:00:00+00:00          Liverpool   1530.000000   
3   Arsenal 2020-10-04 13:00:00+00:00   Sheffield United   1471.421744   
4   Arsenal 2020-10-17 16:30:00+00:00    Manchester City   1518.866699   
5   Arsenal 2020-10-25 19:15:00+00:00     Leicester City   1510.315657   
6   Arsenal 2020-11-01 16:30:00+00:00  Manchester United   1541.174416   
7   Arsenal 2020-11-08 19:15:00+00:00        Aston Villa   1525.805622   
8   Arsenal 2020-11-22 16:30:00+00:00       Leeds United   1491.565720   
9   Arsenal 2020-11-29 19:15:00+00:00      Wolverhampton   1503.190005   
10  Arsenal 2020-12-06 16:30:00+00:00  Tottenham Hotspur   1583.995176   
11  Arsenal 2020-12-13 19:15:00+00:00            Burnley   1462.981151   

    opponent_elo_last5_pl  
0        

In [102]:
home_opponent_strength = (
    pl_team_matches[
        pl_team_matches["venue"] == "home"
    ][
        [
            "match_id",
            "opponent_elo_last5_pl"
        ]
    ]
    .rename(
        columns={
            "opponent_elo_last5_pl":
            "opponent_elo_last5_pl_home"
        }
    )
)


away_opponent_strength = (
    pl_team_matches[
        pl_team_matches["venue"] == "away"
    ][
        [
            "match_id",
            "opponent_elo_last5_pl"
        ]
    ]
    .rename(
        columns={
            "opponent_elo_last5_pl":
            "opponent_elo_last5_pl_away"
        }
    )
)


opponent_strength_features = (
    home_opponent_strength
    .merge(
        away_opponent_strength,
        on="match_id",
        how="inner"
    )
)


print(
    opponent_strength_features.shape
)

print()

print(
    opponent_strength_features.head(10)
)

print()

print(
    "Duplicate match IDs:",
    opponent_strength_features[
        "match_id"
    ].duplicated().sum()
)

print(
    "Missing home:",
    opponent_strength_features[
        "opponent_elo_last5_pl_home"
    ].isna().sum()
)

print(
    "Missing away:",
    opponent_strength_features[
        "opponent_elo_last5_pl_away"
    ].isna().sum()
)

(2280, 3)

       match_id  opponent_elo_last5_pl_home  opponent_elo_last5_pl_away
0  mt_304225642                 1500.000000                 1500.000000
1  mt_291883557                 1510.191670                 1503.527916
2  mt_452119541                 1504.172691                 1512.100995
3  mt_579336752                 1514.355703                 1510.862802
4  mt_304225022                 1517.545623                 1522.542975
5  mt_291883007                 1529.146188                 1522.072075
6  mt_579336119                 1513.507535                 1498.023009
7  mt_847662934                 1526.036092                 1528.725398
8  mt_291883606                 1513.127768                 1518.633099
9  mt_847662528                 1503.424067                 1519.423685

Duplicate match IDs: 0
Missing home: 16
Missing away: 12


In [103]:
opponent_strength_features.to_csv(
    "../data/processed/opponent_strength_features.csv",
    index=False
)

print(
    "Saved:",
    opponent_strength_features.shape
)

Saved: (2280, 3)


In [104]:
# Build match expectation features
# Giving a form relative to expected performance view

pl_team_matches[
    "elo_expected_score"
] = (
    1
    / (
        1
        + 10 ** (
            (
                pl_team_matches["opponent_elo"]
                - pl_team_matches["team_elo"]
            )
            / 400
        )
    )
)


pl_team_matches[
    "actual_score"
] = np.select(
    [
        pl_team_matches["goals_for"]
        > pl_team_matches["goals_against"],

        pl_team_matches["goals_for"]
        == pl_team_matches["goals_against"]
    ],
    [
        1.0,
        0.5
    ],
    default=0.0
)


pl_team_matches[
    "elo_performance_residual"
] = (
    pl_team_matches["actual_score"]
    - pl_team_matches["elo_expected_score"]
)


print(
    pl_team_matches[
        [
            "team",
            "opponent",
            "team_elo",
            "opponent_elo",
            "elo_expected_score",
            "actual_score",
            "elo_performance_residual"
        ]
    ].head(15)
)

print()

print(
    pl_team_matches[
        "elo_performance_residual"
    ].describe()
)

       team           opponent     team_elo  opponent_elo  elo_expected_score  \
0   Arsenal             Fulham  1500.000000   1500.000000            0.500000   
1   Arsenal    West Ham United  1510.000000   1500.575011            0.513560   
2   Arsenal          Liverpool  1529.728079   1530.000000            0.499609   
3   Arsenal   Sheffield United  1520.318268   1471.421744            0.569907   
4   Arsenal    Manchester City  1528.920132   1518.866699            0.514464   
5   Arsenal     Leicester City  1528.086868   1510.315657            0.525553   
6   Arsenal  Manchester United  1527.079303   1541.174416            0.479727   
7   Arsenal        Aston Villa  1547.530664   1525.805622            0.531224   
8   Arsenal       Leeds United  1536.906180   1491.565720            0.564882   
9   Arsenal      Wolverhampton  1545.419309   1503.190005            0.560475   
10  Arsenal  Tottenham Hotspur  1543.266884   1583.995176            0.441654   
11  Arsenal            Burnl

In [105]:
# Turn this into recent form vs expectation
# One surprising result doesn't mean a team is genuinely improving
# Averaging the previous 5 residuals gives a smoother measure of whether a team has consistently performed above or below what the underlying ELO strength predicted

pl_team_matches[
    "elo_performance_last5_pl"
] = (
    pl_team_matches
    .groupby("team")[
        "elo_performance_residual"
    ]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(
                window=5,
                min_periods=1
            )
            .mean()
        )
    )
)


print(
    pl_team_matches[
        [
            "team",
            "date",
            "opponent",
            "elo_expected_score",
            "actual_score",
            "elo_performance_residual",
            "elo_performance_last5_pl"
        ]
    ].head(15)
)

print()

print(
    "Missing values:",
    pl_team_matches[
        "elo_performance_last5_pl"
    ].isna().sum()
)

print()

print(
    pl_team_matches[
        "elo_performance_last5_pl"
    ].describe()
)

       team                      date           opponent  elo_expected_score  \
0   Arsenal 2020-09-12 11:30:00+00:00             Fulham            0.500000   
1   Arsenal 2020-09-19 19:00:00+00:00    West Ham United            0.513560   
2   Arsenal 2020-09-28 19:00:00+00:00          Liverpool            0.499609   
3   Arsenal 2020-10-04 13:00:00+00:00   Sheffield United            0.569907   
4   Arsenal 2020-10-17 16:30:00+00:00    Manchester City            0.514464   
5   Arsenal 2020-10-25 19:15:00+00:00     Leicester City            0.525553   
6   Arsenal 2020-11-01 16:30:00+00:00  Manchester United            0.479727   
7   Arsenal 2020-11-08 19:15:00+00:00        Aston Villa            0.531224   
8   Arsenal 2020-11-22 16:30:00+00:00       Leeds United            0.564882   
9   Arsenal 2020-11-29 19:15:00+00:00      Wolverhampton            0.560475   
10  Arsenal 2020-12-06 16:30:00+00:00  Tottenham Hotspur            0.441654   
11  Arsenal 2020-12-13 19:15:00+00:00   

In [106]:
# Create and save the two features
# RF needs one row per match, but pl_team_matches has one row per team appearance. 
# Splitting the rolling residual into home and away columns lets the model compare whetehr each team has been overperforming or underperforming its ELO expectations

home_elo_performance = (
    pl_team_matches[
        pl_team_matches["venue"] == "home"
    ][
        [
            "match_id",
            "elo_performance_last5_pl"
        ]
    ]
    .rename(
        columns={
            "elo_performance_last5_pl":
            "elo_performance_last5_pl_home"
        }
    )
)


away_elo_performance = (
    pl_team_matches[
        pl_team_matches["venue"] == "away"
    ][
        [
            "match_id",
            "elo_performance_last5_pl"
        ]
    ]
    .rename(
        columns={
            "elo_performance_last5_pl":
            "elo_performance_last5_pl_away"
        }
    )
)


elo_performance_features = (
    home_elo_performance
    .merge(
        away_elo_performance,
        on="match_id",
        how="inner"
    )
)


print(
    elo_performance_features.shape
)

print()

print(
    elo_performance_features.head(10)
)

print()

print(
    "Duplicate match IDs:",
    elo_performance_features[
        "match_id"
    ].duplicated().sum()
)

print(
    "Missing home:",
    elo_performance_features[
        "elo_performance_last5_pl_home"
    ].isna().sum()
)

print(
    "Missing away:",
    elo_performance_features[
        "elo_performance_last5_pl_away"
    ].isna().sum()
)


elo_performance_features.to_csv(
    "../data/processed/elo_performance_features.csv",
    index=False
)

print()
print(
    "Saved elo_performance_features.csv"
)

(2280, 3)

       match_id  elo_performance_last5_pl_home  elo_performance_last5_pl_away
0  mt_304225642                       0.500000                      -0.500000
1  mt_291883557                       0.162277                      -0.481100
2  mt_452119541                       0.080492                       0.103243
3  mt_579336752                      -0.117852                       0.063275
4  mt_304225022                      -0.223170                       0.130658
5  mt_291883007                      -0.215593                      -0.013912
6  mt_579336119                      -0.442112                       0.167128
7  mt_847662934                      -0.417232                      -0.063311
8  mt_291883606                       0.182073                      -0.063501
9  mt_847662528                       0.165740                      -0.276641

Duplicate match IDs: 0
Missing home: 16
Missing away: 12

Saved elo_performance_features.csv


In [108]:
# ---------------------------------------------------------
# COUNT ONLY PREMIER LEAGUE TEAMS
# ---------------------------------------------------------

pl_european_team_rows = []


for _, row in european_matches.iterrows():

    # Count home team only if it is one of our PL teams
    if row["home_team"] in premier_league_teams:

        pl_european_team_rows.append({
            "team": row["home_team"],
            "season": row["season"],
            "competition": row["competition"],
            "match_id": row["match_id"]
        })


    # Count away team only if it is one of our PL teams
    if row["away_team"] in premier_league_teams:

        pl_european_team_rows.append({
            "team": row["away_team"],
            "season": row["season"],
            "competition": row["competition"],
            "match_id": row["match_id"]
        })


pl_european_team_df = pd.DataFrame(
    pl_european_team_rows
)


# ---------------------------------------------------------
# OVERALL COVERAGE
# ---------------------------------------------------------

team_european_counts = (
    pl_european_team_df[
        "team"
    ]
    .value_counts()
)


print("EUROPEAN MATCHES BY PL TEAM")
print()

print(
    team_european_counts
)


print()

print(
    "PL teams with European matches:",
    pl_european_team_df[
        "team"
    ].nunique()
)

print(
    "Total PL teams:",
    len(premier_league_teams)
)


# ---------------------------------------------------------
# TEAM-SEASON COVERAGE
# ---------------------------------------------------------

print()
print("EUROPEAN MATCHES BY TEAM AND SEASON")
print()


team_season_europe = (
    pl_european_team_df
    .groupby(
        [
            "season",
            "team"
        ]
    )
    .size()
    .reset_index(
        name="european_matches"
    )
)


print(
    team_season_europe.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# NUMBER OF PL CLUBS IN EUROPE EACH SEASON
# ---------------------------------------------------------

print()
print("PL CLUBS WITH EUROPEAN MATCHES BY SEASON")
print()


clubs_per_season = (
    pl_european_team_df
    .groupby(
        "season"
    )["team"]
    .nunique()
)


print(
    clubs_per_season
)

EUROPEAN MATCHES BY PL TEAM

team
Manchester City           68
Liverpool                 63
Arsenal                   61
Chelsea                   58
Manchester United         56
Tottenham Hotspur         52
Aston Villa               41
West Ham United           35
Leicester City            22
Newcastle United          18
Crystal Palace            17
Nottingham Forest         16
Brighton & Hove Albion     8
Name: count, dtype: int64

PL teams with European matches: 13
Total PL teams: 28

EUROPEAN MATCHES BY TEAM AND SEASON

 season                   team  european_matches
2020_21                Arsenal                14
2020_21                Chelsea                13
2020_21         Leicester City                 8
2020_21              Liverpool                10
2020_21        Manchester City                13
2020_21      Manchester United                15
2020_21      Tottenham Hotspur                13
2021_22                Chelsea                10
2021_22         Leicester Cit

In [109]:
# ---------------------------------------------------------
# EUROPEAN MATCHES FROM TEAM PERSPECTIVE
# PL CLUBS ONLY
# ---------------------------------------------------------

european_team_rows = []


for _, row in european_matches.iterrows():

    # PL club was the home team
    if row["home_team"] in premier_league_teams:

        european_team_rows.append({
            "match_id": row["match_id"],
            "date": row["date"],
            "season": row["season"],
            "competition": row["competition"],
            "team": row["home_team"],
            "opponent": row["away_team"],
            "venue": "home",
            "goals_for": row["home_goals"],
            "goals_against": row["away_goals"]
        })


    # PL club was the away team
    if row["away_team"] in premier_league_teams:

        european_team_rows.append({
            "match_id": row["match_id"],
            "date": row["date"],
            "season": row["season"],
            "competition": row["competition"],
            "team": row["away_team"],
            "opponent": row["home_team"],
            "venue": "away",
            "goals_for": row["away_goals"],
            "goals_against": row["home_goals"]
        })


european_team_matches = pd.DataFrame(
    european_team_rows
)


european_team_matches["date"] = pd.to_datetime(
    european_team_matches["date"],
    utc=True
)


# ---------------------------------------------------------
# ADD PRE-MATCH ELO
# ---------------------------------------------------------

european_team_matches = (
    european_team_matches
    .merge(
        elo_history_df[
            [
                "match_id",
                "home_elo",
                "away_elo"
            ]
        ],
        on="match_id",
        how="left"
    )
)


european_team_matches[
    "team_elo"
] = np.where(
    european_team_matches["venue"] == "home",
    european_team_matches["home_elo"],
    european_team_matches["away_elo"]
)


european_team_matches[
    "opponent_elo"
] = np.where(
    european_team_matches["venue"] == "home",
    european_team_matches["away_elo"],
    european_team_matches["home_elo"]
)


european_team_matches = (
    european_team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Shape:",
    european_team_matches.shape
)

print()

print(
    european_team_matches[
        [
            "date",
            "team",
            "opponent",
            "competition",
            "venue",
            "team_elo",
            "opponent_elo",
            "goals_for",
            "goals_against"
        ]
    ].head(20)
)


print()

print(
    "Missing team Elo:",
    european_team_matches[
        "team_elo"
    ].isna().sum()
)

print(
    "Missing opponent Elo:",
    european_team_matches[
        "opponent_elo"
    ].isna().sum()
)

print()

print(
    "Rows:",
    len(european_team_matches)
)

print(
    "Unique PL teams:",
    european_team_matches[
        "team"
    ].nunique()
)

Shape: (515, 13)

                        date     team         opponent    competition venue  \
0  2020-10-22 16:55:00+00:00  Arsenal    SK Rapid Wien  europa_league  away   
1  2020-10-29 20:00:00+00:00  Arsenal       Dundalk FC  europa_league  home   
2  2020-11-05 20:00:00+00:00  Arsenal         Molde FK  europa_league  home   
3  2020-11-26 17:55:00+00:00  Arsenal         Molde FK  europa_league  away   
4  2020-12-03 20:00:00+00:00  Arsenal    SK Rapid Wien  europa_league  home   
5  2020-12-10 17:55:00+00:00  Arsenal       Dundalk FC  europa_league  away   
6  2021-02-18 20:00:00+00:00  Arsenal          Benfica  europa_league  away   
7  2021-02-25 17:55:00+00:00  Arsenal          Benfica  europa_league  home   
8  2021-03-11 20:00:00+00:00  Arsenal    Olympiacos FC  europa_league  away   
9  2021-03-18 17:55:00+00:00  Arsenal    Olympiacos FC  europa_league  home   
10 2021-04-08 19:00:00+00:00  Arsenal  SK Slavia Praha  europa_league  home   
11 2021-04-15 19:00:00+00:00  Arse

In [110]:
# Calculate European performance vs ELO expectation

EUROPE_HOME_ADVANTAGE = 40


# ---------------------------------------------------------
# VENUE-ADJUSTED ELO EXPECTATION
# ---------------------------------------------------------

european_team_matches[
    "venue_elo_adjustment"
] = np.where(
    european_team_matches["venue"] == "home",
    EUROPE_HOME_ADVANTAGE,
    -EUROPE_HOME_ADVANTAGE
)


european_team_matches[
    "elo_expected_score"
] = (
    1
    / (
        1
        + 10 ** (
            (
                european_team_matches["opponent_elo"]
                - (
                    european_team_matches["team_elo"]
                    + european_team_matches["venue_elo_adjustment"]
                )
            )
            / 400
        )
    )
)


# ---------------------------------------------------------
# ACTUAL MATCH RESULT
# ---------------------------------------------------------

european_team_matches[
    "actual_score"
] = np.select(
    [
        european_team_matches["goals_for"]
        > european_team_matches["goals_against"],

        european_team_matches["goals_for"]
        == european_team_matches["goals_against"]
    ],
    [
        1.0,
        0.5
    ],
    default=0.0
)


# ---------------------------------------------------------
# PERFORMANCE RELATIVE TO EXPECTATION
# ---------------------------------------------------------

european_team_matches[
    "european_elo_residual"
] = (
    european_team_matches["actual_score"]
    - european_team_matches["elo_expected_score"]
)


# ---------------------------------------------------------
# INSPECT RESULTS
# ---------------------------------------------------------

print(
    european_team_matches[
        [
            "date",
            "team",
            "opponent",
            "venue",
            "team_elo",
            "opponent_elo",
            "elo_expected_score",
            "actual_score",
            "european_elo_residual"
        ]
    ].head(20)
)


print()
print("RESIDUAL SUMMARY")
print()

print(
    european_team_matches[
        "european_elo_residual"
    ].describe()
)


print()
print("AVERAGE BY VENUE")
print()

print(
    european_team_matches
    .groupby("venue")[
        [
            "elo_expected_score",
            "actual_score",
            "european_elo_residual"
        ]
    ]
    .mean()
)

                        date     team         opponent venue     team_elo  \
0  2020-10-22 16:55:00+00:00  Arsenal    SK Rapid Wien  away  1518.630852   
1  2020-10-29 20:00:00+00:00  Arsenal       Dundalk FC  home  1517.575817   
2  2020-11-05 20:00:00+00:00  Arsenal         Molde FK  home  1537.484771   
3  2020-11-26 17:55:00+00:00  Arsenal         Molde FK  away  1535.608535   
4  2020-12-03 20:00:00+00:00  Arsenal    SK Rapid Wien  home  1534.209803   
5  2020-12-10 17:55:00+00:00  Arsenal       Dundalk FC  away  1534.433798   
6  2021-02-18 20:00:00+00:00  Arsenal          Benfica  away  1533.928594   
7  2021-02-25 17:55:00+00:00  Arsenal          Benfica  home  1527.915889   
8  2021-03-11 20:00:00+00:00  Arsenal    Olympiacos FC  away  1547.556006   
9  2021-03-18 17:55:00+00:00  Arsenal    Olympiacos FC  home  1567.109290   
10 2021-04-08 19:00:00+00:00  Arsenal  SK Slavia Praha  home  1545.045343   
11 2021-04-15 19:00:00+00:00  Arsenal  SK Slavia Praha  away  1551.205002   

In [111]:
# ---------------------------------------------------------
# ROLLING EUROPEAN PERFORMANCE
# ---------------------------------------------------------

european_team_matches = (
    european_team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


european_team_matches[
    "european_performance_last5"
] = (
    european_team_matches
    .groupby("team")[
        "european_elo_residual"
    ]
    .transform(
        lambda x: (
            x.rolling(
                window=5,
                min_periods=1
            )
            .mean()
        )
    )
)


# Number of European matches represented
# in the rolling average
european_team_matches[
    "european_matches_in_window"
] = (
    european_team_matches
    .groupby("team")
    .cumcount()
    .add(1)
    .clip(upper=5)
)


print(
    european_team_matches[
        [
            "date",
            "team",
            "opponent",
            "competition",
            "european_elo_residual",
            "european_performance_last5",
            "european_matches_in_window"
        ]
    ].head(25)
)


print()
print("ROLLING PERFORMANCE SUMMARY")
print()

print(
    european_team_matches[
        "european_performance_last5"
    ].describe()
)


print()
print("WINDOW SIZE COUNTS")
print()

print(
    european_team_matches[
        "european_matches_in_window"
    ].value_counts()
    .sort_index()
)

                        date     team         opponent       competition  \
0  2020-10-22 16:55:00+00:00  Arsenal    SK Rapid Wien     europa_league   
1  2020-10-29 20:00:00+00:00  Arsenal       Dundalk FC     europa_league   
2  2020-11-05 20:00:00+00:00  Arsenal         Molde FK     europa_league   
3  2020-11-26 17:55:00+00:00  Arsenal         Molde FK     europa_league   
4  2020-12-03 20:00:00+00:00  Arsenal    SK Rapid Wien     europa_league   
5  2020-12-10 17:55:00+00:00  Arsenal       Dundalk FC     europa_league   
6  2021-02-18 20:00:00+00:00  Arsenal          Benfica     europa_league   
7  2021-02-25 17:55:00+00:00  Arsenal          Benfica     europa_league   
8  2021-03-11 20:00:00+00:00  Arsenal    Olympiacos FC     europa_league   
9  2021-03-18 17:55:00+00:00  Arsenal    Olympiacos FC     europa_league   
10 2021-04-08 19:00:00+00:00  Arsenal  SK Slavia Praha     europa_league   
11 2021-04-15 19:00:00+00:00  Arsenal  SK Slavia Praha     europa_league   
12 2021-04-2

In [112]:
# ---------------------------------------------------------
# PREPARE PREMIER LEAGUE TEAM-MATCH ROWS
# ---------------------------------------------------------

pl_for_europe_merge = (
    pl_team_matches[
        [
            "match_id",
            "date",
            "season",
            "team",
            "venue"
        ]
    ]
    .copy()
)


pl_for_europe_merge["date"] = pd.to_datetime(
    pl_for_europe_merge["date"],
    utc=True
)


# ---------------------------------------------------------
# PREPARE EUROPEAN HISTORY
# ---------------------------------------------------------

europe_history = (
    european_team_matches[
        [
            "team",
            "date",
            "european_performance_last5",
            "european_matches_in_window"
        ]
    ]
    .copy()
    .rename(
        columns={
            "date": "last_european_date"
        }
    )
)


# ---------------------------------------------------------
# MERGE EACH TEAM SEPARATELY
#
# This avoids merge_asof sorting problems when multiple
# teams are mixed together.
# ---------------------------------------------------------

merged_team_parts = []


for team_name in sorted(
    pl_for_europe_merge["team"].unique()
):

    team_pl = (
        pl_for_europe_merge[
            pl_for_europe_merge["team"]
            == team_name
        ]
        .copy()
        .sort_values("date")
    )


    team_europe = (
        europe_history[
            europe_history["team"]
            == team_name
        ]
        .copy()
        .sort_values(
            "last_european_date"
        )
    )


    # Teams with no European history
    if team_europe.empty:

        team_pl[
            "last_european_date"
        ] = pd.NaT

        team_pl[
            "european_performance_last5"
        ] = np.nan

        team_pl[
            "european_matches_in_window"
        ] = np.nan


    else:

        team_pl = pd.merge_asof(
            team_pl,
            team_europe[
                [
                    "last_european_date",
                    "european_performance_last5",
                    "european_matches_in_window"
                ]
            ],
            left_on="date",
            right_on="last_european_date",
            direction="backward",
            allow_exact_matches=False
        )


        team_pl["team"] = team_name


    merged_team_parts.append(
        team_pl
    )


pl_europe_features = pd.concat(
    merged_team_parts,
    ignore_index=True
)


# ---------------------------------------------------------
# DAYS SINCE LAST EUROPEAN MATCH
# ---------------------------------------------------------

pl_europe_features[
    "days_since_european_match"
] = (
    (
        pl_europe_features["date"]
        - pl_europe_features["last_european_date"]
    )
    .dt.total_seconds()
    / 86400
)


# ---------------------------------------------------------
# SANITY CHECKS
# ---------------------------------------------------------

print(
    "Shape:",
    pl_europe_features.shape
)

print(
    "Duplicate team-match rows:",
    pl_europe_features[
        [
            "match_id",
            "team"
        ]
    ].duplicated().sum()
)

print()

print(
    "Rows with prior European information:",
    pl_europe_features[
        "european_performance_last5"
    ].notna().sum()
)

print(
    "Percentage with European information:",
    pl_europe_features[
        "european_performance_last5"
    ].notna().mean()
    * 100
)

print()

print(
    "Negative days since Europe:",
    (
        pl_europe_features[
            "days_since_european_match"
        ]
        < 0
    ).sum()
)


print()
print("SAMPLE ARSENAL ROWS")
print()

print(
    pl_europe_features[
        (
            pl_europe_features["team"]
            == "Arsenal"
        )
        &
        (
            pl_europe_features[
                "european_performance_last5"
            ].notna()
        )
    ][
        [
            "date",
            "team",
            "last_european_date",
            "days_since_european_match",
            "european_performance_last5",
            "european_matches_in_window"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

C:\Users\luisg\AppData\Local\Temp\ipykernel_25972\3025105705.py:139: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  pl_europe_features["date"]


AttributeError: Can only use .dt accessor with datetimelike values

In [113]:
# ---------------------------------------------------------
# FIX DATE TYPES
# ---------------------------------------------------------

pl_europe_features["date"] = pd.to_datetime(
    pl_europe_features["date"],
    utc=True
)

pl_europe_features["last_european_date"] = pd.to_datetime(
    pl_europe_features["last_european_date"],
    utc=True,
    errors="coerce"
)


# ---------------------------------------------------------
# DAYS SINCE LAST EUROPEAN MATCH
# ---------------------------------------------------------

pl_europe_features[
    "days_since_european_match"
] = (
    (
        pl_europe_features["date"]
        - pl_europe_features["last_european_date"]
    )
    .dt.total_seconds()
    / 86400
)


# ---------------------------------------------------------
# SANITY CHECKS
# ---------------------------------------------------------

print(
    "Shape:",
    pl_europe_features.shape
)

print(
    "Duplicate team-match rows:",
    pl_europe_features[
        [
            "match_id",
            "team"
        ]
    ].duplicated().sum()
)

print()

print(
    "Rows with prior European information:",
    pl_europe_features[
        "european_performance_last5"
    ].notna().sum()
)

print(
    "Percentage with European information:",
    pl_europe_features[
        "european_performance_last5"
    ].notna().mean()
    * 100
)

print()

print(
    "Negative days since Europe:",
    (
        pl_europe_features[
            "days_since_european_match"
        ]
        < 0
    ).sum()
)


print()
print("SAMPLE ARSENAL ROWS")
print()

print(
    pl_europe_features[
        (
            pl_europe_features["team"]
            == "Arsenal"
        )
        &
        (
            pl_europe_features[
                "european_performance_last5"
            ].notna()
        )
    ][
        [
            "date",
            "team",
            "last_european_date",
            "days_since_european_match",
            "european_performance_last5",
            "european_matches_in_window"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

Shape: (4560, 9)
Duplicate team-match rows: 0

Rows with prior European information: 2077
Percentage with European information: 45.54824561403509

Negative days since Europe: 0

SAMPLE ARSENAL ROWS

                     date    team        last_european_date  days_since_european_match  european_performance_last5  european_matches_in_window
2020-10-25 19:15:00+00:00 Arsenal 2020-10-22 16:55:00+00:00                   3.097222                    0.530301                         1.0
2020-11-01 16:30:00+00:00 Arsenal 2020-10-29 20:00:00+00:00                   2.854167                    0.474314                         2.0
2020-11-08 19:15:00+00:00 Arsenal 2020-11-05 20:00:00+00:00                   2.968750                    0.464528                         3.0
2020-11-22 16:30:00+00:00 Arsenal 2020-11-05 20:00:00+00:00                  16.854167                    0.464528                         3.0
2020-11-29 19:15:00+00:00 Arsenal 2020-11-26 17:55:00+00:00                   3.055556

In [114]:
# ---------------------------------------------------------
# EUROPEAN FEATURE FRESHNESS
# ---------------------------------------------------------

freshness = (
    pl_europe_features[
        pl_europe_features[
            "days_since_european_match"
        ].notna()
    ][
        "days_since_european_match"
    ]
)


print("FRESHNESS SUMMARY")
print()

print(
    freshness.describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.80,
            0.90,
            0.95
        ]
    )
)


print()
print("COVERAGE AT DIFFERENT FRESHNESS CUTOFFS")
print()


for cutoff in [
    7,
    14,
    21,
    30,
    45,
    60,
    90,
    180
]:

    fresh_rows = (
        pl_europe_features[
            "days_since_european_match"
        ]
        <= cutoff
    ).sum()

    percentage = (
        fresh_rows
        / len(pl_europe_features)
        * 100
    )

    print(
        f"{cutoff:>3} days: "
        f"{fresh_rows:>4} rows "
        f"({percentage:.2f}%)"
    )


# ---------------------------------------------------------
# FRESHNESS BY SEASON
# Using 30 days only as a diagnostic for now
# ---------------------------------------------------------

print()
print("30-DAY COVERAGE BY SEASON")
print()


coverage_30 = (
    pl_europe_features
    .assign(
        fresh_europe_30=(
            pl_europe_features[
                "days_since_european_match"
            ]
            <= 30
        )
    )
    .groupby("season")[
        "fresh_europe_30"
    ]
    .agg(
        [
            "sum",
            "mean"
        ]
    )
)


coverage_30["mean"] = (
    coverage_30["mean"]
    * 100
)


print(
    coverage_30.rename(
        columns={
            "sum": "rows",
            "mean": "percentage"
        }
    )
)

FRESHNESS SUMMARY

count    2077.000000
mean      112.845315
std       194.502761
min         2.000000
25%         7.010417
50%        23.812500
75%       120.791667
80%       168.802083
90%       353.633333
95%       561.948611
max      1115.833333
Name: days_since_european_match, dtype: float64

COVERAGE AT DIFFERENT FRESHNESS CUTOFFS

  7 days:  515 rows (11.29%)
 14 days:  778 rows (17.06%)
 21 days:  998 rows (21.89%)
 30 days: 1113 rows (24.41%)
 45 days: 1235 rows (27.08%)
 60 days: 1337 rows (29.32%)
 90 days: 1474 rows (32.32%)
180 days: 1680 rows (36.84%)

30-DAY COVERAGE BY SEASON

         rows  percentage
season                   
2020_21   165   21.710526
2021_22   178   23.421053
2022_23   127   16.710526
2023_24   183   24.078947
2024_25   203   26.710526
2025_26   257   33.815789


In [115]:
EUROPE_FRESHNESS_DAYS = 30


# ---------------------------------------------------------
# CREATE FRESH EUROPEAN FEATURES
# ---------------------------------------------------------

pl_europe_features[
    "recent_europe"
] = (
    pl_europe_features[
        "days_since_european_match"
    ]
    <= EUROPE_FRESHNESS_DAYS
).astype(int)


pl_europe_features[
    "european_performance_recent"
] = np.where(
    pl_europe_features[
        "recent_europe"
    ] == 1,
    pl_europe_features[
        "european_performance_last5"
    ],
    np.nan
)


# ---------------------------------------------------------
# SPLIT HOME AND AWAY
# ---------------------------------------------------------

europe_home = (
    pl_europe_features[
        pl_europe_features["venue"]
        == "home"
    ][
        [
            "match_id",
            "recent_europe",
            "european_performance_recent",
            "days_since_european_match"
        ]
    ]
    .rename(
        columns={
            "recent_europe":
                "recent_europe_home",

            "european_performance_recent":
                "european_performance_recent_home",

            "days_since_european_match":
                "days_since_european_match_home"
        }
    )
)


europe_away = (
    pl_europe_features[
        pl_europe_features["venue"]
        == "away"
    ][
        [
            "match_id",
            "recent_europe",
            "european_performance_recent",
            "days_since_european_match"
        ]
    ]
    .rename(
        columns={
            "recent_europe":
                "recent_europe_away",

            "european_performance_recent":
                "european_performance_recent_away",

            "days_since_european_match":
                "days_since_european_match_away"
        }
    )
)


# ---------------------------------------------------------
# COMBINE INTO ONE ROW PER PL MATCH
# ---------------------------------------------------------

europe_match_features = (
    europe_home
    .merge(
        europe_away,
        on="match_id",
        how="inner"
    )
)


print(
    "Shape:",
    europe_match_features.shape
)

print(
    "Duplicate match IDs:",
    europe_match_features[
        "match_id"
    ].duplicated().sum()
)


print()
print("RECENT EUROPE COUNTS")
print()

print(
    "Home team:",
    europe_match_features[
        "recent_europe_home"
    ].sum()
)

print(
    "Away team:",
    europe_match_features[
        "recent_europe_away"
    ].sum()
)


print()
print("MATCH COVERAGE")
print()

at_least_one = (
    (
        europe_match_features[
            "recent_europe_home"
        ] == 1
    )
    |
    (
        europe_match_features[
            "recent_europe_away"
        ] == 1
    )
)

both = (
    (
        europe_match_features[
            "recent_europe_home"
        ] == 1
    )
    &
    (
        europe_match_features[
            "recent_europe_away"
        ] == 1
    )
)


print(
    "At least one team recently in Europe:",
    at_least_one.sum(),
    f"({at_least_one.mean() * 100:.2f}%)"
)

print(
    "Both teams recently in Europe:",
    both.sum(),
    f"({both.mean() * 100:.2f}%)"
)


print()
print("MISSING RECENT PERFORMANCE")
print()

print(
    europe_match_features[
        [
            "european_performance_recent_home",
            "european_performance_recent_away"
        ]
    ]
    .isna()
    .sum()
)


# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

europe_match_features.to_csv(
    "../data/processed/european_performance_features.csv",
    index=False
)

print()
print(
    "Saved:",
    "../data/processed/european_performance_features.csv"
)

Shape: (2280, 7)
Duplicate match IDs: 0

RECENT EUROPE COUNTS

Home team: 558
Away team: 555

MATCH COVERAGE

At least one team recently in Europe: 945 (41.45%)
Both teams recently in Europe: 168 (7.37%)

MISSING RECENT PERFORMANCE

european_performance_recent_home    1722
european_performance_recent_away    1725
dtype: int64

Saved: ../data/processed/european_performance_features.csv


In [116]:
# Look beyond raw pre match statistics

# ---------------------------------------------------------
# BUILD ADDITIONAL MATCH-DOMINANCE DIFFERENCES
# ---------------------------------------------------------

team_matches[
    "shot_difference"
] = (
    team_matches["shots_for"]
    - team_matches["shots_against"]
)


team_matches[
    "possession_difference"
] = (
    team_matches["possession_for"]
    - team_matches["possession_against"]
)


team_matches[
    "corner_difference"
] = (
    team_matches["corners_for"]
    - team_matches["corners_against"]
)


# ---------------------------------------------------------
# LEAKAGE-SAFE ROLLING LAST 5
# ---------------------------------------------------------

for stat in [
    "shot_difference",
    "possession_difference",
    "corner_difference"
]:

    team_matches[
        f"{stat}_last5_all"
    ] = (
        team_matches
        .groupby("team")[stat]
        .transform(
            lambda x: (
                x.shift(1)
                .rolling(
                    window=5,
                    min_periods=1
                )
                .mean()
            )
        )
    )


# ---------------------------------------------------------
# PREMIER LEAGUE ROWS ONLY
# ---------------------------------------------------------

dominance_check = (
    team_matches[
        team_matches["competition"]
        == "premier_league"
    ][
        [
            "match_id",
            "date",
            "season",
            "team",
            "venue",
            "points",
            "goal_difference",
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .copy()
)


print(
    "Shape:",
    dominance_check.shape
)


print()
print("MISSING VALUES")
print()

print(
    dominance_check[
        [
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .isna()
    .sum()
)


print()
print("CORRELATION WITH NEXT-MATCH PERFORMANCE")
print()

print(
    dominance_check[
        [
            "points",
            "goal_difference",
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .corr()
    .round(3)
)


print()
print("FEATURE SUMMARY")
print()

print(
    dominance_check[
        [
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .describe()
)

Shape: (4560, 10)

MISSING VALUES

shot_difference_last5_all          18
possession_difference_last5_all    18
corner_difference_last5_all        18
dtype: int64

CORRELATION WITH NEXT-MATCH PERFORMANCE

                                 points  goal_difference  \
points                            1.000            0.850   
goal_difference                   0.850            1.000   
shot_difference_last5_all         0.209            0.230   
possession_difference_last5_all   0.200            0.220   
corner_difference_last5_all       0.164            0.188   

                                 shot_difference_last5_all  \
points                                               0.209   
goal_difference                                      0.230   
shot_difference_last5_all                            1.000   
possession_difference_last5_all                      0.725   
corner_difference_last5_all                          0.771   

                                 possession_difference_last5_a

In [117]:
# Test these signals together and individually

# ---------------------------------------------------------
# KEEP PREMIER LEAGUE DOMINANCE FEATURES
# ---------------------------------------------------------

dominance_team_features = (
    dominance_check[
        [
            "match_id",
            "team",
            "venue",
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# HOME FEATURES
# ---------------------------------------------------------

dominance_home = (
    dominance_team_features[
        dominance_team_features["venue"]
        == "home"
    ][
        [
            "match_id",
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .rename(
        columns={
            "shot_difference_last5_all":
                "shot_difference_last5_all_home",

            "possession_difference_last5_all":
                "possession_difference_last5_all_home",

            "corner_difference_last5_all":
                "corner_difference_last5_all_home"
        }
    )
)


# ---------------------------------------------------------
# AWAY FEATURES
# ---------------------------------------------------------

dominance_away = (
    dominance_team_features[
        dominance_team_features["venue"]
        == "away"
    ][
        [
            "match_id",
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .rename(
        columns={
            "shot_difference_last5_all":
                "shot_difference_last5_all_away",

            "possession_difference_last5_all":
                "possession_difference_last5_all_away",

            "corner_difference_last5_all":
                "corner_difference_last5_all_away"
        }
    )
)


# ---------------------------------------------------------
# ONE ROW PER PREMIER LEAGUE MATCH
# ---------------------------------------------------------

dominance_match_features = (
    dominance_home
    .merge(
        dominance_away,
        on="match_id",
        how="inner"
    )
)


print(
    "Shape:",
    dominance_match_features.shape
)

print(
    "Duplicate match IDs:",
    dominance_match_features[
        "match_id"
    ].duplicated().sum()
)


print()
print("MISSING VALUES")
print()

print(
    dominance_match_features
    .drop(
        columns="match_id"
    )
    .isna()
    .sum()
)


print()
print("SAMPLE")
print()

print(
    dominance_match_features
    .head(10)
    .to_string(index=False)
)


# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

dominance_match_features.to_csv(
    "../data/processed/dominance_features.csv",
    index=False
)

print()
print(
    "Saved:",
    "../data/processed/dominance_features.csv"
)


Shape: (2280, 7)
Duplicate match IDs: 0

MISSING VALUES

shot_difference_last5_all_home          9
possession_difference_last5_all_home    9
corner_difference_last5_all_home        9
shot_difference_last5_all_away          9
possession_difference_last5_all_away    9
corner_difference_last5_all_away        9
dtype: int64

SAMPLE

    match_id  shot_difference_last5_all_home  possession_difference_last5_all_home  corner_difference_last5_all_home  shot_difference_last5_all_away  possession_difference_last5_all_away  corner_difference_last5_all_away
mt_304225642                             8.0                                   8.0                               1.0                            5.00                                  20.0                               2.0
mt_291883557                            -6.2                                  -1.6                              -0.4                           -3.75                                 -18.0                               0.0
mt_452

In [118]:
import pandas as pd
import numpy as np


# ---------------------------------------------------------
# LOAD UPDATED HISTORICAL MATCH DATA
# ---------------------------------------------------------

historical_df = pd.read_csv(
    "../data/processed/historical_matches.csv"
)


historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True,
    errors="coerce"
)


historical_df = (
    historical_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Historical shape:",
    historical_df.shape
)


print()

print(
    "Premier League matches:",
    (
        historical_df[
            "competition"
        ]
        == "premier_league"
    ).sum()
)


print()

print(
    "2026/27 Premier League matches:",
    (
        (
            historical_df[
                "competition"
            ]
            == "premier_league"
        )
        &
        (
            historical_df[
                "season"
            ]
            == "2026_27"
        )
    ).sum()
)

Historical shape: (10673, 22)

Premier League matches: 2300

2026/27 Premier League matches: 20


In [119]:
# ---------------------------------------------------------
# BUILD TEAM-PERSPECTIVE MATCH TABLE
# ---------------------------------------------------------

home_rows = pd.DataFrame({

    "match_id":
        historical_df["match_id"],

    "date":
        historical_df["date"],

    "season":
        historical_df["season"],

    "competition":
        historical_df["competition"],

    "team":
        historical_df["home_team"],

    "opponent":
        historical_df["away_team"],

    "venue":
        "home",

    "goals_for":
        historical_df["home_goals"],

    "goals_against":
        historical_df["away_goals"],

    "xg_for":
        historical_df["home_xg"],

    "xg_against":
        historical_df["away_xg"],

    "shots_for":
        historical_df["home_shots"],

    "shots_against":
        historical_df["away_shots"],

    "shots_on_target_for":
        historical_df["home_shots_on_target"],

    "shots_on_target_against":
        historical_df["away_shots_on_target"],

    "possession_for":
        historical_df["home_possession"],

    "possession_against":
        historical_df["away_possession"],

    "passes_for":
        historical_df["home_passes"],

    "passes_against":
        historical_df["away_passes"],

    "corners_for":
        historical_df["home_corners"],

    "corners_against":
        historical_df["away_corners"]
})


away_rows = pd.DataFrame({

    "match_id":
        historical_df["match_id"],

    "date":
        historical_df["date"],

    "season":
        historical_df["season"],

    "competition":
        historical_df["competition"],

    "team":
        historical_df["away_team"],

    "opponent":
        historical_df["home_team"],

    "venue":
        "away",

    "goals_for":
        historical_df["away_goals"],

    "goals_against":
        historical_df["home_goals"],

    "xg_for":
        historical_df["away_xg"],

    "xg_against":
        historical_df["home_xg"],

    "shots_for":
        historical_df["away_shots"],

    "shots_against":
        historical_df["home_shots"],

    "shots_on_target_for":
        historical_df["away_shots_on_target"],

    "shots_on_target_against":
        historical_df["home_shots_on_target"],

    "possession_for":
        historical_df["away_possession"],

    "possession_against":
        historical_df["home_possession"],

    "passes_for":
        historical_df["away_passes"],

    "passes_against":
        historical_df["home_passes"],

    "corners_for":
        historical_df["away_corners"],

    "corners_against":
        historical_df["home_corners"]
})


team_matches = pd.concat(
    [
        home_rows,
        away_rows
    ],
    ignore_index=True
)


team_matches = (
    team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Team matches shape:",
    team_matches.shape
)


print()

print(
    "2026/27 PL team rows:",
    (
        (
            team_matches["competition"]
            == "premier_league"
        )
        &
        (
            team_matches["season"]
            == "2026_27"
        )
    ).sum()
)

Team matches shape: (21346, 21)

2026/27 PL team rows: 40


In [120]:
# ---------------------------------------------------------
# BASIC TEAM-MATCH FEATURES
# ---------------------------------------------------------

# Result from the team's perspective
team_matches["result"] = np.select(
    [
        team_matches["goals_for"] > team_matches["goals_against"],
        team_matches["goals_for"] < team_matches["goals_against"]
    ],
    [
        "W",
        "L"
    ],
    default="D"
)


# Points earned
team_matches["points"] = np.select(
    [
        team_matches["result"] == "W",
        team_matches["result"] == "D"
    ],
    [
        3,
        1
    ],
    default=0
)


# Goal difference
team_matches["goal_difference"] = (
    team_matches["goals_for"]
    - team_matches["goals_against"]
)


# ---------------------------------------------------------
# DAYS REST
# ---------------------------------------------------------

# team_matches is already sorted by:
# team -> date -> match_id

team_matches["previous_match_date"] = (
    team_matches
    .groupby("team")["date"]
    .shift(1)
)


team_matches["days_rest"] = (
    team_matches["date"]
    - team_matches["previous_match_date"]
).dt.total_seconds() / 86400


# ---------------------------------------------------------
# CHECK
# ---------------------------------------------------------

print(
    "Team matches shape:",
    team_matches.shape
)

print()

print(
    team_matches[
        (
            team_matches["competition"] == "premier_league"
        )
        &
        (
            team_matches["season"] == "2026_27"
        )
    ][
        [
            "date",
            "team",
            "opponent",
            "venue",
            "result",
            "points",
            "goal_difference",
            "days_rest"
        ]
    ]
    .sort_values("date")
    .tail(10)
)

Team matches shape: (21346, 26)

      date               team           opponent venue result  points  \
11715  NaT  Manchester United          Hull City  away      D       1   
11847  NaT  Manchester United       Ipswich Town  home      D       1   
13161  NaT   Newcastle United  Tottenham Hotspur  away      D       1   
13162  NaT   Newcastle United          Liverpool  home      D       1   
13647  NaT  Nottingham Forest          Liverpool  away      D       1   
13894  NaT  Nottingham Forest       Leeds United  home      D       1   
18693  NaT         Sunderland             Fulham  home      D       1   
18728  NaT         Sunderland       Ipswich Town  away      D       1   
19402  NaT  Tottenham Hotspur   Newcastle United  home      D       1   
19492  NaT  Tottenham Hotspur          Brentford  away      D       1   

       goal_difference  days_rest  
11715              NaN        NaN  
11847              NaN        NaN  
13161              NaN        NaN  
13162              

In [127]:
# ---------------------------------------------------------
# RELOAD CORRECTED HISTORICAL DATA
# ---------------------------------------------------------

import pandas as pd
import numpy as np


historical_df = pd.read_csv(
    "../data/processed/historical_matches.csv"
)


historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True,
    errors="coerce"
)


historical_df = (
    historical_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# CHECK CURRENT PREMIER LEAGUE SEASON
# ---------------------------------------------------------

current_pl = historical_df[
    (
        historical_df["competition"]
        == "premier_league"
    )
    &
    (
        historical_df["season"]
        == "2026_27"
    )
].copy()


print(
    "Historical shape:",
    historical_df.shape
)

print(
    "2026/27 PL matches:",
    len(current_pl)
)

print(
    "Missing 2026/27 dates:",
    current_pl["date"].isna().sum()
)

print(
    "Missing 2026/27 scores:",
    current_pl[
        [
            "home_goals",
            "away_goals"
        ]
    ].isna().any(axis=1).sum()
)

print()

print(
    current_pl[
        [
            "date",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals"
        ]
    ]
    .sort_values("date")
    .tail(5)
    .to_string(index=False)
)

Historical shape: (10673, 22)
2026/27 PL matches: 20
Missing 2026/27 dates: 0
Missing 2026/27 scores: 0

                     date         home_team              away_team  home_goals  away_goals
2026-08-30 13:00:00+00:00        Sunderland                 Fulham           1           0
2026-08-30 13:00:00+00:00      Leeds United              Brentford           1           1
2026-08-30 13:00:00+00:00           Chelsea Brighton & Hove Albion           4           3
2026-08-30 15:30:00+00:00 Manchester United           Ipswich Town           5           2
2026-08-31 19:00:00+00:00       Aston Villa                Arsenal           0           1


In [123]:
# ---------------------------------------------------------
# REBUILD TEAM-PERSPECTIVE MATCH TABLE
# ---------------------------------------------------------

home_rows = pd.DataFrame({

    "match_id": historical_df["match_id"],
    "date": historical_df["date"],
    "season": historical_df["season"],
    "competition": historical_df["competition"],

    "team": historical_df["home_team"],
    "opponent": historical_df["away_team"],
    "venue": "home",

    "goals_for": historical_df["home_goals"],
    "goals_against": historical_df["away_goals"],

    "xg_for": historical_df["home_xg"],
    "xg_against": historical_df["away_xg"],

    "shots_for": historical_df["home_shots"],
    "shots_against": historical_df["away_shots"],

    "shots_on_target_for":
        historical_df["home_shots_on_target"],

    "shots_on_target_against":
        historical_df["away_shots_on_target"],

    "possession_for":
        historical_df["home_possession"],

    "possession_against":
        historical_df["away_possession"],

    "passes_for":
        historical_df["home_passes"],

    "passes_against":
        historical_df["away_passes"],

    "corners_for":
        historical_df["home_corners"],

    "corners_against":
        historical_df["away_corners"]
})


away_rows = pd.DataFrame({

    "match_id": historical_df["match_id"],
    "date": historical_df["date"],
    "season": historical_df["season"],
    "competition": historical_df["competition"],

    "team": historical_df["away_team"],
    "opponent": historical_df["home_team"],
    "venue": "away",

    "goals_for": historical_df["away_goals"],
    "goals_against": historical_df["home_goals"],

    "xg_for": historical_df["away_xg"],
    "xg_against": historical_df["home_xg"],

    "shots_for": historical_df["away_shots"],
    "shots_against": historical_df["home_shots"],

    "shots_on_target_for":
        historical_df["away_shots_on_target"],

    "shots_on_target_against":
        historical_df["home_shots_on_target"],

    "possession_for":
        historical_df["away_possession"],

    "possession_against":
        historical_df["home_possession"],

    "passes_for":
        historical_df["away_passes"],

    "passes_against":
        historical_df["home_passes"],

    "corners_for":
        historical_df["away_corners"],

    "corners_against":
        historical_df["home_corners"]
})


team_matches = pd.concat(
    [
        home_rows,
        away_rows
    ],
    ignore_index=True
)


team_matches = (
    team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# CHECK
# ---------------------------------------------------------

current_team_rows = team_matches[
    (
        team_matches["competition"]
        == "premier_league"
    )
    &
    (
        team_matches["season"]
        == "2026_27"
    )
]


print(
    "Team matches shape:",
    team_matches.shape
)

print(
    "2026/27 PL team rows:",
    len(current_team_rows)
)

print(
    "Missing 2026/27 dates:",
    current_team_rows["date"].isna().sum()
)

print()

print(
    current_team_rows[
        [
            "date",
            "team",
            "opponent",
            "venue",
            "goals_for",
            "goals_against"
        ]
    ]
    .sort_values("date")
    .tail(10)
    .to_string(index=False)
)

Team matches shape: (21346, 21)
2026/27 PL team rows: 40
Missing 2026/27 dates: 0

                     date                   team               opponent venue  goals_for  goals_against
2026-08-30 13:00:00+00:00                Chelsea Brighton & Hove Albion  home          4              3
2026-08-30 13:00:00+00:00 Brighton & Hove Albion                Chelsea  away          3              4
2026-08-30 13:00:00+00:00              Brentford           Leeds United  away          1              1
2026-08-30 13:00:00+00:00             Sunderland                 Fulham  home          1              0
2026-08-30 13:00:00+00:00           Leeds United              Brentford  home          1              1
2026-08-30 13:00:00+00:00                 Fulham             Sunderland  away          0              1
2026-08-30 15:30:00+00:00           Ipswich Town      Manchester United  away          2              5
2026-08-30 15:30:00+00:00      Manchester United           Ipswich Town  home        

In [124]:
# ---------------------------------------------------------
# BASIC TEAM-MATCH FEATURES
# ---------------------------------------------------------

team_matches["result"] = np.select(
    [
        team_matches["goals_for"] > team_matches["goals_against"],
        team_matches["goals_for"] < team_matches["goals_against"]
    ],
    [
        "W",
        "L"
    ],
    default="D"
)


team_matches["points"] = np.select(
    [
        team_matches["result"] == "W",
        team_matches["result"] == "D"
    ],
    [
        3,
        1
    ],
    default=0
)


team_matches["goal_difference"] = (
    team_matches["goals_for"]
    - team_matches["goals_against"]
)


# ---------------------------------------------------------
# DAYS REST
# ---------------------------------------------------------

team_matches["previous_match_date"] = (
    team_matches
    .groupby("team")["date"]
    .shift(1)
)


team_matches["days_rest"] = (
    team_matches["date"]
    - team_matches["previous_match_date"]
).dt.total_seconds() / 86400


# ---------------------------------------------------------
# CHECK 2026/27 PREMIER LEAGUE ROWS
# ---------------------------------------------------------

check_rest = (
    team_matches[
        (
            team_matches["competition"] == "premier_league"
        )
        &
        (
            team_matches["season"] == "2026_27"
        )
    ][
        [
            "date",
            "team",
            "opponent",
            "result",
            "points",
            "goal_difference",
            "days_rest"
        ]
    ]
    .sort_values(
        [
            "date",
            "team"
        ]
    )
)


print(
    "Team matches shape:",
    team_matches.shape
)

print()

print(
    "Missing 2026/27 rest values:",
    check_rest["days_rest"].isna().sum()
)

print(
    "Negative 2026/27 rest values:",
    (check_rest["days_rest"] < 0).sum()
)

print()

print(
    check_rest.tail(12).to_string(index=False)
)

Team matches shape: (21346, 26)

Missing 2026/27 rest values: 0
Negative 2026/27 rest values: 0

                     date                   team               opponent result  points  goal_difference  days_rest
2026-08-29 16:30:00+00:00       Newcastle United      Tottenham Hotspur      W       3                2   6.041667
2026-08-29 16:30:00+00:00      Tottenham Hotspur       Newcastle United      L       0               -2   7.000000
2026-08-30 13:00:00+00:00              Brentford           Leeds United      D       1                0   7.854167
2026-08-30 13:00:00+00:00 Brighton & Hove Albion                Chelsea      L       0               -1   7.000000
2026-08-30 13:00:00+00:00                Chelsea Brighton & Hove Albion      W       3                1   5.750000
2026-08-30 13:00:00+00:00                 Fulham             Sunderland      L       0               -1   5.750000
2026-08-30 13:00:00+00:00           Leeds United              Brentford      D       1            

In [125]:
# ---------------------------------------------------------
# ALL-COMPETITION ROLLING FEATURES
# ---------------------------------------------------------

# Win indicator
team_matches["win"] = (
    team_matches["result"] == "W"
).astype(int)


# Match-level performance differences
team_matches["xg_difference"] = (
    team_matches["xg_for"]
    - team_matches["xg_against"]
)


team_matches["shots_on_target_difference"] = (
    team_matches["shots_on_target_for"]
    - team_matches["shots_on_target_against"]
)


# ---------------------------------------------------------
# LEAKAGE-SAFE LAST-5 ALL-COMP FEATURES
# ---------------------------------------------------------

all_comp_rolling_columns = {

    "points":
        "points_last5_all",

    "goal_difference":
        "goal_difference_last5_all",

    "win":
        "win_rate_last5_all",

    "xg_difference":
        "xg_difference_last5_all",

    "shots_on_target_difference":
        "shots_on_target_difference_last5_all"
}


for source_col, new_col in all_comp_rolling_columns.items():

    team_matches[new_col] = (
        team_matches
        .groupby("team")[source_col]
        .transform(
            lambda x:
                x.shift(1)
                .rolling(
                    window=5,
                    min_periods=1
                )
                .mean()
        )
    )


# ---------------------------------------------------------
# CHECK CURRENT PREMIER LEAGUE SEASON
# ---------------------------------------------------------

check_all_form = (
    team_matches[
        (
            team_matches["competition"]
            == "premier_league"
        )
        &
        (
            team_matches["season"]
            == "2026_27"
        )
    ][
        [
            "date",
            "team",
            "opponent",
            "points_last5_all",
            "goal_difference_last5_all",
            "win_rate_last5_all",
            "shots_on_target_difference_last5_all"
        ]
    ]
    .sort_values(
        [
            "date",
            "team"
        ]
    )
)


print(
    "Team matches shape:",
    team_matches.shape
)

print()

print(
    "2026/27 PL rows:",
    len(check_all_form)
)

print()

print(
    "Missing points_last5_all:",
    check_all_form["points_last5_all"].isna().sum()
)

print(
    "Missing goal_difference_last5_all:",
    check_all_form["goal_difference_last5_all"].isna().sum()
)

print(
    "Missing win_rate_last5_all:",
    check_all_form["win_rate_last5_all"].isna().sum()
)

print()

print(
    check_all_form.tail(12).to_string(index=False)
)

Team matches shape: (21346, 34)

2026/27 PL rows: 40

Missing points_last5_all: 0
Missing goal_difference_last5_all: 0
Missing win_rate_last5_all: 0

                     date                   team               opponent  points_last5_all  goal_difference_last5_all  win_rate_last5_all  shots_on_target_difference_last5_all
2026-08-29 16:30:00+00:00       Newcastle United      Tottenham Hotspur               1.6                        0.4                 0.4                                   NaN
2026-08-29 16:30:00+00:00      Tottenham Hotspur       Newcastle United               1.4                       -0.4                 0.4                                   NaN
2026-08-30 13:00:00+00:00              Brentford           Leeds United               1.6                        0.6                 0.4                                   NaN
2026-08-30 13:00:00+00:00 Brighton & Hove Albion                Chelsea               1.2                        0.2                 0.4              

In [126]:
# ---------------------------------------------------------
# PREMIER-LEAGUE-ONLY ROLLING FEATURES
# ---------------------------------------------------------

pl_team_matches = (
    team_matches[
        team_matches["competition"] == "premier_league"
    ]
    .copy()
)


pl_team_matches = (
    pl_team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# LAST-5 PL FEATURES
# ---------------------------------------------------------

pl_last5_columns = {

    "points":
        "points_last5_pl",

    "goal_difference":
        "goal_difference_last5_pl",

    "win":
        "win_rate_last5_pl",

    "xg_difference":
        "xg_difference_last5_pl",

    "shots_on_target_difference":
        "shots_on_target_difference_last5_pl"
}


for source_col, new_col in pl_last5_columns.items():

    pl_team_matches[new_col] = (
        pl_team_matches
        .groupby("team")[source_col]
        .transform(
            lambda x:
                x.shift(1)
                .rolling(
                    window=5,
                    min_periods=1
                )
                .mean()
        )
    )


# ---------------------------------------------------------
# LAST-10 PL FEATURES
# ---------------------------------------------------------

pl_last10_columns = {

    "points":
        "points_last10_pl",

    "goal_difference":
        "goal_difference_last10_pl",

    "win":
        "win_rate_last10_pl"
}


for source_col, new_col in pl_last10_columns.items():

    pl_team_matches[new_col] = (
        pl_team_matches
        .groupby("team")[source_col]
        .transform(
            lambda x:
                x.shift(1)
                .rolling(
                    window=10,
                    min_periods=1
                )
                .mean()
        )
    )


# ---------------------------------------------------------
# CHECK
# ---------------------------------------------------------

check_pl_form = (
    pl_team_matches[
        pl_team_matches["season"] == "2026_27"
    ][
        [
            "date",
            "team",
            "opponent",
            "points_last5_pl",
            "goal_difference_last5_pl",
            "win_rate_last5_pl",
            "shots_on_target_difference_last5_pl",
            "points_last10_pl",
            "goal_difference_last10_pl",
            "win_rate_last10_pl"
        ]
    ]
    .sort_values(
        [
            "date",
            "team"
        ]
    )
)


print(
    "PL team matches shape:",
    pl_team_matches.shape
)

print()

print(
    "2026/27 PL team rows:",
    len(check_pl_form)
)

print()

print(
    "Missing points_last5_pl:",
    check_pl_form["points_last5_pl"].isna().sum()
)

print(
    "Missing points_last10_pl:",
    check_pl_form["points_last10_pl"].isna().sum()
)

print()

print(
    check_pl_form.tail(12).to_string(index=False)
)

PL team matches shape: (4600, 42)

2026/27 PL team rows: 40

Missing points_last5_pl: 2
Missing points_last10_pl: 2

                     date                   team               opponent  points_last5_pl  goal_difference_last5_pl  win_rate_last5_pl  shots_on_target_difference_last5_pl  points_last10_pl  goal_difference_last10_pl  win_rate_last10_pl
2026-08-29 16:30:00+00:00       Newcastle United      Tottenham Hotspur              1.6                       0.4                0.4                                  NaN               1.1                       -0.1                 0.3
2026-08-29 16:30:00+00:00      Tottenham Hotspur       Newcastle United              1.4                      -0.4                0.4                                  NaN               1.2                       -0.5                 0.3
2026-08-30 13:00:00+00:00              Brentford           Leeds United              1.6                       0.6                0.4                                  NaN     

In [128]:
# ---------------------------------------------------------
# RELOAD CORRECTED HISTORICAL DATA
# ---------------------------------------------------------

import pandas as pd
import numpy as np


historical_df = pd.read_csv(
    "../data/processed/historical_matches.csv"
)


historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True,
    errors="coerce"
)


historical_df = (
    historical_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Historical shape:",
    historical_df.shape
)


# ---------------------------------------------------------
# BUILD TEAM-PERSPECTIVE MATCH DATA
# ---------------------------------------------------------

home_rows = pd.DataFrame({

    "match_id":
        historical_df["match_id"],

    "date":
        historical_df["date"],

    "season":
        historical_df["season"],

    "competition":
        historical_df["competition"],

    "team":
        historical_df["home_team"],

    "opponent":
        historical_df["away_team"],

    "venue":
        "home",

    "goals_for":
        historical_df["home_goals"],

    "goals_against":
        historical_df["away_goals"],

    "xg_for":
        historical_df["home_xg"],

    "xg_against":
        historical_df["away_xg"],

    "shots_for":
        historical_df["home_shots"],

    "shots_against":
        historical_df["away_shots"],

    "shots_on_target_for":
        historical_df["home_shots_on_target"],

    "shots_on_target_against":
        historical_df["away_shots_on_target"],

    "possession_for":
        historical_df["home_possession"],

    "possession_against":
        historical_df["away_possession"],

    "passes_for":
        historical_df["home_passes"],

    "passes_against":
        historical_df["away_passes"],

    "corners_for":
        historical_df["home_corners"],

    "corners_against":
        historical_df["away_corners"]
})


away_rows = pd.DataFrame({

    "match_id":
        historical_df["match_id"],

    "date":
        historical_df["date"],

    "season":
        historical_df["season"],

    "competition":
        historical_df["competition"],

    "team":
        historical_df["away_team"],

    "opponent":
        historical_df["home_team"],

    "venue":
        "away",

    "goals_for":
        historical_df["away_goals"],

    "goals_against":
        historical_df["home_goals"],

    "xg_for":
        historical_df["away_xg"],

    "xg_against":
        historical_df["home_xg"],

    "shots_for":
        historical_df["away_shots"],

    "shots_against":
        historical_df["home_shots"],

    "shots_on_target_for":
        historical_df["away_shots_on_target"],

    "shots_on_target_against":
        historical_df["home_shots_on_target"],

    "possession_for":
        historical_df["away_possession"],

    "possession_against":
        historical_df["home_possession"],

    "passes_for":
        historical_df["away_passes"],

    "passes_against":
        historical_df["home_passes"],

    "corners_for":
        historical_df["away_corners"],

    "corners_against":
        historical_df["home_corners"]
})


team_matches = pd.concat(
    [
        home_rows,
        away_rows
    ],
    ignore_index=True
)


team_matches = (
    team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Team matches shape:",
    team_matches.shape
)


# ---------------------------------------------------------
# BASIC FEATURES
# ---------------------------------------------------------

team_matches["result"] = np.select(

    [
        team_matches["goals_for"]
        > team_matches["goals_against"],

        team_matches["goals_for"]
        < team_matches["goals_against"]
    ],

    [
        "W",
        "L"
    ],

    default="D"
)


team_matches["points"] = np.select(

    [
        team_matches["result"] == "W",
        team_matches["result"] == "D"
    ],

    [
        3,
        1
    ],

    default=0
)


team_matches["goal_difference"] = (
    team_matches["goals_for"]
    -
    team_matches["goals_against"]
)


team_matches["previous_match_date"] = (
    team_matches
    .groupby("team")["date"]
    .shift(1)
)


team_matches["days_rest"] = (

    team_matches["date"]
    -
    team_matches["previous_match_date"]

).dt.days


team_matches["win"] = (
    team_matches["result"] == "W"
).astype(int)


team_matches["xg_difference"] = (
    team_matches["xg_for"]
    -
    team_matches["xg_against"]
)


team_matches["shots_on_target_difference"] = (
    team_matches["shots_on_target_for"]
    -
    team_matches["shots_on_target_against"]
)


# ---------------------------------------------------------
# ALL-COMPETITION LAST-5 FEATURES
# ---------------------------------------------------------

all_comp_last5 = {

    "points":
        "points_last5_all",

    "goal_difference":
        "goal_difference_last5_all",

    "win":
        "win_rate_last5_all",

    "xg_difference":
        "xg_difference_last5_all",

    "shots_on_target_difference":
        "shots_on_target_difference_last5_all"
}


for source_col, new_col in all_comp_last5.items():

    team_matches[new_col] = (
        team_matches
        .groupby("team")[source_col]
        .transform(
            lambda x:
                x.shift(1)
                .rolling(
                    window=5,
                    min_periods=1
                )
                .mean()
        )
    )


# ---------------------------------------------------------
# PREMIER LEAGUE ONLY DATA
# ---------------------------------------------------------

pl_team_matches = (
    team_matches[
        team_matches["competition"]
        == "premier_league"
    ]
    .copy()
)


pl_team_matches = (
    pl_team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# PL LAST-5 FEATURES
# ---------------------------------------------------------

pl_last5_columns = {

    "points":
        "points_last5_pl",

    "goal_difference":
        "goal_difference_last5_pl",

    "win":
        "win_rate_last5_pl",

    "xg_difference":
        "xg_difference_last5_pl",

    "shots_on_target_difference":
        "shots_on_target_difference_last5_pl"
}


for source_col, new_col in pl_last5_columns.items():

    pl_team_matches[new_col] = (
        pl_team_matches
        .groupby("team")[source_col]
        .transform(
            lambda x:
                x.shift(1)
                .rolling(
                    window=5,
                    min_periods=1
                )
                .mean()
        )
    )


# ---------------------------------------------------------
# PL LAST-10 FEATURES
# ---------------------------------------------------------

pl_last10_columns = {

    "points":
        "points_last10_pl",

    "goal_difference":
        "goal_difference_last10_pl",

    "win":
        "win_rate_last10_pl"
}


for source_col, new_col in pl_last10_columns.items():

    pl_team_matches[new_col] = (
        pl_team_matches
        .groupby("team")[source_col]
        .transform(
            lambda x:
                x.shift(1)
                .rolling(
                    window=10,
                    min_periods=1
                )
                .mean()
        )
    )


# ---------------------------------------------------------
# QUICK CHECK
# ---------------------------------------------------------

check_2026 = (
    pl_team_matches[
        pl_team_matches["season"]
        == "2026_27"
    ]
)


print()

print(
    "PL team matches shape:",
    pl_team_matches.shape
)

print(
    "2026/27 rows:",
    len(check_2026)
)

print()

print(
    "Missing SOT difference:",
    check_2026[
        "shots_on_target_difference"
    ].isna().sum()
)

print(
    "Missing SOT last5 all:",
    check_2026[
        "shots_on_target_difference_last5_all"
    ].isna().sum()
)

print(
    "Missing SOT last5 PL:",
    check_2026[
        "shots_on_target_difference_last5_pl"
    ].isna().sum()
)

Historical shape: (10673, 22)
Team matches shape: (21346, 21)

PL team matches shape: (4600, 42)
2026/27 rows: 40

Missing SOT difference: 0
Missing SOT last5 all: 0
Missing SOT last5 PL: 2


In [129]:
# ---------------------------------------------------------
# MATCH CONGESTION FEATURES
# ALL COMPETITIONS
# ---------------------------------------------------------

import numpy as np
import pandas as pd


# Work from all team matches because cup/European matches
# also contribute to fixture congestion.
congestion_df = (
    team_matches[
        [
            "match_id",
            "team",
            "date"
        ]
    ]
    .copy()
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


congestion_df["matches_last_7_days"] = 0
congestion_df["matches_last_14_days"] = 0


# ---------------------------------------------------------
# COUNT PRIOR MATCHES ONLY
# ---------------------------------------------------------

for team, group in congestion_df.groupby(
    "team",
    sort=False
):

    group = group.sort_values(
        [
            "date",
            "match_id"
        ]
    )

    dates = group["date"].values.astype(
        "datetime64[ns]"
    )


    counts_7 = []
    counts_14 = []


    for i, current_date in enumerate(dates):

        previous_dates = dates[:i]


        # Previous 7 days
        cutoff_7 = (
            current_date
            -
            np.timedelta64(7, "D")
        )

        count_7 = np.sum(
            previous_dates >= cutoff_7
        )


        # Previous 14 days
        cutoff_14 = (
            current_date
            -
            np.timedelta64(14, "D")
        )

        count_14 = np.sum(
            previous_dates >= cutoff_14
        )


        counts_7.append(
            count_7
        )

        counts_14.append(
            count_14
        )


    congestion_df.loc[
        group.index,
        "matches_last_7_days"
    ] = counts_7


    congestion_df.loc[
        group.index,
        "matches_last_14_days"
    ] = counts_14


# ---------------------------------------------------------
# MERGE CONGESTION INTO PL DATA
# ---------------------------------------------------------

pl_team_matches = (
    pl_team_matches
    .drop(
        columns=[
            "matches_last_7_days",
            "matches_last_14_days"
        ],
        errors="ignore"
    )
    .merge(
        congestion_df[
            [
                "match_id",
                "team",
                "matches_last_7_days",
                "matches_last_14_days"
            ]
        ],
        on=[
            "match_id",
            "team"
        ],
        how="left"
    )
)


# ---------------------------------------------------------
# CHECK 2026/27
# ---------------------------------------------------------

check_2026_congestion = (
    pl_team_matches[
        pl_team_matches["season"]
        == "2026_27"
    ]
    .sort_values(
        [
            "date",
            "team"
        ]
    )
)


print(
    "PL team matches shape:",
    pl_team_matches.shape
)

print(
    "2026/27 rows:",
    len(check_2026_congestion)
)

print()

print(
    "Missing matches_last_7_days:",
    check_2026_congestion[
        "matches_last_7_days"
    ].isna().sum()
)

print(
    "Missing matches_last_14_days:",
    check_2026_congestion[
        "matches_last_14_days"
    ].isna().sum()
)

print()

print(
    check_2026_congestion[
        [
            "date",
            "team",
            "opponent",
            "days_rest",
            "matches_last_7_days",
            "matches_last_14_days"
        ]
    ]
    .tail(12)
    .to_string(index=False)
)

PL team matches shape: (4600, 44)
2026/27 rows: 40

Missing matches_last_7_days: 0
Missing matches_last_14_days: 0

                     date                   team               opponent  days_rest  matches_last_7_days  matches_last_14_days
2026-08-29 16:30:00+00:00       Newcastle United      Tottenham Hotspur        6.0                    1                     1
2026-08-29 16:30:00+00:00      Tottenham Hotspur       Newcastle United        7.0                    1                     1
2026-08-30 13:00:00+00:00              Brentford           Leeds United        7.0                    0                     1
2026-08-30 13:00:00+00:00 Brighton & Hove Albion                Chelsea        7.0                    1                     1
2026-08-30 13:00:00+00:00                Chelsea Brighton & Hove Albion        5.0                    1                     1
2026-08-30 13:00:00+00:00                 Fulham             Sunderland        5.0                    1                     1
20

In [130]:
# ---------------------------------------------------------
# REBUILD LOCKED V1 CROSS-COMPETITION ELO
# ---------------------------------------------------------

import pandas as pd


STARTING_ELO = 1500
K_FACTOR = 20


# ---------------------------------------------------------
# SORT EVERY MATCH CHRONOLOGICALLY
# ---------------------------------------------------------

elo_source = (
    historical_df[
        [
            "match_id",
            "date",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals"
        ]
    ]
    .dropna(
        subset=[
            "date",
            "home_goals",
            "away_goals"
        ]
    )
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


ratings = {}
elo_rows = []


# ---------------------------------------------------------
# PROCESS MATCHES ONE AT A TIME
# ---------------------------------------------------------

for _, match in elo_source.iterrows():

    home_team = match["home_team"]
    away_team = match["away_team"]


    # First time we see a team -> 1500
    home_elo = ratings.get(
        home_team,
        STARTING_ELO
    )

    away_elo = ratings.get(
        away_team,
        STARTING_ELO
    )


    # -----------------------------------------------------
    # STORE PRE-MATCH ELO
    # -----------------------------------------------------

    elo_rows.append({

        "match_id":
            match["match_id"],

        "home_elo":
            home_elo,

        "away_elo":
            away_elo,

        "elo_difference":
            home_elo - away_elo
    })


    # -----------------------------------------------------
    # EXPECTED RESULT
    # NO HOME-ADVANTAGE BONUS
    # -----------------------------------------------------

    expected_home = (
        1
        /
        (
            1
            +
            10 ** (
                (away_elo - home_elo)
                / 400
            )
        )
    )


    expected_away = (
        1
        -
        expected_home
    )


    # -----------------------------------------------------
    # ACTUAL RESULT
    # WIN = 1
    # DRAW = 0.5
    # LOSS = 0
    # -----------------------------------------------------

    if (
        match["home_goals"]
        >
        match["away_goals"]
    ):

        actual_home = 1.0
        actual_away = 0.0


    elif (
        match["home_goals"]
        <
        match["away_goals"]
    ):

        actual_home = 0.0
        actual_away = 1.0


    else:

        actual_home = 0.5
        actual_away = 0.5


    # -----------------------------------------------------
    # UPDATE RATINGS AFTER THE MATCH
    # -----------------------------------------------------

    ratings[home_team] = (
        home_elo
        +
        K_FACTOR
        *
        (
            actual_home
            -
            expected_home
        )
    )


    ratings[away_team] = (
        away_elo
        +
        K_FACTOR
        *
        (
            actual_away
            -
            expected_away
        )
    )


# ---------------------------------------------------------
# CREATE ELO DATAFRAME
# ---------------------------------------------------------

elo_history_df = pd.DataFrame(
    elo_rows
)


print(
    "Elo history shape:",
    elo_history_df.shape
)

print(
    "Teams rated:",
    len(ratings)
)


# ---------------------------------------------------------
# MERGE PRE-MATCH ELO INTO PL TEAM DATA
# ---------------------------------------------------------

home_elo_df = (
    elo_history_df[
        [
            "match_id",
            "home_elo",
            "away_elo",
            "elo_difference"
        ]
    ]
    .rename(
        columns={
            "home_elo":
                "team_elo",

            "away_elo":
                "opponent_elo",

            "elo_difference":
                "team_elo_difference"
        }
    )
)


away_elo_df = (
    elo_history_df[
        [
            "match_id",
            "home_elo",
            "away_elo",
            "elo_difference"
        ]
    ]
    .rename(
        columns={
            "away_elo":
                "team_elo",

            "home_elo":
                "opponent_elo",

            "elo_difference":
                "home_elo_difference"
        }
    )
)


# Away team's own difference should be team - opponent
away_elo_df["team_elo_difference"] = (
    away_elo_df["team_elo"]
    -
    away_elo_df["opponent_elo"]
)


away_elo_df = away_elo_df[
    [
        "match_id",
        "team_elo",
        "opponent_elo",
        "team_elo_difference"
    ]
]


# ---------------------------------------------------------
# ADD ELO BASED ON VENUE
# ---------------------------------------------------------

pl_team_matches = (
    pl_team_matches
    .drop(
        columns=[
            "team_elo",
            "opponent_elo",
            "team_elo_difference"
        ],
        errors="ignore"
    )
)


pl_home = (
    pl_team_matches[
        pl_team_matches["venue"] == "home"
    ]
    .merge(
        home_elo_df,
        on="match_id",
        how="left"
    )
)


pl_away = (
    pl_team_matches[
        pl_team_matches["venue"] == "away"
    ]
    .merge(
        away_elo_df,
        on="match_id",
        how="left"
    )
)


pl_team_matches = pd.concat(
    [
        pl_home,
        pl_away
    ],
    ignore_index=True
)


pl_team_matches = (
    pl_team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# CHECK CURRENT SEASON
# ---------------------------------------------------------

check_2026_elo = (
    pl_team_matches[
        pl_team_matches["season"]
        == "2026_27"
    ]
    .sort_values(
        [
            "date",
            "team"
        ]
    )
)


print()

print(
    "PL team matches shape:",
    pl_team_matches.shape
)

print(
    "2026/27 rows:",
    len(check_2026_elo)
)

print()

print(
    "Missing team Elo:",
    check_2026_elo[
        "team_elo"
    ].isna().sum()
)

print(
    "Missing opponent Elo:",
    check_2026_elo[
        "opponent_elo"
    ].isna().sum()
)

print()

print(
    check_2026_elo[
        [
            "date",
            "team",
            "opponent",
            "venue",
            "team_elo",
            "opponent_elo",
            "team_elo_difference"
        ]
    ]
    .tail(12)
    .to_string(index=False)
)

Elo history shape: (10673, 4)
Teams rated: 522

PL team matches shape: (4600, 47)
2026/27 rows: 40

Missing team Elo: 0
Missing opponent Elo: 0

                     date                   team               opponent venue    team_elo  opponent_elo  team_elo_difference
2026-08-29 16:30:00+00:00       Newcastle United      Tottenham Hotspur  away 1661.790938   1596.945635            64.845303
2026-08-29 16:30:00+00:00      Tottenham Hotspur       Newcastle United  home 1596.945635   1661.790938           -64.845303
2026-08-30 13:00:00+00:00              Brentford           Leeds United  away 1673.470763   1659.223569            14.247194
2026-08-30 13:00:00+00:00 Brighton & Hove Albion                Chelsea  away 1686.358773   1671.413483            14.945290
2026-08-30 13:00:00+00:00                Chelsea Brighton & Hove Albion  home 1671.413483   1686.358773           -14.945290
2026-08-30 13:00:00+00:00                 Fulham             Sunderland  away 1629.483862   1599.991833  

In [131]:
# ---------------------------------------------------------
# BUILD MATCH-LEVEL PREMIER LEAGUE FEATURE DATASET
# ---------------------------------------------------------

# Home team rows
home_match_features = (
    pl_team_matches[
        pl_team_matches["venue"] == "home"
    ]
    .copy()
)


# Away team rows
away_match_features = (
    pl_team_matches[
        pl_team_matches["venue"] == "away"
    ]
    .copy()
)


# ---------------------------------------------------------
# COLUMNS WE WANT TO KEEP
# ---------------------------------------------------------

feature_columns = [

    "points_last5_all",
    "goal_difference_last5_all",
    "win_rate_last5_all",
    "shots_on_target_difference_last5_all",

    "points_last5_pl",
    "goal_difference_last5_pl",
    "win_rate_last5_pl",
    "shots_on_target_difference_last5_pl",

    "points_last10_pl",
    "goal_difference_last10_pl",
    "win_rate_last10_pl",

    "days_rest",
    "matches_last_7_days",
    "matches_last_14_days",

    "team_elo"
]


# ---------------------------------------------------------
# HOME SIDE
# ---------------------------------------------------------

home_keep = (
    [
        "match_id",
        "date",
        "season",
        "team",
        "opponent",
        "goals_for",
        "goals_against"
    ]
    +
    feature_columns
)


home_match_features = (
    home_match_features[
        home_keep
    ]
    .rename(
        columns={
            "team":
                "home_team",

            "opponent":
                "away_team",

            "goals_for":
                "home_goals",

            "goals_against":
                "away_goals"
        }
    )
)


home_match_features = (
    home_match_features
    .rename(
        columns={
            col: f"{col}_home"
            for col in feature_columns
        }
    )
)


# ---------------------------------------------------------
# AWAY SIDE
# ---------------------------------------------------------

away_keep = (
    [
        "match_id",
        "team"
    ]
    +
    feature_columns
)


away_match_features = (
    away_match_features[
        away_keep
    ]
    .rename(
        columns={
            "team":
                "away_team_check"
        }
    )
)


away_match_features = (
    away_match_features
    .rename(
        columns={
            col: f"{col}_away"
            for col in feature_columns
        }
    )
)


# ---------------------------------------------------------
# MERGE HOME + AWAY
# ---------------------------------------------------------

model_df = (
    home_match_features
    .merge(
        away_match_features,
        on="match_id",
        how="left"
    )
)


# ---------------------------------------------------------
# VERIFY AWAY TEAM MATCH
# ---------------------------------------------------------

model_df["away_team_match"] = (
    model_df["away_team"]
    ==
    model_df["away_team_check"]
)


print(
    "Away team mismatches:",
    (~model_df["away_team_match"]).sum()
)


model_df = model_df.drop(
    columns=[
        "away_team_check",
        "away_team_match"
    ]
)


# ---------------------------------------------------------
# TARGET
# ---------------------------------------------------------

model_df["target"] = np.select(

    [
        model_df["home_goals"]
        >
        model_df["away_goals"],

        model_df["home_goals"]
        <
        model_df["away_goals"]
    ],

    [
        "H",
        "A"
    ],

    default="D"
)


# ---------------------------------------------------------
# LOCKED RF V1 ELO DIFFERENCE
# HOME ELO - AWAY ELO
# ---------------------------------------------------------

model_df["team_elo_difference_home"] = (
    model_df["team_elo_home"]
    -
    model_df["team_elo_away"]
)


# ---------------------------------------------------------
# LOCKED 31 FEATURES
# ---------------------------------------------------------

baseline_features = [

    "points_last5_all_home",
    "goal_difference_last5_all_home",
    "win_rate_last5_all_home",
    "shots_on_target_difference_last5_all_home",

    "points_last5_all_away",
    "goal_difference_last5_all_away",
    "win_rate_last5_all_away",
    "shots_on_target_difference_last5_all_away",

    "points_last5_pl_home",
    "goal_difference_last5_pl_home",
    "win_rate_last5_pl_home",
    "shots_on_target_difference_last5_pl_home",
    "points_last10_pl_home",
    "goal_difference_last10_pl_home",
    "win_rate_last10_pl_home",

    "points_last5_pl_away",
    "goal_difference_last5_pl_away",
    "win_rate_last5_pl_away",
    "shots_on_target_difference_last5_pl_away",
    "points_last10_pl_away",
    "goal_difference_last10_pl_away",
    "win_rate_last10_pl_away",

    "days_rest_home",
    "days_rest_away",
    "matches_last_7_days_home",
    "matches_last_7_days_away",
    "matches_last_14_days_home",
    "matches_last_14_days_away",

    "team_elo_home",
    "team_elo_away",
    "team_elo_difference_home"
]


# ---------------------------------------------------------
# CHECK
# ---------------------------------------------------------

model_2026 = (
    model_df[
        model_df["season"] == "2026_27"
    ]
    .sort_values("date")
)


print()

print(
    "Model dataframe shape:",
    model_df.shape
)

print(
    "Locked feature count:",
    len(baseline_features)
)

print(
    "2026/27 matches:",
    len(model_2026)
)

print()

print(
    "Missing locked-feature values in 2026/27:",
    model_2026[
        baseline_features
    ]
    .isna()
    .sum()
    .sum()
)

print()

print(
    model_2026[
        [
            "date",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals",
            "target"
        ]
    ]
    .to_string(index=False)
)

Away team mismatches: 0

Model dataframe shape: (2300, 39)
Locked feature count: 31
2026/27 matches: 20

Missing locked-feature values in 2026/27: 14

                     date              home_team              away_team  home_goals  away_goals target
2026-08-21 19:00:00+00:00                Arsenal          Coventry City           3           0      H
2026-08-22 11:30:00+00:00              Hull City      Manchester United           2           0      H
2026-08-22 14:00:00+00:00      Nottingham Forest           Leeds United           0           1      A
2026-08-22 14:00:00+00:00           Ipswich Town             Sunderland           2           1      H
2026-08-22 14:00:00+00:00                Everton         Crystal Palace           2           0      H
2026-08-22 16:30:00+00:00              Brentford      Tottenham Hotspur           3           0      H
2026-08-23 13:00:00+00:00 Brighton & Hove Albion            Aston Villa           4           0      H
2026-08-23 13:00:00+00:00

In [132]:
# ---------------------------------------------------------
# SAVE UPDATED MATCH-LEVEL FEATURE DATASET
# ---------------------------------------------------------

output_path = "../data/processed/pl_match_features.csv"


model_df.to_csv(
    output_path,
    index=False
)


print(
    "Saved:",
    output_path
)

print(
    "Shape:",
    model_df.shape
)

print(
    "Columns:",
    len(model_df.columns)
)

print()

print(
    "Seasons:"
)

print(
    model_df["season"]
    .value_counts()
    .sort_index()
)

Saved: ../data/processed/pl_match_features.csv
Shape: (2300, 39)
Columns: 39

Seasons:
season
2020_21    380
2021_22    380
2022_23    380
2023_24    380
2024_25    380
2025_26    380
2026_27     20
Name: count, dtype: int64
